# bob, explained

An FPGA built inside an FPGA, taken apart one topic at a time.

**Run the first two cells once.** After that each episode is a single cell: run it and its video appears inline.

The magic line carries the quality (`-ql` draft, `-qm` medium, `-qh` 1080p60) and the scene to render. Every episode cell lists its section classes in a comment at the top - put one of those on the magic line to re-watch a single idea instead of the whole episode.

| # | episode |
|---|---|
| 1 | **The CLB** - which bit goes where, and why |
| 2 | **JTAG** - the TAP, the instruction register, the boundary ring |
| 3 | **The configuration engine** - one memory, two write paths, and the startup that gates them |
| 4 | **I/O, pads and the clock** -  |
| 5 | **BRAM** -  |
| 6 | **DSP** -  |
| 7 | **Routing** -  |
| 8 | **Synthesis with yosys** -  |
| 9 | **VPR and place-and-route** -  |
| 10 | **Bitstream generation** -  |
| 11 | **The whole flow, by hand and by tool** -  |
| 12 | **What is ours: bob against OpenFPGA and Aegis** -  |
| 13 | **Every error we hit, and the rule it became** -  |


In [ ]:
!pip install manim

In [ ]:
# run once: palette, helpers and the BobScene base class
# every episode cell below uses these

# =============================================================================
#  shared prelude - palette, helpers and the BobScene base class.
#  docs/manim/build.py pastes this into the top of every episode cell.
# =============================================================================

from manim import *
import numpy as np

# ---------------------------------------------------------------- palette ----
BG    = "#11121a"
INK   = "#e8e8ea"
DIM   = "#8b93a7"
C_PY  = "#7aa2f7"   # blue    - Python / tools / the device description
C_VPR = "#f7768e"   # red     - VPR / external tools
C_RTL = "#9ece6a"   # green   - hardware, Verilog, things on the die
C_BIT = "#e0af68"   # amber   - configuration bits, FASM, the bitstream
C_GRF = "#bb9af7"   # purple  - graphs, JTAG, protocol
C_ERR = "#ff7a93"   # pink    - bugs, refusals, errors
MONO  = "monospace"


# ---------------------------------------------------------------- helpers ----
def mono(s, size=22, color=INK):
    """One line of monospace text (Pango crashes on '', so blanks become ' ')."""
    return Text(s if s else " ", font=MONO, font_size=size, color=color)


def code_block(lines, size=20, color=INK):
    g = VGroup(*[mono(l, size, color) for l in lines])
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.14)
    return g


def panel(mob, color=DIM, pad=0.32, fill=0.06):
    r = SurroundingRectangle(mob, color=color, buff=pad)
    r.set_fill(color, opacity=fill)
    return VGroup(r, mob)


def chip(label, color, w=2.6, h=0.95, size=22, weight="NORMAL"):
    box = RoundedRectangle(width=w, height=h, corner_radius=0.14,
                           color=color, stroke_width=3)
    box.set_fill(color, opacity=0.12)
    txt = Text(label, font_size=size, color=INK, weight=weight, line_spacing=0.75)
    if txt.width > w - 0.3:
        txt.scale_to_fit_width(w - 0.3)
    if txt.height > h - 0.2:
        txt.scale_to_fit_height(h - 0.2)
    return VGroup(box, txt.move_to(box.get_center()))


def arrow(a, b, color=DIM, buff=0.15, sw=3):
    return Arrow(a, b, buff=buff, color=color, stroke_width=sw,
                 max_tip_length_to_length_ratio=0.18)


def mux_symbol(color=C_RTL, h=1.9, w=0.8):
    """Classic trapezoid multiplexer symbol."""
    p = Polygon([-w / 2,  h / 2, 0], [w / 2,  h / 2 - 0.3, 0],
                [ w / 2, -h / 2 + 0.3, 0], [-w / 2, -h / 2, 0],
                color=color, stroke_width=3)
    p.set_fill(color, opacity=0.14)
    return p


def bitcells(n, size=0.3, on=(), color=C_BIT, off_color=DIM):
    """A strip of n little squares; indices in `on` are filled."""
    g = VGroup()
    for i in range(n):
        s = Square(size, color=off_color, stroke_width=1.6)
        if i in on:
            s.set_stroke(color).set_fill(color, opacity=0.85)
        g.add(s)
    g.arrange(RIGHT, buff=0.035)
    return g


def fieldbar(fields, total_w=11.0, h=0.62, size=15):
    """
    fields: [(label, nbits, color), ...] -> one horizontal bar split to scale,
    each slice labelled above and its bit range below. Returns VGroup(bar, labels, ranges).
    """
    nbits = sum(f[1] for f in fields)
    bar, labs, rngs = VGroup(), VGroup(), VGroup()
    x, lo = -total_w / 2, 0
    for label, n, col in fields:
        w = max(total_w * n / nbits, 0.34)
        r = Rectangle(width=w, height=h, color=col, stroke_width=2)
        r.set_fill(col, opacity=0.28).move_to(np.array([x + w / 2, 0, 0]))
        bar.add(r)
        t = Text(label, font_size=size, color=col)
        if t.width > w * 1.9:
            t.scale_to_fit_width(max(w * 1.9, 0.5))
        t.next_to(r, UP, buff=0.14)
        labs.add(t)
        rt = mono(f"{lo}" if n == 1 else f"{lo}..{lo + n - 1}", size - 2, DIM)
        rt.next_to(r, DOWN, buff=0.12)
        if rt.width > w * 1.9:
            rt.scale_to_fit_width(max(w * 1.9, 0.5))
        rngs.add(rt)
        x += w
        lo += n
    return VGroup(bar, labs, rngs)


def filecard(path, role, color):
    """A small card naming a repo file and what it is."""
    t = mono(path, 17, color)
    r = Text(role, font_size=14, color=DIM)
    g = VGroup(t, r).arrange(DOWN, aligned_edge=LEFT, buff=0.08)
    box = SurroundingRectangle(g, color=color, buff=0.16)
    box.set_fill(color, opacity=0.07)
    return VGroup(box, g)


def mid(a, b):
    """midpoint, defined here so nothing depends on manim exporting space_ops."""
    return (a + b) / 2


def clear_all(sc, run_time=0.6):
    if sc.mobjects:
        sc.play(*[FadeOut(m) for m in sc.mobjects], run_time=run_time)


class BobScene(Scene):
    def setup(self):
        self.camera.background_color = BG

    def heading(self, text, kicker=None):
        t = Text(text, font_size=32, color=INK, weight="BOLD")
        t.to_corner(UL).shift(DOWN * 0.1)
        rule = Line(LEFT * 6.6, RIGHT * 6.6, color=DIM, stroke_width=1.5)
        rule.next_to(t, DOWN, buff=0.2).align_to(t, LEFT)
        g = VGroup(t, rule)
        self.play(FadeIn(t, shift=RIGHT * 0.3), Create(rule), run_time=0.7)
        if kicker:
            k = Text(kicker, font_size=19, color=DIM)
            if k.width > 13.0:
                k.scale_to_fit_width(13.0)
            k.next_to(rule, DOWN, buff=0.16).align_to(t, LEFT)
            g.add(k)
            self.play(FadeIn(k), run_time=0.4)
        return g

    def titlecard(self, number, title, subtitle):
        n = Text(number, font_size=26, color=C_BIT, weight="BOLD")
        t = Text(title, font_size=60, color=INK, weight="BOLD")
        s = Text(subtitle, font_size=26, color=DIM)
        if t.width > 12.5:
            t.scale_to_fit_width(12.5)
        if s.width > 12.5:
            s.scale_to_fit_width(12.5)
        g = VGroup(n, t, s).arrange(DOWN, buff=0.4)
        self.play(FadeIn(n), run_time=0.4)
        self.play(Write(t), run_time=1.1)
        self.play(FadeIn(s, shift=UP * 0.2), run_time=0.7)
        self.wait(1.6)
        self.play(FadeOut(g), run_time=0.6)

    def files_used(self, inputs, generated, verified):
        """Closing card: what this episode's topic is built from and checked by."""
        self.heading("Files", "what this part is written in, what is generated, and what proves it")
        cols = []
        for title, items, col in (("written by hand", inputs, C_RTL),
                                  ("generated", generated, C_PY),
                                  ("verified by", verified, C_BIT)):
            head = Text(title, font_size=21, color=col, weight="BOLD")
            cards = VGroup(*[filecard(p, r, col) for p, r in items])
            cards.arrange(DOWN, aligned_edge=LEFT, buff=0.18)
            g = VGroup(head, cards).arrange(DOWN, aligned_edge=LEFT, buff=0.28)
            cols.append(g)
        row = VGroup(*cols).arrange(RIGHT, buff=0.7, aligned_edge=UP)
        if row.width > 13.2:
            row.scale_to_fit_width(13.2)
        row.next_to(self.mobjects[1], DOWN, buff=0.55).set_x(0)
        for c in cols:
            self.play(FadeIn(c, shift=UP * 0.2), run_time=0.7)
        self.wait(2.4)

In [ ]:
%%manim -qm Ep01CLB
# ===========================================================================
#  EPISODE 1: The CLB
#  which bit goes where, and why
#
#  render one section instead, by putting its class on the magic line above:
#      E01S1Where
#      E01S2Datapath
#      E01S3Lut
#      E01S4Carry
#      E01S5Ff
#      E01S6Bits
#      E01S7Example
#      E01S8Files
# ===========================================================================

# =============================================================================
#  EPISODE 1 - the CLB: which bit goes where, and why
# =============================================================================

def s1_where(sc):
    sc.heading("Where the CLB sits",
               "bob today: a 14 x 12 VPR grid - an I/O ring around a 12 x 10 core of 100 CLBs")

    grid = VGroup()
    for x in range(14):
        for y in range(12):
            edge = x in (0, 13) or y in (0, 11)
            corner = (x in (0, 13)) and (y in (0, 11))
            if corner:
                continue
            if edge:
                col, op = C_GRF, 0.18
            elif x == 3:
                col, op = C_VPR, 0.22
            elif x == 8:
                col, op = C_BIT, 0.22
            else:
                col, op = C_RTL, 0.16
            s = Square(0.38, color=col, stroke_width=1.6).set_fill(col, opacity=op)
            s.move_to(np.array([-3.4 + x * 0.44, -2.5 + y * 0.44, 0]))
            grid.add(s)
    grid.shift(LEFT * 1.6 + DOWN * 0.25)
    sc.play(LaggedStart(*[FadeIn(s, scale=0.6) for s in grid], lag_ratio=0.004),
            run_time=2.0)

    key = VGroup(
        VGroup(Square(0.26, color=C_RTL).set_fill(C_RTL, 0.16),
               Text("100 CLBs", font_size=20, color=C_RTL)).arrange(RIGHT, buff=0.2),
        VGroup(Square(0.26, color=C_VPR).set_fill(C_VPR, 0.22),
               Text("2 BRAM  (column x = 3)", font_size=20, color=C_VPR)).arrange(RIGHT, buff=0.2),
        VGroup(Square(0.26, color=C_BIT).set_fill(C_BIT, 0.22),
               Text("2 DSP   (column x = 8)", font_size=20, color=C_BIT)).arrange(RIGHT, buff=0.2),
        VGroup(Square(0.26, color=C_GRF).set_fill(C_GRF, 0.18),
               Text("44 I/O pads", font_size=20, color=C_GRF)).arrange(RIGHT, buff=0.2),
    ).arrange(DOWN, aligned_edge=LEFT, buff=0.3)
    key.to_edge(RIGHT, buff=1.0).shift(DOWN * 0.2)
    sc.play(LaggedStart(*[FadeIn(k, shift=RIGHT * 0.2) for k in key], lag_ratio=0.2),
            run_time=1.4)
    sc.wait(1.0)

    one = grid[60]
    ring = Circle(radius=0.42, color=INK, stroke_width=3).move_to(one)
    sc.play(Create(ring), run_time=0.5)
    sc.play(FadeOut(key), run_time=0.4)

    ble = chip("one CLB\n= one BLE", C_RTL, 3.0, 1.5, 24)
    ble.to_edge(RIGHT, buff=1.4)
    sc.play(GrowArrow(arrow(ring.get_right(), ble.get_left(), INK, 0.2)),
            FadeIn(ble), run_time=0.8)

    parts = code_block([
        "one LUT6      (fracturable, O6 and O5)",
        "one carry bit (MUXCY + XORCY)",
        "one flip-flop (FDRE / FDSE)",
        "",
        "71 configuration bits.",
    ], 21, INK)
    parts[4].set_color(C_BIT)
    parts.next_to(ble, DOWN, buff=0.5).set_x(ble.get_x())
    sc.play(LaggedStart(*[FadeIn(p, shift=UP * 0.15) for p in parts], lag_ratio=0.25),
            run_time=1.8)

    note = Text("AMD calls this a slice's BLE. bob has exactly one per CLB - "
                "OpenFPGA's reference has ten.", font_size=19, color=DIM)
    note.scale_to_fit_width(12.6).to_edge(DOWN, buff=0.35)
    sc.play(FadeIn(note), run_time=0.7)
    sc.wait(2.2)


def s2_datapath(sc):
    sc.heading("The datapath", "hw/src/clb/clb.sv - four inputs from the routing, three ways out")

    lut = chip("LUT6", C_RTL, 1.9, 2.2, 26)
    lut.move_to(np.array([-3.4, 0.4, 0]))
    sc.play(FadeIn(lut), run_time=0.6)

    ins = VGroup()
    for k in range(6):
        y = 1.35 - k * 0.38
        ln = Line(np.array([-5.6, y, 0]), np.array([-4.35, y, 0]), color=DIM, stroke_width=2)
        t = mono(f"i[{k}]", 16, DIM).next_to(ln, LEFT, buff=0.1)
        ins.add(VGroup(ln, t))
    sc.play(LaggedStart(*[Create(i) for i in ins], lag_ratio=0.08), run_time=0.9)
    src = Text("from the connection box", font_size=17, color=C_GRF)
    src.next_to(ins, DOWN, buff=0.3)
    sc.play(FadeIn(src), run_time=0.5)

    o6 = mono("O6", 18, C_RTL)
    o5 = mono("O5", 18, C_RTL)
    o6.next_to(lut, RIGHT, buff=0.2).shift(UP * 0.5)
    o5.next_to(lut, RIGHT, buff=0.2).shift(DOWN * 0.5)
    sc.play(FadeIn(o6), FadeIn(o5), run_time=0.5)

    cy = chip("carry\nMUXCY\nXORCY", C_BIT, 1.7, 2.2, 19)
    cy.move_to(np.array([-0.6, 0.4, 0]))
    sc.play(FadeIn(cy),
            GrowArrow(arrow(o6.get_right(), cy.get_left() + UP * 0.5, DIM, 0.12)),
            GrowArrow(arrow(o5.get_right(), cy.get_left() + DOWN * 0.5, DIM, 0.12)),
            run_time=0.8)
    cin = arrow(np.array([-0.6, -1.5, 0]), np.array([-0.6, -0.75, 0]), C_BIT, 0.05)
    cint = mono("cin  (from the CLB below)", 16, C_BIT).next_to(cin, DOWN, buff=0.12)
    cout = arrow(np.array([-0.6, 1.55, 0]), np.array([-0.6, 2.3, 0]), C_BIT, 0.05)
    coutt = mono("cout  (to the CLB above)", 16, C_BIT).next_to(cout, UP, buff=0.1)
    sc.play(GrowArrow(cin), FadeIn(cint), GrowArrow(cout), FadeIn(coutt), run_time=0.7)

    ff = chip("flip-flop\nFDRE / FDSE", C_GRF, 2.3, 1.5, 19)
    ff.move_to(np.array([2.5, 0.4, 0]))
    sc.play(FadeIn(ff), GrowArrow(arrow(cy.get_right(), ff.get_left(), DIM, 0.12)),
            run_time=0.7)
    lab = mono("comb", 16, DIM)
    lab.move_to(mid(cy.get_right(), ff.get_left()) + UP * 0.22)
    sc.play(FadeIn(lab), run_time=0.3)

    omux = mux_symbol(C_RTL, 1.3, 0.6).move_to(np.array([4.6, 0.4, 0]))
    sc.play(Create(omux),
            GrowArrow(arrow(ff.get_right(), omux.get_left() + DOWN * 0.25, DIM, 0.12)),
            run_time=0.6)
    bypass = VMobject(color=DIM, stroke_width=2)
    bypass.set_points_as_corners([cy.get_right() + RIGHT * 0.1,
                                  np.array([1.6, -1.35, 0]),
                                  np.array([4.25, -1.35, 0]),
                                  omux.get_left() + UP * 0.28])
    sc.play(Create(bypass), run_time=0.7)
    oarr = arrow(omux.get_right(), np.array([6.1, 0.4, 0]), C_RTL, 0.08)
    ot = mono("o", 20, C_RTL).next_to(oarr, RIGHT, buff=0.1)
    sc.play(GrowArrow(oarr), FadeIn(ot), run_time=0.5)

    sel = mono("ff_en", 15, C_BIT).next_to(omux, DOWN, buff=0.22)
    sc.play(FadeIn(sel), run_time=0.4)

    ctrl = code_block([
        "ce  routed clock enable      sr  routed synchronous set/reset",
        "gce global user-clock enable gsr / gwe  startup",
    ], 18, DIM)
    ctrl.to_edge(DOWN, buff=0.35)
    sc.play(FadeIn(ctrl), run_time=0.7)
    sc.wait(2.4)


def s3_lut(sc):
    sc.heading("The LUT is a tree of 2-to-1 multiplexers",
               "hw/src/clb/lutk.sv - 64 configuration bits at the leaves, one answer at the root")

    lvl0 = VGroup(*[Square(0.3, color=C_BIT, stroke_width=1.5).set_fill(C_BIT, 0.5)
                    for _ in range(16)])
    lvl0.arrange(RIGHT, buff=0.06).shift(UP * 2.0)
    lab0 = mono("INIT  (the truth table)", 19, C_BIT).next_to(lvl0, UP, buff=0.2)
    sc.play(LaggedStart(*[FadeIn(s) for s in lvl0], lag_ratio=0.04), FadeIn(lab0),
            run_time=1.2)
    note = mono("16 shown - there are really 2**6 = 64", 16, DIM)
    note.next_to(lvl0, DOWN, buff=0.16)
    sc.play(FadeIn(note), run_time=0.4)

    levels = [lvl0]
    for lv in range(4):
        n = 16 >> (lv + 1)
        row = VGroup(*[mux_symbol(C_RTL, 0.34, 0.26) for _ in range(n)])
        row.arrange(RIGHT, buff=0.06 + lv * 0.42)
        row.move_to(np.array([0, 1.15 - lv * 0.78, 0]))
        edges = VGroup()
        for j, m in enumerate(row):
            for c in (2 * j, 2 * j + 1):
                edges.add(Line(levels[-1][c].get_bottom(), m.get_top(),
                               color=DIM, stroke_width=1.2))
        sel = mono(f"i[{lv}]", 16, C_GRF).next_to(row, LEFT, buff=0.35)
        sc.play(Create(edges), LaggedStart(*[Create(m) for m in row], lag_ratio=0.06),
                FadeIn(sel), run_time=0.8)
        levels.append(row)

    root = levels[-1][0]
    o6 = mono("O6  = INIT[ i[5:0] ]", 22, C_RTL)
    o6.next_to(root, DOWN, buff=0.5)
    sc.play(GrowArrow(arrow(root.get_bottom(), o6.get_top(), C_RTL, 0.12)),
            FadeIn(o6), run_time=0.7)
    sc.wait(1.0)

    frac = SurroundingRectangle(levels[3][0], color=C_VPR, buff=0.12)
    o5 = mono("O5  = the K-1 sub-tree, INIT[31:0] over i[4:0]", 20, C_VPR)
    o5.next_to(o6, DOWN, buff=0.3)
    sc.play(Create(frac), FadeIn(o5), run_time=0.8)
    why = Text("that free second output is UG474's LUT6_2 fracture - bob keeps it, "
               "and the router uses it for a second load", font_size=18, color=DIM)
    why.scale_to_fit_width(12.6).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(why), run_time=0.7)
    sc.wait(2.2)


def s4_carry(sc):
    sc.heading("The carry chain", "UG474's MUXCY and XORCY - and why it runs south to north")

    eqs = code_block([
        "prop   = O6                      the LUT computes 'propagate'",
        "di     = cy_di_sel ? O5 : i[0]   'generate'",
        "",
        "cout   = prop ? cin : di         MUXCY",
        "sum    = prop ^ cin              XORCY  -> the datapath",
    ], 22, INK)
    eqs[0].set_color(C_RTL); eqs[1].set_color(C_RTL)
    eqs[3].set_color(C_BIT); eqs[4].set_color(C_BIT)
    eqs.to_edge(LEFT, buff=0.8).shift(UP * 1.2)
    for l in eqs:
        sc.play(FadeIn(l, shift=RIGHT * 0.15), run_time=0.45)
    sc.wait(0.8)

    adder = Text("set prop = A xor B and di = A, and one CLB is one full-adder bit",
                 font_size=21, color=C_BIT)
    adder.next_to(eqs, DOWN, buff=0.55).align_to(eqs, LEFT)
    sc.play(FadeIn(adder), run_time=0.7)

    col = VGroup()
    for r in range(5):
        c = chip(f"CLB  y = {r + 1}", C_RTL, 2.4, 0.66, 18)
        col.add(c)
    col.arrange(DOWN, buff=0.42).to_edge(RIGHT, buff=1.1)
    sc.play(LaggedStart(*[FadeIn(c) for c in col], lag_ratio=0.12), run_time=1.0)

    links = VGroup(*[arrow(col[i].get_top(), col[i - 1].get_bottom(), C_BIT, 0.06)
                     for i in range(len(col) - 1, 0, -1)])
    sc.play(LaggedStart(*[GrowArrow(a) for a in links], lag_ratio=0.2), run_time=1.2)
    dirn = mono("carry runs up the column", 18, C_BIT)
    dirn.next_to(col, DOWN, buff=0.35)
    sc.play(FadeIn(dirn), run_time=0.5)

    pt = code_block([
        "in the routing graph this is a VPR 'direct':",
        "an IPIN driven by exactly one OPIN.",
        "No choice to make, so NO configuration bits -",
        "it is a plain wire in bob_fabric.v.",
    ], 19, C_GRF)
    pt.next_to(adder, DOWN, buff=0.6).align_to(eqs, LEFT)
    sc.play(LaggedStart(*[FadeIn(l) for l in pt], lag_ratio=0.2), run_time=1.4)

    cut = Text("a chain cannot cross a column, so the tools cut it to the column height "
               "and add a generator CLB", font_size=18, color=DIM)
    cut.scale_to_fit_width(12.8).to_edge(DOWN, buff=0.28)
    sc.play(FadeIn(cut), run_time=0.7)
    sc.wait(2.2)


def s5_ff(sc):
    sc.heading("The flip-flop, and who wins",
               "FDRE / FDSE semantics plus the two startup globals - the order matters")

    code = code_block([
        "always @(posedge clk) begin",
        "    if (gsr)                  q <= ff_rstval;",
        "    else if (gwe && gce) begin",
        "        if (sr_eff)           q <= ff_rstval;",
        "        else if (ce_eff)      q <= comb;",
        "    end",
        "end",
    ], 23, INK)
    code.to_edge(LEFT, buff=0.7).shift(UP * 0.9)
    sc.play(FadeIn(code), run_time=0.9)

    ladder = [
        ("gsr", "startup reset - beats everything, even GWE", C_ERR),
        ("gwe && gce", "frozen unless startup finished AND this is a user-clock tick", C_GRF),
        ("sr_eff", "synchronous set/reset - beats the clock enable (FDRE/FDSE)", C_BIT),
        ("ce_eff", "clock enable - the ordinary case", C_RTL),
    ]
    rows = VGroup()
    for i, (name, why, col) in enumerate(ladder):
        n = mono(name, 21, col)
        w = Text(why, font_size=17, color=DIM)
        r = VGroup(n, w).arrange(RIGHT, buff=0.35, aligned_edge=UP)
        rows.add(r)
    rows.arrange(DOWN, aligned_edge=LEFT, buff=0.26)
    if rows.width > 12.8:
        rows.scale_to_fit_width(12.8)
    rows.next_to(code, DOWN, buff=0.7).set_x(0)
    for i, r in enumerate(rows):
        box = SurroundingRectangle(code[i + 1], color=ladder[i][2], buff=0.06)
        sc.play(Create(box), FadeIn(r, shift=RIGHT * 0.2), run_time=0.6)
        sc.wait(0.5)
        sc.play(FadeOut(box), run_time=0.25)

    two = code_block([
        "ff_ce_en = 0  ->  ce_eff = 1     the FF ignores the routed CE",
        "ff_sr_en = 0  ->  sr_eff = 0     the FF ignores the routed SR",
        "ff_rstval     ->  0 = FDRE, 1 = FDSE, and the power-up value too",
    ], 19, C_BIT)
    two.to_edge(DOWN, buff=0.35)
    sc.play(FadeIn(two), run_time=0.8)
    sc.wait(2.2)


def s6_bits(sc):
    sc.heading("The 71 bits", "tools/bob/device.py names them once; clb_pkg.sv derives the same "
                              "layout independently and pytest compares the two")

    fields = [("INIT", 64, C_BIT), ("ff_en", 1, C_RTL), ("ff_rstval", 1, C_RTL),
              ("ff_ce_en", 1, C_GRF), ("ff_sr_en", 1, C_GRF),
              ("cy_en", 1, C_VPR), ("cy_di_sel", 1, C_VPR), ("ff_d_sel", 1, C_PY)]
    bar = fieldbar(fields, total_w=9.0, h=0.7, size=14)
    bar.shift(UP * 1.35)
    sc.play(Create(bar[0]), run_time=1.0)
    sc.play(LaggedStart(*[FadeIn(l) for l in bar[1]], lag_ratio=0.12), run_time=1.0)
    sc.play(LaggedStart(*[FadeIn(r) for r in bar[2]], lag_ratio=0.12), run_time=0.9)

    rows = [
        ("INIT[63:0]", "the truth table. O6 = INIT[i], O5 = INIT[i & 31]"),
        ("ff_en",      "o = the flip-flop (1) or straight combinational (0)"),
        ("ff_rstval",  "reset / power-up value: 0 = FDRE, 1 = FDSE"),
        ("ff_ce_en",   "honour the routed CE pin?"),
        ("ff_sr_en",   "honour the routed SR pin?"),
        ("cy_en",      "carry mode: datapath becomes XORCY's sum, cout becomes MUXCY"),
        ("cy_di_sel",  "carry generate comes from i[0] (0) or from O5 (1)"),
        ("ff_d_sel",   "datapath is O6 (0) or O5 (1) - ignored when cy_en"),
    ]
    tbl = VGroup()
    for name, why in rows:
        n = mono(name, 19, C_BIT)
        n.scale_to_fit_height(0.2)
        w = Text(why, font_size=17, color=DIM)
        tbl.add(VGroup(n, w).arrange(RIGHT, buff=0.3, aligned_edge=DOWN))
    tbl.arrange(DOWN, aligned_edge=LEFT, buff=0.19)
    if tbl.width > 12.6:
        tbl.scale_to_fit_width(12.6)
    tbl.next_to(bar, DOWN, buff=0.65).set_x(0)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.15) for r in tbl], lag_ratio=0.15),
            run_time=2.2)

    sc.wait(1.2)
    k4 = Text("K is a parameter: at K = 4 the same CLB is 16 + 7 = 23 bits, "
              "and make check runs the whole fabric that way too",
              font_size=18, color=C_PY)
    k4.scale_to_fit_width(12.6).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(k4), run_time=0.8)
    sc.wait(2.0)


def s7_example(sc):
    sc.heading("Worked example: a 2-input AND",
               "from a truth table to the bits that actually sit in the configuration memory")

    tt = code_block([
        "i[1] i[0]   o",
        "  0    0    0",
        "  0    1    0",
        "  1    0    0",
        "  1    1    1",
    ], 24, INK)
    tt[0].set_color(DIM)
    ttp = panel(tt, DIM)
    ttp.to_edge(LEFT, buff=1.0).shift(UP * 0.9)
    sc.play(FadeIn(ttp), run_time=0.8)

    step1 = code_block([
        "address i[5:0] = {0,0,0,0,i1,i0}",
        "INIT bit is 1 only when i1 & i0",
        "-> bits 3, 7, 11, 15, ... every",
        "   address whose low 2 bits are 11",
    ], 19, DIM)
    step1.next_to(ttp, DOWN, buff=0.5).align_to(ttp, LEFT)
    sc.play(FadeIn(step1), run_time=0.9)

    strip = bitcells(32, 0.26, on=[i for i in range(32) if i % 4 == 3])
    strip.to_edge(RIGHT, buff=0.7).shift(UP * 1.5)
    slab = mono("INIT[31:0]   (the other 32 bits repeat)", 17, DIM)
    slab.next_to(strip, UP, buff=0.18)
    sc.play(Create(strip), FadeIn(slab), run_time=1.1)

    val = mono("INIT = 64'h8888_8888_8888_8888", 26, C_BIT)
    val.next_to(strip, DOWN, buff=0.55)
    sc.play(Write(val), run_time=0.9)
    sc.wait(0.8)

    flags = code_block([
        "ff_en     = 0      combinational, no register",
        "ff_rstval = 0",
        "ff_ce_en  = 0      CE ignored",
        "ff_sr_en  = 0      SR ignored",
        "cy_en     = 0      not an adder",
        "cy_di_sel = 0",
        "ff_d_sel  = 0      take O6",
    ], 18, C_RTL)
    flags.next_to(val, DOWN, buff=0.5).set_x(val.get_x())
    sc.play(LaggedStart(*[FadeIn(f) for f in flags], lag_ratio=0.12), run_time=1.3)

    fasm = mono("clb_x2y3.init = 64'h8888888888888888", 22, C_PY)
    fasm.to_edge(DOWN, buff=0.75)
    ftag = Text("this is exactly what bob's FASM says, and bitgen turns it into "
                "71 bits at that tile's place in the chain",
                font_size=17, color=DIM)
    ftag.scale_to_fit_width(12.4).next_to(fasm, DOWN, buff=0.2)
    sc.play(FadeIn(fasm), FadeIn(ftag), run_time=0.9)
    sc.wait(2.4)


def s8_files(sc):
    sc.files_used(
        inputs=[("hw/src/clb/clb_pkg.sv", "field offsets, derived from K"),
                ("hw/src/clb/lutk.sv", "the mux tree, K a parameter"),
                ("hw/src/clb/clb.sv", "LUT -> carry -> flip-flop")],
        generated=[("hw/src/generated/bob_params.vh", "BOB_CLB_* offsets"),
                   ("tools/bob/device.json", "the same fields, for Python"),
                   ("tools/bob/model.py", "the cycle model of one CLB")],
        verified=[("hw/tb/tb_clb.sv", "7040 checks vs model.py"),
                  ("tests/test_lutk.py", "lutk(6) == the proven lut6.sv"),
                  ("tests/test_device.py", "clb_pkg.sv == bob_params.vh in iverilog"),
                  ("sim/mutate_fabric.sh", "no-gsr, no-gwe-freeze, ce-not-routed, ...")])


EP01 = [s1_where, s2_datapath, s3_lut, s4_carry, s5_ff, s6_bits, s7_example, s8_files]


class Ep01CLB(BobScene):
    def construct(self):
        self.titlecard("EPISODE 1", "The CLB",
                       "which bit goes where, and why")
        for i, part in enumerate(EP01):
            part(self)
            if i < len(EP01) - 1:
                clear_all(self)


class E01S1Where(BobScene):
    def construct(self): s1_where(self)


class E01S2Datapath(BobScene):
    def construct(self): s2_datapath(self)


class E01S3Lut(BobScene):
    def construct(self): s3_lut(self)


class E01S4Carry(BobScene):
    def construct(self): s4_carry(self)


class E01S5Ff(BobScene):
    def construct(self): s5_ff(self)


class E01S6Bits(BobScene):
    def construct(self): s6_bits(self)


class E01S7Example(BobScene):
    def construct(self): s7_example(self)


class E01S8Files(BobScene):
    def construct(self): s8_files(self)

In [ ]:
%%manim -qm Ep02JTAG
# ===========================================================================
#  EPISODE 2: JTAG
#  the TAP, the instruction register, the boundary ring
#
#  render one section instead, by putting its class on the magic line above:
#      E02S1Why
#      E02S2Tap
#      E02S3Timing
#      E02S4Ir
#      E02S5Status
#      E02S6Boundary
#      E02S7Files
# ===========================================================================

# =============================================================================
#  EPISODE 2 - JTAG: the TAP, the instruction register, the boundary ring
# =============================================================================

def s1_why(sc):
    sc.heading("Four wires and a state machine",
               "everything that ever reaches bob - configuration, readback, test - goes through here")

    mac = chip("Mac\nhost/*.py", C_PY, 2.4, 1.3, 21).move_to(np.array([-5.0, 1.4, 0]))
    pico = chip("Pico\nDirtyJTAG", C_VPR, 2.4, 1.3, 21).move_to(np.array([-1.6, 1.4, 0]))
    pl = chip("XC7Z020 PL\nbob", C_RTL, 2.8, 1.3, 21).move_to(np.array([2.6, 1.4, 0]))
    sc.play(FadeIn(mac), run_time=0.4)
    sc.play(GrowArrow(arrow(mac.get_right(), pico.get_left(), DIM, 0.08)),
            FadeIn(pico), run_time=0.5)
    usb = mono("USB", 15, DIM).move_to(mid(mac.get_right(), pico.get_left()) + UP * 0.25)
    sc.play(FadeIn(usb), run_time=0.3)

    wires = VGroup()
    names = [("TCK", "the only clock in the whole chip"),
             ("TMS", "walks the state machine"),
             ("TDI", "data in"),
             ("TDO", "data out")]
    for k, (n, why) in enumerate(names):
        y = 1.4 + 0.45 - k * 0.3
        ln = Line(pico.get_right() + RIGHT * 0.05, np.array([1.2, y, 0]),
                  color=C_GRF, stroke_width=2.2).set_y(y)
        ln.put_start_and_end_on(np.array([-0.35, y, 0]), np.array([1.2, y, 0]))
        t = mono(n, 15, C_GRF).next_to(ln, RIGHT, buff=0.08)
        wires.add(VGroup(ln, t))
    sc.play(FadeIn(pl), LaggedStart(*[Create(w) for w in wires], lag_ratio=0.15),
            run_time=1.2)

    tbl = VGroup()
    for n, why in names:
        tbl.add(VGroup(mono(n, 19, C_GRF), Text(why, font_size=17, color=DIM))
                .arrange(RIGHT, buff=0.3, aligned_edge=DOWN))
    tbl.arrange(DOWN, aligned_edge=LEFT, buff=0.22)
    tbl.next_to(pl, DOWN, buff=1.0).set_x(-0.6)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.2) for r in tbl], lag_ratio=0.2),
            run_time=1.4)

    big = Text("TCK clocks the entire chip. There is no free-running core clock for a "
               "scan to race against - one clock domain, no CDC to get wrong.",
               font_size=20, color=C_BIT)
    big.scale_to_fit_width(12.8).to_edge(DOWN, buff=0.35)
    sc.play(FadeIn(big), run_time=0.9)
    sc.wait(2.2)


def s2_tap(sc):
    sc.heading("The TAP state machine (IEEE 1149.1)",
               "sixteen states, and TMS alone decides which one you are in")

    def st(label, x, y, col=DIM):
        b = RoundedRectangle(width=1.85, height=0.46, corner_radius=0.1,
                             color=col, stroke_width=2).set_fill(col, opacity=0.10)
        t = Text(label, font_size=15, color=INK)
        if t.width > 1.7:
            t.scale_to_fit_width(1.7)
        return VGroup(b.move_to(np.array([x, y, 0])), t.move_to(np.array([x, y, 0])))

    tlr = st("Test-Logic-Reset", -4.6, 2.35, C_ERR)
    rti = st("Run-Test/Idle", -4.6, 1.35, C_BIT)
    sc.play(FadeIn(tlr), FadeIn(rti),
            GrowArrow(arrow(tlr.get_bottom(), rti.get_top(), DIM, 0.05)), run_time=0.7)

    ys = [1.35, 0.55, -0.25, -1.05, -1.85, -2.65, -3.3]
    dr_names = ["Select-DR", "Capture-DR", "Shift-DR", "Exit1-DR",
                "Pause-DR", "Exit2-DR", "Update-DR"]
    ir_names = [n.replace("DR", "IR") for n in dr_names]
    dr = VGroup(*[st(n, -1.2, y, C_GRF) for n, y in zip(dr_names, ys)])
    ir = VGroup(*[st(n, 2.9, y, C_PY) for n, y in zip(ir_names, ys)])

    sc.play(LaggedStart(*[FadeIn(s) for s in dr], lag_ratio=0.1), run_time=1.0)
    sc.play(LaggedStart(*[FadeIn(s) for s in ir], lag_ratio=0.1), run_time=1.0)
    e1 = VGroup(*[arrow(dr[i].get_bottom(), dr[i + 1].get_top(), DIM, 0.04, 2)
                  for i in range(6)])
    e2 = VGroup(*[arrow(ir[i].get_bottom(), ir[i + 1].get_top(), DIM, 0.04, 2)
                  for i in range(6)])
    sc.play(Create(e1), Create(e2),
            GrowArrow(arrow(rti.get_right(), dr[0].get_left(), DIM, 0.05, 2)),
            GrowArrow(arrow(dr[0].get_right(), ir[0].get_left(), DIM, 0.05, 2)),
            run_time=1.0)

    lg = code_block(["TMS = 1 moves you along;  TMS = 0 holds or drops in",
                     "five 1s from anywhere always land in Test-Logic-Reset"], 18, DIM)
    lg.to_edge(DOWN, buff=0.28)
    sc.play(FadeIn(lg), run_time=0.7)
    sc.wait(1.0)

    walk = [rti, dr[0], dr[1], dr[2], dr[3], dr[6]]
    cap = Text("one DR scan", font_size=22, color=C_BIT).move_to(np.array([5.6, 2.2, 0]))
    sc.play(FadeIn(cap), run_time=0.4)
    notes = ["idle", "select", "CAPTURE: load the register",
             "SHIFT: one bit per TCK", "leave", "UPDATE: commit"]
    prev = None
    for s, n in zip(walk, notes):
        box = SurroundingRectangle(s, color=C_BIT, buff=0.06)
        t = Text(n, font_size=18, color=C_BIT).move_to(np.array([5.6, s.get_y(), 0]))
        if t.width > 3.2:
            t.scale_to_fit_width(3.2)
        sc.play(Create(box), FadeIn(t), run_time=0.45)
        if prev:
            sc.play(FadeOut(prev[0]), FadeOut(prev[1]), run_time=0.18)
        prev = (box, t)
        sc.wait(0.45)
    sc.wait(1.6)


def s3_timing(sc):
    sc.heading("The timing contract",
               "proven on hardware and never changed since the original bob - one edge wrong and nothing answers")

    n, w, h, x0, y0 = 6, 0.95, 0.55, -5.4, 1.5
    pts = [np.array([x0, y0, 0])]
    for i in range(n):
        x = x0 + i * w
        pts += [np.array([x + w / 2, y0, 0]), np.array([x + w / 2, y0 + h, 0]),
                np.array([x + w, y0 + h, 0]), np.array([x + w, y0, 0])]
    clk = VMobject(color=C_GRF, stroke_width=3)
    clk.set_points_as_corners(pts)
    lab = mono("TCK", 20, C_GRF).next_to(clk, LEFT, buff=0.25).set_y(y0 + h / 2)
    sc.play(Create(clk), FadeIn(lab), run_time=1.2)

    rise = VGroup(*[DashedLine(np.array([x0 + (i + 0.5) * w, y0 + h + 0.35, 0]),
                               np.array([x0 + (i + 0.5) * w, -2.4, 0]),
                               color=C_BIT, stroke_width=1.6, dash_length=0.08)
                    for i in range(n)])
    fall = VGroup(*[DashedLine(np.array([x0 + (i + 1) * w, y0 + h + 0.35, 0]),
                               np.array([x0 + (i + 1) * w, -2.4, 0]),
                               color=C_VPR, stroke_width=1.6, dash_length=0.08)
                    for i in range(n - 1)])
    sc.play(Create(rise), run_time=0.7)
    r1 = Text("RISING edge", font_size=20, color=C_BIT)
    r1.next_to(clk, UP, buff=0.5).set_x(-2.0)
    sc.play(FadeIn(r1), run_time=0.4)
    rl = code_block([
        "TMS is sampled here  -> the state machine advances",
        "TDI is sampled here  -> one bit enters the shift register",
    ], 19, C_BIT)
    rl.next_to(clk, DOWN, buff=0.85).set_x(0)
    sc.play(FadeIn(rl), run_time=0.8)
    sc.wait(0.8)

    sc.play(Create(fall), run_time=0.7)
    f1 = Text("FALLING edge", font_size=20, color=C_VPR)
    f1.next_to(clk, UP, buff=0.5).set_x(2.6)
    sc.play(FadeIn(f1), run_time=0.4)
    fl = code_block([
        "TDO is launched here          - half a cycle of margin",
        "IR / DR update latches fire   - the boundary moves as a unit",
        "the configuration memory takes its frame",
    ], 19, C_VPR)
    fl.next_to(rl, DOWN, buff=0.45).align_to(rl, LEFT)
    sc.play(FadeIn(fl), run_time=0.9)

    lsb = Text("Every shift register is LSB-first: chain bit k is the k-th bit shifted in.",
               font_size=21, color=INK)
    lsb.to_edge(DOWN, buff=0.35)
    sc.play(FadeIn(lsb), run_time=0.7)
    sc.wait(2.2)


def s4_ir(sc):
    sc.heading("A 6-bit instruction register",
               "AMD 7-series codes where they exist, so urjtag's discovery maps bob with no database entry")

    rows = [
        ("001001", "IDCODE",     "0xFBEEF093 - which build is in the PL", C_GRF),
        ("001000", "USERCODE",   "the milestone number", C_GRF),
        ("000001", "SAMPLE",     "watch the pins, touch nothing", C_PY),
        ("100110", "EXTEST",     "the boundary drives the pads", C_PY),
        ("000111", "INTEST",     "the boundary drives the FABRIC   (private)", C_PY),
        ("000010", "USER1",      "ce / sr / cin / step + 16 CLB outputs", C_BIT),
        ("000011", "USER2 = CFG_CTRL", "chain CRC and status, behind a write key", C_BIT),
        ("100010", "USER3 = CAPTURE",  "a snapshot of all 100 CLB outputs", C_BIT),
        ("100011", "USER4",      "BRAM contents / drive / SELECT", C_BIT),
        ("101000", "DSP",        "DSP drive word and both P values   (private)", C_BIT),
        ("000101", "CFG_IN",     "frame packets in", C_VPR),
        ("000100", "CFG_OUT",    "frame packets / readback out", C_VPR),
        ("110101", "CHAIN_IN",   "the whole memory as one scan   (private)", C_VPR),
        ("110100", "CHAIN_OUT",  "the same, read back   (private)", C_VPR),
        ("001011", "JPROGRAM",   "clear the configuration, drop DONE", C_ERR),
        ("001100", "JSTART",     "step the startup sequence", C_ERR),
        ("111111", "BYPASS",     "one flip-flop", DIM),
    ]
    g = VGroup()
    for code, name, why, col in rows:
        c = mono(code, 16, col)
        n = mono(name, 16, INK)
        n.align_to(c, DOWN)
        w = Text(why, font_size=14, color=DIM)
        row = VGroup(c, n, w).arrange(RIGHT, buff=0.3, aligned_edge=DOWN)
        g.add(row)
    for r in g:
        r[1].align_to(g[0][1], LEFT).shift(RIGHT * 0.9)
        r[2].align_to(g[0][2], LEFT).shift(RIGHT * 3.4)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.12)
    g.scale_to_fit_height(5.3).next_to(sc.mobjects[1], DOWN, buff=0.5).set_x(-0.6)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.15) for r in g], lag_ratio=0.07),
            run_time=3.0)
    sc.wait(1.4)

    note = Text("unknown codes fall back to BYPASS - never to something that writes",
                font_size=19, color=DIM)
    note.to_edge(DOWN, buff=0.25)
    sc.play(FadeIn(note), run_time=0.7)
    sc.wait(1.8)


def s5_status(sc):
    sc.heading("Every IR scan is also a status read",
               "Capture-IR loads status instead of zeros - in the spirit of 7-series parts")

    bits = [("DONE", C_RTL), ("INIT_B", C_GRF), ("COMMITTED", C_BIT),
            ("CRC_ERR", C_ERR), ("0", DIM), ("1", DIM)]
    cells = VGroup()
    for name, col in bits:
        b = Square(1.0, color=col, stroke_width=2.5).set_fill(col, opacity=0.18)
        t = Text(name, font_size=15, color=col)
        if t.width > 0.9:
            t.scale_to_fit_width(0.9)
        cells.add(VGroup(b, t.move_to(b)))
    cells.arrange(RIGHT, buff=0.12).shift(UP * 1.2)
    idx = VGroup(*[mono(str(5 - i), 16, DIM).next_to(c, UP, buff=0.14)
                   for i, c in enumerate(cells)])
    sc.play(LaggedStart(*[FadeIn(c, scale=0.7) for c in cells], lag_ratio=0.12),
            FadeIn(idx), run_time=1.3)

    exp = code_block([
        "DONE       startup finished, LD3 is lit",
        "INIT_B     no CRC / length / packet error",
        "COMMITTED  a good load is in the memory",
        "CRC_ERR    the last load was rejected",
        "the two low bits are 01, which IEEE 1149.1 requires",
    ], 20, INK)
    exp[4].set_color(DIM)
    exp.next_to(cells, DOWN, buff=0.7).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l, shift=RIGHT * 0.15) for l in exp], lag_ratio=0.2),
            run_time=1.8)

    why = Text("so a host can ask 'are you alive, and did my bitstream take?' "
               "without selecting any data register at all",
               font_size=20, color=C_BIT)
    why.scale_to_fit_width(12.6).to_edge(DOWN, buff=0.4)
    sc.play(FadeIn(why), run_time=0.8)
    sc.wait(2.2)


def s6_boundary(sc):
    sc.heading("The boundary ring", "hw/src/core/bsc_cell.v - two flip-flops per cell, "
                                    "and that split is the whole point")

    dpin = mono("data_in", 17, DIM).move_to(np.array([-5.6, 1.3, 0]))
    dout = mono("data_out", 17, DIM).move_to(np.array([3.6, 1.3, 0]))
    wire = Line(np.array([-4.8, 1.3, 0]), np.array([2.2, 1.3, 0]),
                color=DIM, stroke_width=2)
    m = mux_symbol(C_RTL, 1.1, 0.5).move_to(np.array([2.6, 1.3, 0]))
    sc.play(FadeIn(dpin), Create(wire), Create(m), FadeIn(dout), run_time=0.9)

    cap = chip("capture_reg", C_GRF, 2.4, 0.8, 19).move_to(np.array([-2.0, -0.4, 0]))
    upd = chip("update_reg", C_BIT, 2.4, 0.8, 19).move_to(np.array([1.2, -0.4, 0]))
    sc.play(FadeIn(cap), FadeIn(upd),
            GrowArrow(arrow(cap.get_right(), upd.get_left(), DIM, 0.08)), run_time=0.8)
    sc.play(GrowArrow(arrow(np.array([-2.0, 1.2, 0]), cap.get_top(), DIM, 0.08)),
            GrowArrow(arrow(upd.get_top(), m.get_bottom(), DIM, 0.1)), run_time=0.6)

    si = arrow(np.array([-4.6, -0.4, 0]), cap.get_left(), C_GRF, 0.08)
    sit = mono("scan_in", 16, C_GRF).next_to(si, LEFT, buff=0.1)
    so = arrow(cap.get_bottom(), np.array([-2.0, -1.6, 0]), C_GRF, 0.08)
    sot = mono("scan_out", 16, C_GRF).next_to(so, DOWN, buff=0.1)
    sc.play(GrowArrow(si), FadeIn(sit), GrowArrow(so), FadeIn(sot), run_time=0.6)

    exp = code_block([
        "capture_reg  is IN the scan chain. It samples the system value at Capture-DR,",
        "             then shifts - so shifting never disturbs what the cell drives.",
        "update_reg   holds what the cell drives. It only moves at Update-DR, so the",
        "             whole boundary changes at once, at a point the test controls.",
    ], 18, INK)
    exp.next_to(VGroup(cap, upd), DOWN, buff=1.0).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in exp], lag_ratio=0.2), run_time=1.8)
    sc.wait(1.4)

    sc.play(FadeOut(VGroup(dpin, dout, wire, m, cap, upd, si, sit, so, sot, exp)),
            run_time=0.6)

    modes = VGroup()
    for k, (n, c, w) in enumerate([
            ("SAMPLE", C_PY, "drives nothing, just watches - the fabric runs from the real switches"),
            ("EXTEST", C_VPR, "the boundary drives the PADS; the fabric is isolated"),
            ("INTEST", C_BIT, "the boundary drives the FABRIC's inputs and captures its outputs")]):
        modes.add(VGroup(mono(n, 24, c), Text(w, font_size=18, color=DIM))
                  .arrange(RIGHT, buff=0.4, aligned_edge=DOWN))
    modes.arrange(DOWN, aligned_edge=LEFT, buff=0.4).set_x(0).shift(UP * 0.7)
    if modes.width > 12.8:
        modes.scale_to_fit_width(12.8)
    sc.play(LaggedStart(*[FadeIn(m2, shift=RIGHT * 0.2) for m2 in modes], lag_ratio=0.25),
            run_time=1.6)

    size = code_block([
        "44 pads  x  2 cells  =  88 boundary cells",
        "cell k      = pad k's OUTPUT cell (fabric -> world, forced 0 while GTS)",
        "cell 44 + k = pad k's INPUT  cell (world -> fabric)",
    ], 19, C_GRF)
    size.next_to(modes, DOWN, buff=0.7).set_x(0)
    sc.play(FadeIn(size), run_time=0.9)

    gotcha = Text("Gotcha: a BC_1 cell captures its system input in EVERY mode - "
                  "so in INTEST the input cells read the real switches, not your vector.",
                  font_size=18, color=C_ERR)
    gotcha.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(gotcha), run_time=0.9)
    sc.wait(2.4)


def s7_files(sc):
    sc.files_used(
        inputs=[("hw/src/core/jtag_tap6.v", "state machine, 6-bit IR, DR select"),
                ("hw/src/core/bsc_cell.v", "one BC_1 boundary cell"),
                ("hw/src/fabric/bob_fpga.v", "wires the ring, 2 cells per pad")],
        generated=[("hw/src/generated/bob_params.vh", "BOB_BSR_W = 88, pad numbers"),
                   ("tools/bob/device.json", "bsr cells, board pad map")],
        verified=[("hw/tb/tb_bob.v", "IR, BYPASS, boundary, INTEST/EXTEST"),
                  ("host/dirtyjtag.py", "the same shifts on real hardware"),
                  ("docs/hwtest/*.md", "idcode + bypass open every board run")])


EP02 = [s1_why, s2_tap, s3_timing, s4_ir, s5_status, s6_boundary, s7_files]


class Ep02JTAG(BobScene):
    def construct(self):
        self.titlecard("EPISODE 2", "JTAG",
                       "the TAP, the instruction register, the boundary ring")
        for i, part in enumerate(EP02):
            part(self)
            if i < len(EP02) - 1:
                clear_all(self)


class E02S1Why(BobScene):
    def construct(self): s1_why(self)


class E02S2Tap(BobScene):
    def construct(self): s2_tap(self)


class E02S3Timing(BobScene):
    def construct(self): s3_timing(self)


class E02S4Ir(BobScene):
    def construct(self): s4_ir(self)


class E02S5Status(BobScene):
    def construct(self): s5_status(self)


class E02S6Boundary(BobScene):
    def construct(self): s6_boundary(self)


class E02S7Files(BobScene):
    def construct(self): s7_files(self)

In [ ]:
%%manim -qm Ep03Config
# ===========================================================================
#  EPISODE 3: The configuration engine
#  one memory, two write paths, and the startup that gates them
#
#  render one section instead, by putting its class on the magic line above:
#      E03S1Memory
#      E03S2Layout
#      E03S3Chain
#      E03S4Frames
#      E03S5Crc
#      E03S6Startup
#      E03S7Guards
#      E03S8Files
# ===========================================================================

# =============================================================================
#  EPISODE 3 - the configuration engine: memory, two write paths, startup
# =============================================================================

def s1_memory(sc):
    sc.heading("The configuration memory",
               "one array of flip-flops that every mux, LUT and flag in the fabric reads")

    big = mono("18 560 bits", 54, C_BIT).shift(UP * 1.9)
    sc.play(Write(big), run_time=1.0)

    split = code_block([
        "=  145 frames  x  4 words  x  32 bits",
    ], 26, INK)
    split.next_to(big, DOWN, buff=0.45)
    sc.play(FadeIn(split), run_time=0.7)

    frames = VGroup()
    for i in range(29):
        r = Rectangle(width=0.36, height=0.75, color=C_BIT, stroke_width=1.6)
        r.set_fill(C_BIT, opacity=0.18)
        frames.add(r)
    frames.arrange(RIGHT, buff=0.05).next_to(split, DOWN, buff=0.7)
    dots = mono("...  145 of them", 18, DIM).next_to(frames, RIGHT, buff=0.25)
    sc.play(LaggedStart(*[FadeIn(f) for f in frames], lag_ratio=0.03), FadeIn(dots),
            run_time=1.3)

    one = frames[4]
    z = SurroundingRectangle(one, color=INK, buff=0.06)
    words = VGroup(*[Rectangle(width=0.9, height=0.5, color=C_BIT, stroke_width=2)
                     .set_fill(C_BIT, opacity=0.25) for _ in range(4)])
    words.arrange(RIGHT, buff=0.12).next_to(frames, DOWN, buff=0.9).set_x(0)
    wl = VGroup(*[mono(f"w{i}", 15, DIM).next_to(w, DOWN, buff=0.1)
                  for i, w in enumerate(words)])
    sc.play(Create(z), run_time=0.4)
    sc.play(*[TransformFromCopy(one, w) for w in words], FadeIn(wl), run_time=1.0)
    fb = mono("one frame = 128 bits", 20, C_BIT).next_to(words, UP, buff=0.3)
    sc.play(FadeIn(fb), run_time=0.5)

    who = code_block([
        "what lives in those bits:",
        "   100 CLBs   x 71   = 7100",
        "   3391 routing muxes      (2 or 3 bits each)",
        "   2 BRAM x 8,  2 DSP x 16",
        "   1 ctrl tile x 8         (the user clock)",
        "   ... padded to whole frames",
    ], 19, INK)
    who[0].set_color(DIM)
    who.to_edge(DOWN, buff=0.3).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in who], lag_ratio=0.15), run_time=1.6)
    sc.wait(2.2)


def s2_layout(sc):
    sc.heading("Where a tile's bits actually are",
               "frames are column-major, like a 7-series device - and the chain is just all frames end to end")

    cols = VGroup()
    labels = ["ctrl", "x=0", "x=1", "x=2", "x=3", "x=4", "...", "x=13"]
    counts = ["1", "2", "13", "13", "6", "13", "", "1"]
    for lab, cnt in zip(labels, counts):
        h = 3.0 if lab not in ("ctrl", "x=0", "x=13", "...") else 1.4
        r = Rectangle(width=1.12, height=h, color=C_BIT, stroke_width=2)
        r.set_fill(C_BIT, opacity=0.16)
        t = Text(lab, font_size=16, color=INK)
        c = mono(cnt + (" frames" if cnt else ""), 13, DIM)
        g = VGroup(r, t.move_to(r.get_center() + UP * (h / 2 - 0.3)),
                   c.move_to(r.get_center() + DOWN * (h / 2 - 0.25)))
        cols.add(g)
    cols.arrange(RIGHT, buff=0.12, aligned_edge=DOWN).shift(UP * 0.85)
    far = VGroup(*[mono(f"FAR col {i}", 12, C_GRF).next_to(c, UP, buff=0.14)
                   for i, c in enumerate(cols)])
    sc.play(LaggedStart(*[FadeIn(c, shift=UP * 0.2) for c in cols], lag_ratio=0.1),
            run_time=1.4)
    sc.play(FadeIn(far), run_time=0.6)

    rule = code_block([
        "FAR column 0      the 8-bit ctrl tile, padded to one whole frame",
        "FAR column x + 1  VPR grid column x: tiles bottom to top, each one's",
        "                  block fields first, then its routing muxes by node id",
    ], 19, INK)
    rule.next_to(cols, DOWN, buff=0.7).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in rule], lag_ratio=0.2), run_time=1.5)

    key = Text("So the scan chain IS all the frames concatenated: chain bit k = memory bit k. "
               "One .bit word loads identically down either path.",
               font_size=20, color=C_BIT)
    key.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.35)
    sc.play(FadeIn(key), run_time=0.9)
    sc.wait(1.6)

    bug = Text("Bug we paid for: the first attempt made the BLOCK order column-major too, "
               "which broke tb_bob's counter probes. Frames are column-major; blocks stay row-major.",
               font_size=17, color=C_ERR)
    bug.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.3)
    sc.play(FadeOut(key), FadeIn(bug), run_time=0.8)
    sc.wait(2.0)


def s3_chain(sc):
    sc.heading("Write path A: the scan chain",
               "CHAIN_IN - one enormous DR scan, streamed through a single 128-bit buffer")

    tdi = mono("TDI", 20, C_GRF).move_to(np.array([-6.0, 1.5, 0]))
    buf = VGroup(*[Square(0.26, color=C_GRF, stroke_width=1.6) for _ in range(16)])
    buf.arrange(RIGHT, buff=0.04).move_to(np.array([-2.4, 1.5, 0]))
    bl = mono("one 128-bit frame buffer", 17, C_GRF).next_to(buf, UP, buff=0.2)
    sc.play(FadeIn(tdi), Create(buf), FadeIn(bl),
            GrowArrow(arrow(tdi.get_right(), buf.get_left(), DIM, 0.15)), run_time=0.9)

    mem = VGroup(*[Rectangle(width=0.5, height=0.42, color=C_BIT, stroke_width=1.6)
                   .set_fill(C_BIT, opacity=0.14) for _ in range(6)])
    mem.arrange(DOWN, buff=0.07).move_to(np.array([2.6, 1.1, 0]))
    ml = mono("the memory,\nframe by frame", 16, C_BIT).next_to(mem, RIGHT, buff=0.3)
    sc.play(Create(mem), FadeIn(ml), run_time=0.7)

    for k in range(3):
        sc.play(buf.animate.set_fill(C_GRF, opacity=0.55), run_time=0.35)
        sc.play(TransformFromCopy(buf, mem[k]),
                mem[k].animate.set_fill(C_BIT, opacity=0.8), run_time=0.4)
        sc.play(buf.animate.set_fill(C_GRF, opacity=0.0), run_time=0.2)
    every = mono("every 128th bit completes a frame, written on the next falling edge",
                 18, DIM)
    every.next_to(mem, DOWN, buff=0.7).set_x(0)
    sc.play(FadeIn(every), run_time=0.7)

    guards = code_block([
        "and it only lands if all of this holds:",
        "   GWE = 0            a running design is never written underneath itself",
        "   bit count == W     exactly 18 560 bits were shifted",
        "   CRC-32C matches    reflected 0x82F63B78, init and final XOR 0xFFFFFFFF",
        "                      check value CRC('123456789') = 0xE3069283",
    ], 19, INK)
    guards[0].set_color(DIM)
    guards[1].set_color(C_ERR); guards[2].set_color(C_ERR); guards[3].set_color(C_BIT)
    guards.next_to(every, DOWN, buff=0.5).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in guards], lag_ratio=0.18), run_time=1.8)

    why = Text("The non-zero CRC seed is deliberate: a stuck-low TDI cannot produce a "
               "matching all-zero chain.", font_size=19, color=C_BIT)
    why.scale_to_fit_width(12.8).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(why), run_time=0.8)
    sc.wait(2.2)


def s4_frames(sc):
    sc.heading("Write path B: frames and packets",
               "UG470 chapter 5, simplified - this is what ./bob load sends by default")

    stream = code_block([
        "FFFFFFFF FFFFFFFF     dummy",
        "AA995566              the sync word - the controller hunts for it BIT by bit",
        "20000000              NOP",
        "30008001 00000007     CMD    <- RCRC     reset the CRC",
        "30018001 FBEEF093     IDCODE <- must match, or nothing is written",
        "30002001 00000000     FAR    <- column 0, minor 0",
        "30008001 00000001     CMD    <- WCFG     arm frame writes",
        "30004000 50000244     FDRI, type-2, count = 4 x 145",
        "<580 words>           the frames. FAR auto-increments.",
        "30002001 00800000     FAR    <- BRAM 0            (M15)",
        "30004000 50000400     FDRI, 1024 words            contents",
        "30000001 <CRC>        CRC    <- expected",
        "30008001 00000003     CMD    <- LFRM",
        "30008001 00000005     CMD    <- START   needs CRC_OK after the last FDRI",
        "30008001 0000000D     CMD    <- DESYNC",
    ], 17, INK)
    stream[1].set_color(C_GRF)
    for i in (4, 11, 13):
        stream[i].set_color(C_BIT)
    stream.next_to(sc.mobjects[1], DOWN, buff=0.4).to_edge(LEFT, buff=0.7)
    sc.play(LaggedStart(*[FadeIn(l, shift=RIGHT * 0.1) for l in stream], lag_ratio=0.1),
            run_time=3.0)

    hdr = code_block([
        "type 1   001 op reg[17:13] .. count[10:0]",
        "type 2   010 op ............ count[26:0]",
        "",
        "registers",
        "  0 CRC   1 FAR   2 FDRI",
        "  3 FDRO  4 CMD   7 STAT",
        " 12 IDCODE",
        "",
        "commands",
        "  0 NULL   1 WCFG   3 LFRM",
        "  4 RCFG   5 START  7 RCRC",
        "  8 AGHIGH 13 DESYNC",
    ], 17, C_VPR)
    hdrp = panel(hdr, C_VPR)
    hdrp.to_edge(RIGHT, buff=0.5).set_y(0.1)
    sc.play(FadeIn(hdrp), run_time=0.9)
    sc.wait(1.4)

    far = code_block([
        "FAR  [25:23] block type   000 = configuration,  001 = BRAM contents",
        "     [22] top/bottom   [21:17] row   [16:7] column   [6:0] minor",
    ], 18, C_GRF)
    far.to_edge(DOWN, buff=0.3).set_x(-1.0)
    sc.play(FadeIn(far), run_time=0.8)
    sc.wait(2.0)


def s5_crc(sc):
    sc.heading("The CRC that gates everything",
               "CRC-32C, and it covers the register address as well as the data")

    w = code_block([
        "for every WRITE data word (except writes to CRC itself):",
        "",
        "        37 bits  =  { register[4:0] , data[31:0] }",
        "",
        "        fed LSB first into CRC-32C, reflected 0x82F63B78, init 0",
    ], 22, INK)
    w[2].set_color(C_BIT)
    w.shift(UP * 1.3)
    sc.play(LaggedStart(*[FadeIn(l) for l in w], lag_ratio=0.25), run_time=1.8)

    bar = fieldbar([("register[4:0]", 5, C_GRF), ("data[31:0]", 32, C_BIT)],
                   total_w=9.0, h=0.7, size=17)
    bar.next_to(w, DOWN, buff=0.7)
    sc.play(Create(bar[0]), FadeIn(bar[1]), FadeIn(bar[2]), run_time=1.0)

    why = code_block([
        "including the register address means a corrupted HEADER is caught too,",
        "not just corrupted data - this is how prjxray documents the 7-series CRC.",
        "",
        "Any FDRI data word clears CRC_OK, so a CRC check MUST follow the frames",
        "before START is accepted. A half-written design can never run.",
    ], 19, INK)
    why[0].set_color(DIM); why[1].set_color(DIM)
    why[3].set_color(C_ERR); why[4].set_color(C_ERR)
    why.next_to(bar, DOWN, buff=0.7).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in why], lag_ratio=0.2), run_time=1.8)
    sc.wait(2.2)


def s6_startup(sc):
    sc.heading("Startup", "UG470's sequence: the fabric is held inert until a good load "
                          "has been proven, then released one stage at a time")

    boxes = VGroup(
        chip("JPROGRAM", C_ERR, 2.3, 0.8, 19),
        chip("load\n(chain or frames)", C_BIT, 2.8, 0.9, 17),
        chip("JSTART\n+ 12 TCK in RTI", C_GRF, 2.8, 0.9, 17),
    ).arrange(RIGHT, buff=0.9).shift(UP * 2.0)
    for i, b in enumerate(boxes):
        sc.play(FadeIn(b), run_time=0.4)
        if i < 2:
            sc.play(GrowArrow(arrow(b.get_right(), boxes[i + 1].get_left(), DIM, 0.05)),
                    run_time=0.25)

    sig = ["GSR", "GTS", "GWE", "DONE"]
    init = ["1", "1", "0", "0"]
    rows = VGroup()
    for n, v in zip(sig, init):
        lab = mono(n, 22, INK)
        val = mono(v, 22, C_ERR)
        rows.add(VGroup(lab, val).arrange(RIGHT, buff=0.6))
    rows.arrange(DOWN, aligned_edge=LEFT, buff=0.42)
    rows.next_to(boxes, DOWN, buff=0.9).set_x(-4.0)
    meaning = VGroup(
        Text("every flip-flop forced to its INIT value", font_size=17, color=DIM),
        Text("every pad output forced to 0", font_size=17, color=DIM),
        Text("no flip-flop may change at all", font_size=17, color=DIM),
        Text("LD3 - the design is live", font_size=17, color=DIM),
    )
    for m, r in zip(meaning, rows):
        m.next_to(r, RIGHT, buff=0.7).set_y(r.get_y())
    sc.play(FadeIn(rows), FadeIn(meaning), run_time=0.9)
    at = mono("after power-up or JPROGRAM", 18, C_ERR)
    at.next_to(rows, UP, buff=0.35).align_to(rows, LEFT)
    sc.play(FadeIn(at), run_time=0.5)
    sc.wait(1.2)

    phase = mono("phase 0", 20, C_GRF).to_edge(RIGHT, buff=1.3).set_y(rows.get_y())
    sc.play(FadeIn(phase), run_time=0.4)
    steps = [(0, "0", "GSR released - flip-flops may move"),
             (1, "0", "GTS released - pads drive the LEDs"),
             (2, "1", "GWE asserted - flip-flops may change"),
             (3, "1", "DONE - LD3 lights")]
    for idx, newv, note in steps:
        nt = Text(note, font_size=19, color=C_BIT)
        nt.to_edge(DOWN, buff=0.5)
        sc.play(rows[idx][1].animate.become(
                    mono(newv, 22, C_RTL).move_to(rows[idx][1])),
                phase.animate.become(mono(f"phase {idx + 1}", 20, C_GRF)
                                     .move_to(phase)),
                FadeIn(nt), run_time=0.6)
        sc.wait(0.55)
        sc.play(FadeOut(nt), run_time=0.2)

    gate = Text("JSTART does nothing unless COMMITTED or the frame path accepted START. "
                "Without a proven load, DONE never rises.",
                font_size=20, color=C_ERR)
    gate.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.35)
    sc.play(FadeIn(gate), run_time=0.9)
    sc.wait(2.2)


def s7_guards(sc):
    sc.heading("The refusals", "each of these is a rule in RTL, a scenario in tb_frames, "
                               "and a mutant in make mutate")

    rows = [
        ("frames while GWE = 1", "WR_ERROR - unless a partial reconfiguration freeze is acknowledged"),
        ("IDCODE never matched", "ID_ERROR - FDRI is refused outright"),
        ("wrong CRC", "CRC_ERROR - the parser parks and only JPROGRAM revives it"),
        ("START without CRC_OK", "refused - CRC_OK must come AFTER the last FDRI word"),
        ("chain with wrong length", "LEN_ERR - COMMITTED stays 0, JSTART refuses"),
        ("FDRO without RCFG", "WR_ERROR, zeros"),
        ("AGHIGH without IDCODE", "freeze not entered"),
        ("LFRM without a CRC", "WR_ERROR and the fabric STAYS frozen"),
    ]
    g = VGroup()
    for a, b in rows:
        g.add(VGroup(mono(a, 19, C_ERR), Text(b, font_size=17, color=DIM))
              .arrange(RIGHT, buff=0.4, aligned_edge=DOWN))
    for r in g:
        r[1].align_to(g[0][1], LEFT).shift(RIGHT * 4.2)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.26)
    if g.width > 13.0:
        g.scale_to_fit_width(13.0)
    g.next_to(sc.mobjects[1], DOWN, buff=0.6).set_x(0)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.2) for r in g], lag_ratio=0.15),
            run_time=2.6)

    m = Text("Two of these survived their first mutation run and had to earn new scenarios: "
             "'LFRM ignores the CRC' and 'FDRO without RCFG'.",
             font_size=19, color=C_BIT)
    m.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.35)
    sc.play(FadeIn(m), run_time=0.9)
    sc.wait(2.4)


def s8_files(sc):
    sc.files_used(
        inputs=[("hw/src/core/cfg_store.v", "the memory + both write paths"),
                ("hw/src/core/cfg_frames.v", "the UG470 packet parser"),
                ("hw/src/core/cfg_ctrl.v", "chain CRC, length, startup FSM")],
        generated=[("hw/src/generated/bob_params.vh", "NFRAMES, FAR_TABLE"),
                   ("tools/bob/device.json", "frames, columns, chain order"),
                   ("tools/bob/packets.py", "stream builder + a bit-level model")],
        verified=[("hw/tb/tb_frames.v", "71 checks, expectations from the model"),
                  ("sim/mutate_frames.sh", "29 mutants, all killed"),
                  ("tests/test_hwtest_fake.py", "a stand-in board, good and broken")])


EP03 = [s1_memory, s2_layout, s3_chain, s4_frames, s5_crc, s6_startup, s7_guards, s8_files]


class Ep03Config(BobScene):
    def construct(self):
        self.titlecard("EPISODE 3", "The configuration engine",
                       "one memory, two write paths, and the startup that gates them")
        for i, part in enumerate(EP03):
            part(self)
            if i < len(EP03) - 1:
                clear_all(self)


class E03S1Memory(BobScene):
    def construct(self): s1_memory(self)


class E03S2Layout(BobScene):
    def construct(self): s2_layout(self)


class E03S3Chain(BobScene):
    def construct(self): s3_chain(self)


class E03S4Frames(BobScene):
    def construct(self): s4_frames(self)


class E03S5Crc(BobScene):
    def construct(self): s5_crc(self)


class E03S6Startup(BobScene):
    def construct(self): s6_startup(self)


class E03S7Guards(BobScene):
    def construct(self): s7_guards(self)


class E03S8Files(BobScene):
    def construct(self): s8_files(self)

In [ ]:
%%manim -qm Ep04IO
# ===========================================================================
#  EPISODE 4: I/O, pads and the clock
#  I/O, pads and the clock
#
#  render one section instead, by putting its class on the magic line above:
#      E04S1Iotile
#      E04S2Padmap
#      E04S3Userclock
#      E04S4Gap
#      E04S5Sync
#      E04S6Files
# ===========================================================================

# =============================================================================
#  EPISODE 4 - I/O, pads and the user clock
# =============================================================================

def s1_iotile(sc):
    sc.heading("An I/O block is smaller than you expect",
               "no io_tile.v at all - one VPR pad plus two boundary cells, wired in bob_fpga.v")

    world = chip("the world\n(a package pin)", C_GRF, 2.6, 1.2, 19)
    world.move_to(np.array([-5.0, 0.6, 0]))
    sc.play(FadeIn(world), run_time=0.5)

    inc = chip("input cell\ncell 44 + k", C_PY, 2.3, 0.9, 17).move_to(np.array([-1.7, 1.6, 0]))
    outc = chip("output cell\ncell k", C_PY, 2.3, 0.9, 17).move_to(np.array([-1.7, -0.5, 0]))
    sc.play(FadeIn(inc), FadeIn(outc), run_time=0.6)

    opin = chip("OPIN\ninpad", C_RTL, 1.9, 0.9, 17).move_to(np.array([1.7, 1.6, 0]))
    ipin = chip("IPIN mux\noutpad", C_RTL, 1.9, 0.9, 17).move_to(np.array([1.7, -0.5, 0]))
    fab = chip("the fabric", C_RTL, 2.2, 2.4, 20).move_to(np.array([4.7, 0.55, 0]))
    sc.play(FadeIn(opin), FadeIn(ipin), FadeIn(fab), run_time=0.6)

    sc.play(GrowArrow(arrow(world.get_right(), inc.get_left(), DIM, 0.08)),
            GrowArrow(arrow(inc.get_right(), opin.get_left(), DIM, 0.08)),
            GrowArrow(arrow(opin.get_right(), fab.get_left() + UP * 0.6, DIM, 0.08)),
            run_time=0.7)
    sc.play(GrowArrow(arrow(fab.get_left() + DOWN * 0.6, ipin.get_right(), DIM, 0.08)),
            GrowArrow(arrow(ipin.get_left(), outc.get_right(), DIM, 0.08)),
            GrowArrow(arrow(outc.get_left(), world.get_right() + DOWN * 0.35, DIM, 0.08)),
            run_time=0.7)

    gts = mono("GTS", 20, C_ERR).move_to(np.array([-1.7, -1.8, 0]))
    sc.play(FadeIn(gts), GrowArrow(arrow(gts.get_top(), outc.get_bottom(), C_ERR, 0.1)),
            run_time=0.6)

    notes = code_block([
        "the outpad IPIN is an ordinary routing mux - it costs configuration bits",
        "the inpad OPIN is a source - it costs none",
        "GTS forces every pad output to 0 during startup",
        "",
        "there is no true tri-state: the board's pins have fixed directions,",
        "so bob drives 0 instead of releasing. An honest simplification.",
    ], 19, INK)
    notes[2].set_color(C_ERR)
    notes[4].set_color(DIM); notes[5].set_color(DIM)
    notes.next_to(VGroup(world, fab), DOWN, buff=0.9).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in notes], lag_ratio=0.18), run_time=1.8)
    sc.wait(2.2)


def s2_padmap(sc):
    sc.heading("44 pads, 9 of them wired to something you can see",
               "tools/bob/device.py picks the numbers; bob_top.v is the only file that knows about a board")

    ring = VGroup()
    pos = {}
    idx = 0
    for x in range(14):
        for y in range(12):
            if (x in (0, 13)) and (y in (0, 11)):
                continue
            if not (x in (0, 13) or y in (0, 11)):
                continue
            s = Square(0.3, color=C_GRF, stroke_width=1.5).set_fill(C_GRF, 0.14)
            s.move_to(np.array([-3.0 + x * 0.42, -2.2 + y * 0.42, 0]))
            pos[idx] = s
            ring.add(s)
            idx += 1
    ring.shift(LEFT * 1.4)
    sc.play(LaggedStart(*[FadeIn(s, scale=0.6) for s in ring], lag_ratio=0.02),
            run_time=1.6)

    board = [("SW0", 12, C_RTL), ("SW1", 14, C_RTL), ("BTN0", 16, C_RTL),
             ("BTN1", 18, C_RTL), ("BTN2", 20, C_RTL), ("BTN3", 22, C_RTL),
             ("LD0", 13, C_BIT), ("LD1", 15, C_BIT), ("LD2", 17, C_BIT)]
    tbl = VGroup()
    for n, p, c in board:
        tbl.add(VGroup(mono(f"pad {p:2d}", 18, DIM), mono(n, 18, c))
                .arrange(RIGHT, buff=0.4))
    tbl.arrange(DOWN, aligned_edge=LEFT, buff=0.2).to_edge(RIGHT, buff=1.4)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.2) for r in tbl], lag_ratio=0.12),
            run_time=1.4)
    for n, p, c in board:
        if p in pos:
            sc.play(pos[p].animate.set_stroke(c).set_fill(c, 0.7), run_time=0.12)

    rest = Text("every other pad's input is 0 and its output goes nowhere - "
                "they exist, and boundary scan is the only way to reach them",
                font_size=19, color=DIM)
    rest.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.35)
    sc.play(FadeIn(rest), run_time=0.8)
    sc.wait(2.0)


def s3_userclock(sc):
    sc.heading("The user clock is not a clock",
               "hw/src/core/clock_ctrl.v - UG949 prefers a clock ENABLE to a logic-generated clock")

    sys = chip("sysclk\n125 MHz, H16, on a BUFG", C_GRF, 4.0, 1.1, 19)
    sys.move_to(np.array([-4.0, 1.7, 0]))
    ff = chip("every fabric flip-flop,\nBRAM and DSP register", C_RTL, 4.2, 1.1, 19)
    ff.move_to(np.array([2.4, 1.7, 0]))
    sc.play(FadeIn(sys), FadeIn(ff),
            GrowArrow(arrow(sys.get_right(), ff.get_left(), DIM, 0.1)), run_time=0.8)
    gce = mono("gce", 26, C_BIT).next_to(ff, DOWN, buff=0.7)
    sc.play(FadeIn(gce), GrowArrow(arrow(gce.get_top(), ff.get_bottom(), C_BIT, 0.1)),
            run_time=0.6)
    gl = Text("the enable. One high cycle = one user clock tick.",
              font_size=19, color=C_BIT).next_to(gce, RIGHT, buff=0.5)
    sc.play(FadeIn(gl), run_time=0.6)

    modes = VGroup(
        VGroup(mono("clk_mode = 0", 22, C_PY),
               Text("JTAG-stepped: one gce per TCK rising edge while USER1 ce is set,",
                    font_size=17, color=DIM),
               Text("plus one per step / INTEST autostep pulse. This is how every",
                    font_size=17, color=DIM),
               Text("cycle-exact test drives the fabric one clock at a time.",
                    font_size=17, color=DIM)).arrange(DOWN, aligned_edge=LEFT, buff=0.12),
        VGroup(mono("clk_mode = 1", 22, C_VPR),
               Text("free-running: one gce every 2**(clk_div + 8) sysclk cycles.",
                    font_size=17, color=DIM),
               Text("clk_div = 15 gives 14.9 Hz - slow enough to watch an LED blink.",
                    font_size=17, color=DIM)).arrange(DOWN, aligned_edge=LEFT, buff=0.12),
    ).arrange(DOWN, aligned_edge=LEFT, buff=0.5)
    modes.next_to(gce, DOWN, buff=0.8).set_x(-0.4)
    sc.play(LaggedStart(*[FadeIn(m, shift=RIGHT * 0.2) for m in modes], lag_ratio=0.3),
            run_time=2.0)

    ctrl = Text("Both live in the 8-bit ctrl tile - the very first bits of the chain, "
                "FAR column 0. ./bob build --clock run --div 15 sets them.",
                font_size=19, color=C_BIT)
    ctrl.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.35)
    sc.play(FadeIn(ctrl), run_time=0.8)
    sc.wait(2.2)


def s4_gap(sc):
    sc.heading("The 256-cycle guarantee",
               "the single most important line of timing in the project, and it is enforced in RTL")

    bad = code_block([
        "M7's Vivado report:   WNS  -1102 ns      1858 failing endpoints",
        "and yet the board passed 26 / 26.",
    ], 22, C_ERR)
    bad.shift(UP * 2.0)
    sc.play(FadeIn(bad), run_time=0.9)
    sc.wait(0.8)

    why = code_block([
        "The analyser cannot know a configuration, so it walked a path no legal,",
        "loop-free bitstream ever uses: fabric register -> 1202 logic levels -> BRAM.",
        "It bounced between dsp0 and dsp1 thirty-four times.",
    ], 19, DIM)
    why.next_to(bad, DOWN, buff=0.5)
    sc.play(LaggedStart(*[FadeIn(l) for l in why], lag_ratio=0.2), run_time=1.5)
    sc.wait(0.8)

    fix = code_block([
        "clock_ctrl.v GUARANTEES gce pulses are >= 2**8 = 256 sysclk cycles apart,",
        "in BOTH modes - a request that arrives sooner waits (one is kept pending).",
        "",
        "So the XDC can say, truthfully:",
        "    set_multicycle_path -setup 256 -from [get_clocks sysclk] -to [get_clocks sysclk]",
        "",
        "256 cycles = 2048 ns > the 1110 ns path.  WNS at M13: +0.877 ns, 0 failing.",
    ], 19, INK)
    fix[0].set_color(C_BIT); fix[1].set_color(C_BIT)
    fix[4].set_color(C_RTL); fix[6].set_color(C_RTL)
    fix.next_to(why, DOWN, buff=0.6)
    sc.play(LaggedStart(*[FadeIn(l) for l in fix], lag_ratio=0.18), run_time=2.2)

    lesson = Text("A multicycle exception is a PROMISE. tb_clock_gap.v exists to prove "
                  "bob keeps it - 11 checks, including while frozen.",
                  font_size=19, color=C_BIT)
    lesson.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(lesson), run_time=0.9)
    sc.wait(2.2)


def s5_sync(sc):
    sc.heading("Two clocks, and only one safe way across",
               "TCK and sysclk are declared asynchronous - everything that crosses is synchronised")

    t = chip("TCK domain\nTAP, configuration,\nboundary, CFG_CTRL", C_GRF, 3.6, 1.8, 19)
    t.move_to(np.array([-4.0, 0.8, 0]))
    s = chip("sysclk domain\nthe fabric, BRAM,\nDSP, clock_ctrl", C_RTL, 3.6, 1.8, 19)
    s.move_to(np.array([4.0, 0.8, 0]))
    sc.play(FadeIn(t), FadeIn(s), run_time=0.7)

    ff1 = Square(0.55, color=C_BIT, stroke_width=2.5).set_fill(C_BIT, 0.18)
    ff2 = ff1.copy()
    pair = VGroup(ff1, ff2).arrange(RIGHT, buff=0.35).move_to(np.array([0, 0.8, 0]))
    pl = mono("ASYNC_REG\ntwo-flop synchroniser", 16, C_BIT).next_to(pair, UP, buff=0.3)
    sc.play(FadeIn(pair), FadeIn(pl),
            GrowArrow(arrow(t.get_right(), pair.get_left(), DIM, 0.12)),
            GrowArrow(arrow(pair.get_right(), s.get_left(), DIM, 0.12)), run_time=0.9)

    crossers = code_block([
        "what crosses:   TCK itself (as data),  USER1 ce / step / cin,",
        "                GSR, GWE, the quasi-static clock configuration,",
        "                and the partial-reconfiguration freeze and its acknowledgement",
    ], 19, INK)
    crossers.next_to(pair, DOWN, buff=1.0).set_x(0)
    sc.play(FadeIn(crossers), run_time=0.9)

    xdc = code_block([
        "set_clock_groups -asynchronous -group [get_clocks tck] -group [get_clocks sysclk]",
    ], 18, C_PY)
    xdc.next_to(crossers, DOWN, buff=0.5)
    sc.play(FadeIn(xdc), run_time=0.6)

    rule = Text("And the rule that makes it sound: configuration only changes while GWE = 0. "
                "The fabric never samples bits while they move.",
                font_size=19, color=C_BIT)
    rule.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.35)
    sc.play(FadeIn(rule), run_time=0.9)
    sc.wait(2.2)


def s6_files(sc):
    sc.files_used(
        inputs=[("hw/src/core/clock_ctrl.v", "gce, both modes, the gap guard"),
                ("hw/src/fabric/bob_fpga.v", "the boundary ring and GTS"),
                ("hw/src/top/bob_top.v", "the only file that knows a board exists"),
                ("hw/constr/pynq_z2.xdc", "pins and the multicycle promises")],
        generated=[("hw/src/generated/bob_params.vh", "BOB_PAD_*, NPAD, GCE_MIN_GAP_SHIFT"),
                   ("tools/bob/device.json", "pads, board_inputs, board_outputs")],
        verified=[("hw/tb/tb_clock_gap.v", "11 checks on gce spacing and freeze"),
                  ("tests/test_layout.py", "the XDC is plain XDC, no Tcl"),
                  ("tests/test_reports.py", "WNS >= 0, 0 failing endpoints, from M13")])


EP04 = [s1_iotile, s2_padmap, s3_userclock, s4_gap, s5_sync, s6_files]


class Ep04IO(BobScene):
    def construct(self):
        self.titlecard("EPISODE 4", "I/O and the clock",
                       "pads, the boundary, and the enable that pretends to be a clock")
        for i, part in enumerate(EP04):
            part(self)
            if i < len(EP04) - 1:
                clear_all(self)


class E04S1Iotile(BobScene):
    def construct(self): s1_iotile(self)


class E04S2Padmap(BobScene):
    def construct(self): s2_padmap(self)


class E04S3Userclock(BobScene):
    def construct(self): s3_userclock(self)


class E04S4Gap(BobScene):
    def construct(self): s4_gap(self)


class E04S5Sync(BobScene):
    def construct(self): s5_sync(self)


class E04S6Files(BobScene):
    def construct(self): s6_files(self)

In [ ]:
%%manim -qm Ep05BRAM
# ===========================================================================
#  EPISODE 5: BRAM
#  BRAM
#
#  render one section instead, by putting its class on the magic line above:
#      E05S1What
#      E05S2Cfg
#      E05S3Modes
#      E05S4Contents
#      E05S5Files
# ===========================================================================

# =============================================================================
#  EPISODE 5 - BRAM: a UG473 RAMB18E1 subset, and why contents are separate
# =============================================================================

def s1_what(sc):
    sc.heading("bob's BRAM is a real RAMB18",
               "hw/src/tiles/bram_core.v is written to Vivado's inference template on purpose")

    mem = Rectangle(width=3.4, height=2.6, color=C_VPR, stroke_width=3)
    mem.set_fill(C_VPR, opacity=0.12)
    t = Text("1024 x 18\ntrue dual port", font_size=24, color=INK, line_spacing=0.8)
    core = VGroup(mem, t.move_to(mem)).move_to(np.array([0, 0.9, 0]))
    sc.play(FadeIn(core), run_time=0.7)

    for side, x, col in (("port A", -3.6, C_RTL), ("port B", 3.6, C_GRF)):
        pins = code_block(["addr[9:0]", "di[17:0]", "we", "en", "rst", "regce",
                           "", "do[17:0]"], 17, col)
        pins.move_to(np.array([x, 0.9, 0]))
        sc.play(FadeIn(pins), run_time=0.6)
        sc.play(GrowArrow(arrow(pins.get_right() if x < 0 else pins.get_left(),
                                mem.get_left() if x < 0 else mem.get_right(),
                                DIM, 0.25)), run_time=0.3)

    facts = code_block([
        "one block  ->  one RAMB18E1 on the host FPGA, not a pile of LUTs",
        "synchronous read (UG473): the address is registered, data comes next cycle",
        "bob has two of them, in grid column x = 3, each 5 rows tall",
    ], 19, INK)
    facts.next_to(core, DOWN, buff=1.2).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in facts], lag_ratio=0.2), run_time=1.6)

    why = Text("A guest FPGA whose memories melt into the host's LUTs would not fit. "
               "Inference is the whole reason this block is written the way it is.",
               font_size=19, color=C_BIT)
    why.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.35)
    sc.play(FadeIn(why), run_time=0.9)
    sc.wait(2.0)


def s2_cfg(sc):
    sc.heading("Eight configuration bits", "that is the entire BRAM tile - the data is somewhere else")

    fields = [("wmode_a", 2, C_RTL), ("wmode_b", 2, C_GRF), ("reg_a", 1, C_BIT),
              ("reg_b", 1, C_BIT), ("jtag_a", 1, C_PY), ("jtag_b", 1, C_PY)]
    bar = fieldbar(fields, total_w=8.0, h=0.8, size=17)
    bar.shift(UP * 1.5)
    sc.play(Create(bar[0]), FadeIn(bar[1]), FadeIn(bar[2]), run_time=1.2)

    rows = [
        ("wmode_a / wmode_b", "0 WRITE_FIRST, 1 READ_FIRST, 2 NO_CHANGE  (UG473)"),
        ("reg_a / reg_b", "DOA_REG / DOB_REG - one more pipeline stage on the output"),
        ("jtag_a / jtag_b", "let USER4 drive that port's pins, so write modes can be"),
        ("", "stepped one clock at a time from the host"),
    ]
    g = VGroup()
    for a, b in rows:
        g.add(VGroup(mono(a, 19, C_BIT), Text(b, font_size=17, color=DIM))
              .arrange(RIGHT, buff=0.4, aligned_edge=DOWN))
    for r in g:
        r[1].align_to(g[0][1], LEFT).shift(RIGHT * 3.6)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.24)
    if g.width > 12.8:
        g.scale_to_fit_width(12.8)
    g.next_to(bar, DOWN, buff=0.9).set_x(0)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.2) for r in g], lag_ratio=0.18),
            run_time=1.8)

    trim = code_block([
        "trimmed from a real RAMB18E1, and we say so:",
        "   the 9 / 4 / 1-bit width modes   (bit-level addressing stops inference)",
        "   per-byte write enables",
        "   separate RSTRAM / RSTREG pins",
    ], 18, DIM)
    trim[0].set_color(C_ERR)
    trim.to_edge(DOWN, buff=0.35).set_x(0)
    sc.play(FadeIn(trim), run_time=0.9)
    sc.wait(2.2)


def s3_modes(sc):
    sc.heading("Three write modes out of one template",
               "only READ_FIRST is inferable - the other two are rebuilt around it")

    base = chip("bram_core.v\nVivado's READ_FIRST template", C_VPR, 5.0, 1.2, 20)
    base.shift(UP * 1.8)
    sc.play(FadeIn(base), run_time=0.6)

    outs = VGroup(
        chip("READ_FIRST\nthe old word appears", C_RTL, 3.4, 1.1, 18),
        chip("WRITE_FIRST\nthe new word appears", C_BIT, 3.4, 1.1, 18),
        chip("NO_CHANGE\nthe output holds", C_GRF, 3.4, 1.1, 18),
    ).arrange(RIGHT, buff=0.5).next_to(base, DOWN, buff=1.1)
    for o in outs:
        sc.play(GrowArrow(arrow(base.get_bottom(), o.get_top(), DIM, 0.12)),
                FadeIn(o), run_time=0.45)

    how = code_block([
        "WRITE_FIRST and NO_CHANGE are built from READ_FIRST with a register and a",
        "mux on the output path - which is exactly why the write mode can be a",
        "CONFIGURATION BIT rather than a synthesis-time parameter.",
        "",
        "A real FPGA picks the mode when you instantiate the primitive.",
        "bob picks it when you load a bitstream.",
    ], 19, INK)
    how[4].set_color(DIM)
    how[5].set_color(C_BIT)
    how.next_to(outs, DOWN, buff=0.8).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in how], lag_ratio=0.18), run_time=2.0)

    proof = Text("tb_bram.v sweeps all 36 mode x register x port combinations "
                 "against model.py: 5292 checks.", font_size=19, color=C_BIT)
    proof.scale_to_fit_width(12.8).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(proof), run_time=0.8)
    sc.wait(2.2)


def s4_contents(sc):
    sc.heading("Contents are not configuration",
               "UG470 keeps BRAM data in its own block type, and so does bob - with two ways in")

    mem = chip("the 1024 words", C_VPR, 3.2, 1.0, 21).shift(UP * 1.9)
    sc.play(FadeIn(mem), run_time=0.5)

    a = chip("USER4\nSELECT, LOAD_PTR,\nWRITE, READ", C_PY, 3.4, 1.4, 18)
    b = chip("FAR block type 001\nframes, inside the\nsame CRC stream", C_BIT, 3.8, 1.4, 18)
    a.move_to(np.array([-3.4, -0.1, 0]))
    b.move_to(np.array([3.4, -0.1, 0]))
    sc.play(GrowArrow(arrow(a.get_top(), mem.get_bottom(), DIM, 0.12)), FadeIn(a),
            run_time=0.6)
    sc.play(GrowArrow(arrow(b.get_top(), mem.get_bottom(), DIM, 0.12)), FadeIn(b),
            run_time=0.6)
    al = mono("M5, still used by --mode chain", 15, DIM).next_to(a, DOWN, buff=0.18)
    bl = mono("M15, the default", 15, DIM).next_to(b, DOWN, buff=0.18)
    sc.play(FadeIn(al), FadeIn(bl), run_time=0.4)

    frame = code_block([
        "block type 001:  column = BRAM index,  frame n = row x 128 + minor",
        "frame n holds addresses 4n .. 4n+3,  word = { 14'b0, data[17:0] }",
        "256 frames per BRAM, and FAR auto-increments into the next one",
    ], 18, C_BIT)
    frame.next_to(VGroup(a, b), DOWN, buff=0.8).set_x(0)
    sc.play(FadeIn(frame), run_time=0.9)

    rule = code_block([
        "both paths write only while GWE = 0 - you cannot rewrite a running memory",
        "JPROGRAM does NOT clear contents: they survive a reprogram",
    ], 19, C_ERR)
    rule.next_to(frame, DOWN, buff=0.55).set_x(0)
    sc.play(FadeIn(rule), run_time=0.8)
    sc.wait(1.2)

    bug = Text("M11's bug, found on the board: bob load wrote only up to the last non-zero "
               "word, so zero words kept the PREVIOUS design's values. "
               "Now all 1024 words of every used BRAM are always written.",
               font_size=18, color=C_ERR)
    bug.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(bug), run_time=0.9)
    sc.wait(2.4)


def s5_files(sc):
    sc.files_used(
        inputs=[("hw/src/tiles/bram_core.v", "the inferable 1024x18 TDP memory"),
                ("hw/src/tiles/bram_block.v", "fabric pins or the USER4 drive word"),
                ("hw/src/tiles/bram_jtag.v", "USER4 commands + the M15 frame sequencer")],
        generated=[("tools/bob/device.json", "the 8 fields, USER4 command codes"),
                   ("tools/bob/model.py", "class Bram - the reference behaviour")],
        verified=[("hw/tb/tb_bram.v", "5292 checks, all 36 combinations"),
                  ("sim/mutate_fabric.sh", "bram-no-write-first, bram-en-ignored, ..."),
                  ("docs/hwtest/results.log", "ram-readback on the real board")])


EP05 = [s1_what, s2_cfg, s3_modes, s4_contents, s5_files]


class Ep05BRAM(BobScene):
    def construct(self):
        self.titlecard("EPISODE 5", "BRAM",
                       "a UG473 subset, and why contents are not configuration")
        for i, part in enumerate(EP05):
            part(self)
            if i < len(EP05) - 1:
                clear_all(self)


class E05S1What(BobScene):
    def construct(self): s1_what(self)


class E05S2Cfg(BobScene):
    def construct(self): s2_cfg(self)


class E05S3Modes(BobScene):
    def construct(self): s3_modes(self)


class E05S4Contents(BobScene):
    def construct(self): s4_contents(self)


class E05S5Files(BobScene):
    def construct(self): s5_files(self)

In [ ]:
%%manim -qm Ep06DSP
# ===========================================================================
#  EPISODE 6: DSP
#  DSP
#
#  render one section instead, by putting its class on the magic line above:
#      E06S1Datapath
#      E06S2Cfg
#      E06S3Opmodes
#      E06S4Jtag
#      E06S5Files
# ===========================================================================

# =============================================================================
#  EPISODE 6 - DSP: a trimmed DSP48E1, and the cascade
# =============================================================================

def s1_datapath(sc):
    sc.heading("The datapath", "hw/src/tiles/dsp_core.v, following AMD UG479's DSP48E1")

    a = chip("A  25", C_RTL, 1.6, 0.7, 19).move_to(np.array([-5.4, 1.9, 0]))
    d = chip("D  25", C_RTL, 1.6, 0.7, 19).move_to(np.array([-5.4, 0.9, 0]))
    b = chip("B  18", C_GRF, 1.6, 0.7, 19).move_to(np.array([-5.4, -0.3, 0]))
    c = chip("C  48", C_BIT, 1.6, 0.7, 19).move_to(np.array([-5.4, -1.5, 0]))
    sc.play(LaggedStart(FadeIn(a), FadeIn(d), FadeIn(b), FadeIn(c), lag_ratio=0.15),
            run_time=0.9)

    pre = Circle(radius=0.42, color=C_RTL, stroke_width=3).set_fill(C_RTL, 0.12)
    pret = Text("+/-", font_size=22, color=INK).move_to(pre)
    pre = VGroup(pre, pret).move_to(np.array([-3.0, 1.4, 0]))
    sc.play(FadeIn(pre),
            GrowArrow(arrow(a.get_right(), pre.get_left(), DIM, 0.12)),
            GrowArrow(arrow(d.get_right(), pre.get_left(), DIM, 0.12)), run_time=0.7)
    prel = mono("pre-adder  D +/- A", 16, C_RTL).next_to(pre, UP, buff=0.25)
    sc.play(FadeIn(prel), run_time=0.4)

    mul = chip("x\n25 x 18\nsigned", C_VPR, 1.7, 1.6, 19).move_to(np.array([-0.6, 0.6, 0]))
    sc.play(FadeIn(mul),
            GrowArrow(arrow(pre.get_right(), mul.get_left() + UP * 0.35, DIM, 0.12)),
            GrowArrow(arrow(b.get_right(), mul.get_left() + DOWN * 0.35, DIM, 0.12)),
            run_time=0.8)
    m = mono("M", 18, C_VPR).next_to(mul, RIGHT, buff=0.25)
    sc.play(FadeIn(m), run_time=0.3)

    alu = chip("+", C_BIT, 1.3, 1.6, 26).move_to(np.array([2.4, 0.2, 0]))
    sc.play(FadeIn(alu),
            GrowArrow(arrow(m.get_right(), alu.get_left() + UP * 0.35, DIM, 0.12)),
            GrowArrow(arrow(c.get_right(), alu.get_left() + DOWN * 0.45, DIM, 0.2)),
            run_time=0.8)
    p = chip("P  48", C_BIT, 1.6, 0.8, 20).move_to(np.array([5.0, 0.2, 0]))
    sc.play(FadeIn(p), GrowArrow(arrow(alu.get_right(), p.get_left(), DIM, 0.12)),
            run_time=0.5)

    fb = VMobject(color=DIM, stroke_width=2)
    fb.set_points_as_corners([p.get_bottom(), np.array([5.0, -1.9, 0]),
                              np.array([2.4, -1.9, 0]), alu.get_bottom()])
    fbl = mono("P feeds back  (accumulate)", 15, DIM).next_to(fb, DOWN, buff=0.05).set_x(3.6)
    sc.play(Create(fb), FadeIn(fbl), run_time=0.7)

    trim = Text("trimmed: dynamic OPMODE / INMODE / ALUMODE, ALU functions other than add, "
                "the 30-bit A port, pattern detect, CARRYIN",
                font_size=18, color=DIM)
    trim.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(trim), run_time=0.8)
    sc.wait(2.0)


def s2_cfg(sc):
    sc.heading("Sixteen configuration bits per slice",
               "every pipeline register is a bit - that is what makes a DSP configurable")

    fields = [("opmode", 2, C_VPR), ("use_d", 1, C_RTL), ("d_sub", 1, C_RTL),
              ("areg", 1, C_BIT), ("breg", 1, C_BIT), ("creg", 1, C_BIT),
              ("dreg", 1, C_BIT), ("mreg", 1, C_BIT), ("preg", 1, C_BIT),
              ("jtag_a", 1, C_PY), ("jtag_b", 1, C_PY), ("jtag_c", 1, C_PY),
              ("jtag_d", 1, C_PY), ("jtag_ctrl", 1, C_PY), ("rsv", 1, DIM)]
    bar = fieldbar(fields, total_w=11.5, h=0.75, size=12)
    bar.shift(UP * 1.5)
    sc.play(Create(bar[0]), FadeIn(bar[1]), FadeIn(bar[2]), run_time=1.3)

    g = VGroup(
        VGroup(mono("opmode", 19, C_VPR),
               Text("which of the four static opmodes this slice computes", font_size=17, color=DIM))
        .arrange(RIGHT, buff=0.4, aligned_edge=DOWN),
        VGroup(mono("use_d / d_sub", 19, C_RTL),
               Text("bring the pre-adder in, and whether it subtracts", font_size=17, color=DIM))
        .arrange(RIGHT, buff=0.4, aligned_edge=DOWN),
        VGroup(mono("areg..preg", 19, C_BIT),
               Text("six independent pipeline registers - latency is configurable", font_size=17, color=DIM))
        .arrange(RIGHT, buff=0.4, aligned_edge=DOWN),
        VGroup(mono("jtag_*", 19, C_PY),
               Text("drive that bus from the JTAG drive word instead of the fabric", font_size=17, color=DIM))
        .arrange(RIGHT, buff=0.4, aligned_edge=DOWN),
    )
    for r in g:
        r[1].align_to(g[0][1], LEFT).shift(RIGHT * 2.6)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.3)
    if g.width > 12.8:
        g.scale_to_fit_width(12.8)
    g.next_to(bar, DOWN, buff=0.9).set_x(0)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.2) for r in g], lag_ratio=0.2),
            run_time=1.9)

    ce = Text("CE and RST are grouped four ways (C shares P's) - a real DSP48E1 has more, "
              "and we list the difference rather than pretend.",
              font_size=18, color=DIM)
    ce.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(ce), run_time=0.8)
    sc.wait(2.2)


def s3_opmodes(sc):
    sc.heading("Four opmodes and one cascade",
               "static, because a dynamic OPMODE pin would cost fabric routing on every slice")

    modes = VGroup(
        VGroup(mono("0", 22, C_VPR), mono("P = M", 24, INK),
               Text("a plain multiply", font_size=17, color=DIM)),
        VGroup(mono("1", 22, C_VPR), mono("P = M + C", 24, INK),
               Text("multiply-add", font_size=17, color=DIM)),
        VGroup(mono("2", 22, C_VPR), mono("P = P + M", 24, INK),
               Text("accumulate - P fed back from its own register", font_size=17, color=DIM)),
        VGroup(mono("3", 22, C_VPR), mono("P = (PCIN >>> 17) + M", 24, INK),
               Text("the cascade: a wider multiply across two slices", font_size=17, color=DIM)),
    )
    for m in modes:
        m.arrange(RIGHT, buff=0.5, aligned_edge=DOWN)
    for m in modes:
        m[2].align_to(modes[3][2], LEFT).shift(RIGHT * 5.4)
    modes.arrange(DOWN, aligned_edge=LEFT, buff=0.38)
    if modes.width > 13.0:
        modes.scale_to_fit_width(13.0)
    modes.shift(UP * 1.3)
    sc.play(LaggedStart(*[FadeIn(m, shift=RIGHT * 0.2) for m in modes], lag_ratio=0.25),
            run_time=2.0)

    s0 = chip("dsp0", C_VPR, 2.2, 0.9, 21)
    s1 = chip("dsp1", C_VPR, 2.2, 0.9, 21)
    pair = VGroup(s0, s1).arrange(UP, buff=1.0).next_to(modes, DOWN, buff=0.9).set_x(-3.0)
    sc.play(FadeIn(pair), run_time=0.6)
    casc = arrow(s0.get_top(), s1.get_bottom(), C_BIT, 0.08)
    cl = mono("PCOUT -> PCIN", 18, C_BIT).next_to(casc, RIGHT, buff=0.3)
    sc.play(GrowArrow(casc), FadeIn(cl), run_time=0.6)

    note = code_block([
        "in the routing graph the cascade is a DIRECT, like the carry chain:",
        "48 wires, no multiplexers, no configuration bits.",
        "",
        "In VPR's architecture it is declared as a <direct> so the packer keeps",
        "a two-slice multiply together.",
    ], 18, INK)
    note[0].set_color(C_GRF); note[1].set_color(C_GRF)
    note.next_to(pair, RIGHT, buff=1.2).set_y(pair.get_y())
    if note.width > 7.0:
        note.scale_to_fit_width(7.0)
    sc.play(FadeIn(note), run_time=0.9)
    sc.wait(2.2)


def s4_jtag(sc):
    sc.heading("The private DSP instruction",
               "all four AMD USER codes were taken, so M6 took a private one: 101000")

    dr = fieldbar([("drive word: A, B, C, D and 8 controls, per slice", 248, C_PY),
                   ("P0", 48, C_BIT), ("P1", 48, C_BIT)], total_w=11.0, h=0.8, size=15)
    dr.shift(UP * 1.4)
    sc.play(Create(dr[0]), FadeIn(dr[1]), FadeIn(dr[2]), run_time=1.1)

    why = code_block([
        "Why drive a DSP from JTAG at all?",
        "",
        "Because a fabric-fed DSP can only be tested through routing, and at M6",
        "there was no routing yet. The drive word lets the host put any operand",
        "on any bus and read both P values back - so tb_dsp.v and the board test",
        "check the SAME vectors against model.py, 1344 of them.",
        "",
        "It stayed after M7 because it is still the fastest way to prove a slice",
        "works without building a design around it.",
    ], 19, INK)
    why[0].set_color(C_BIT)
    why.next_to(dr, DOWN, buff=0.8).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in why], lag_ratio=0.15), run_time=2.4)
    sc.wait(2.0)


def s5_files(sc):
    sc.files_used(
        inputs=[("hw/src/tiles/dsp_core.v", "the arithmetic, registers and opmodes"),
                ("hw/src/tiles/dsp_block.v", "per-bus source: fabric or drive word"),
                ("hw/src/tiles/dsp_jtag.v", "the 248-bit drive register")],
        generated=[("tools/bob/device.json", "16 fields, opmodes, cascade"),
                   ("tools/bob/model.py", "class Dsp + the cascade helpers")],
        verified=[("hw/tb/tb_dsp.v", "1344 checks: every opmode x pre-adder, 2 slices"),
                  ("sim/mutate_fabric.sh", "dsp-no-d-sub, dsp-shift-16, dsp-no-cascade"),
                  ("examples/fir.v", "a 2-tap FIR on both slices, live on the board")])


EP06 = [s1_datapath, s2_cfg, s3_opmodes, s4_jtag, s5_files]


class Ep06DSP(BobScene):
    def construct(self):
        self.titlecard("EPISODE 6", "DSP",
                       "a trimmed DSP48E1, and the cascade that costs no bits")
        for i, part in enumerate(EP06):
            part(self)
            if i < len(EP06) - 1:
                clear_all(self)


class E06S1Datapath(BobScene):
    def construct(self): s1_datapath(self)


class E06S2Cfg(BobScene):
    def construct(self): s2_cfg(self)


class E06S3Opmodes(BobScene):
    def construct(self): s3_opmodes(self)


class E06S4Jtag(BobScene):
    def construct(self): s4_jtag(self)


class E06S5Files(BobScene):
    def construct(self): s5_files(self)

In [ ]:
%%manim -qm Ep07Routing
# ===========================================================================
#  EPISODE 7: Routing
#  Routing
#
#  render one section instead, by putting its class on the magic line above:
#      E07S1Boxes
#      E07S2Arch
#      E07S3Mux
#      E07S4Directs
#      E07S5Loops
#      E07S6Files
# ===========================================================================

# =============================================================================
#  EPISODE 7 - Routing: boxes, muxes, directs, and loops by construction
# =============================================================================

def s1_boxes(sc):
    sc.heading("Two classic structures, one implementation",
               "bob never writes a 'connection box' or a 'switch box' - it writes muxes, one per graph node")

    cb = chip("connection box", C_GRF, 3.6, 1.0, 21).move_to(np.array([-3.6, 1.9, 0]))
    sb = chip("switch box", C_VPR, 3.6, 1.0, 21).move_to(np.array([3.6, 1.9, 0]))
    sc.play(FadeIn(cb), FadeIn(sb), run_time=0.6)
    cbt = Text("picks which track drives a block input pin", font_size=18, color=DIM)
    sbt = Text("picks which track drives another track", font_size=18, color=DIM)
    cbt.next_to(cb, DOWN, buff=0.25)
    sbt.next_to(sb, DOWN, buff=0.25)
    sc.play(FadeIn(cbt), FadeIn(sbt), run_time=0.5)

    down = VGroup(
        chip("an IPIN node with fan-in", C_GRF, 4.6, 0.85, 19),
        chip("a CHANX / CHANY node with fan-in", C_VPR, 5.4, 0.85, 19),
    ).arrange(RIGHT, buff=0.7).next_to(VGroup(cbt, sbt), DOWN, buff=0.9)
    sc.play(GrowArrow(arrow(cbt.get_bottom(), down[0].get_top(), DIM, 0.1)),
            GrowArrow(arrow(sbt.get_bottom(), down[1].get_top(), DIM, 0.1)),
            FadeIn(down), run_time=0.8)

    one = chip("bob_mux", C_RTL, 3.2, 0.9, 24, "BOLD")
    one.next_to(down, DOWN, buff=0.9).set_x(0)
    sc.play(GrowArrow(arrow(down[0].get_bottom(), one.get_top(), DIM, 0.1)),
            GrowArrow(arrow(down[1].get_bottom(), one.get_top(), DIM, 0.1)),
            FadeIn(one), run_time=0.8)

    pt = Text("The distinction is architectural vocabulary. In the RTL there is only "
              "one module, instantiated 3391 times.", font_size=20, color=C_BIT)
    pt.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.4)
    sc.play(FadeIn(pt), run_time=0.9)
    sc.wait(2.0)


def s2_arch(sc):
    sc.heading("The routing architecture",
               "every number here is OpenFPGA's k6_frac_N10 tileable reference, kept on purpose")

    rows = [
        ("W = 24", "tracks per channel, 12 each way", C_VPR),
        ("L4", "every wire spans 4 tiles, unidirectional", C_VPR),
        ("Wilton, Fs = 3", "each incoming track can turn 3 ways at a switch point", C_GRF),
        ("fc_in = 0.15", "a block input taps 15% of the channel -> fan-in 4", C_GRF),
        ("fc_out = 0.10", "a block output reaches 10% of the channel", C_GRF),
        ("fc = 0 on carry and clock pins", "they are directs and globals, not routed", C_BIT),
    ]
    g = VGroup()
    for a, b, col in rows:
        g.add(VGroup(mono(a, 21, col), Text(b, font_size=18, color=DIM))
              .arrange(RIGHT, buff=0.5, aligned_edge=DOWN))
    for r in g:
        r[1].align_to(g[0][1], LEFT).shift(RIGHT * 4.4)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.3)
    if g.width > 13.0:
        g.scale_to_fit_width(13.0)
    g.shift(UP * 1.2)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.2) for r in g], lag_ratio=0.18),
            run_time=2.2)

    sweep = code_block([
        "W was chosen by sweeping VPR, and the cost is configuration bits:",
        "",
        "    W = 16  ->  4.2 k routing bits",
        "    W = 24  ->  6.0 k             <- chosen: IPIN fan-in 4 (0.15 x 24)",
        "    W = 32  ->  7.8 k",
    ], 19, INK)
    sweep[3].set_color(C_BIT)
    sweep.next_to(g, DOWN, buff=0.8).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in sweep], lag_ratio=0.18), run_time=1.6)

    div = Text("Where bob diverges from the reference, it says so: 1 BLE per CLB "
               "(N10 is too many config flops for an XC7Z020), carry south-to-north, "
               "bob's own BRAM and DSP blocks.", font_size=18, color=DIM)
    div.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(div), run_time=0.9)
    sc.wait(2.0)


def s3_mux(sc):
    sc.heading("The encoding, and the property it buys",
               "hw/src/fabric/bob_mux.v - four lines of Verilog with a deliberate choice in them")

    tbl = code_block([
        "sel = 0                  const 0",
        "sel = 1   (IPIN only)    const 1",
        "sel = base + i           input i          base = 1, or 2 on an IPIN",
        "sel  anything larger     const 0",
    ], 23, INK)
    tbl[0].set_color(C_BIT)
    tbl[3].set_color(DIM)
    tbl.shift(UP * 1.7)
    sc.play(LaggedStart(*[FadeIn(l) for l in tbl], lag_ratio=0.2), run_time=1.6)

    box = SurroundingRectangle(tbl[0], color=C_BIT, buff=0.1)
    sc.play(Create(box), run_time=0.5)
    big = Text("an all-zero configuration is a dark, loop-free fabric",
               font_size=30, color=C_BIT, weight="BOLD")
    big.scale_to_fit_width(11.5).next_to(tbl, DOWN, buff=0.9)
    sc.play(FadeIn(big), run_time=0.8)
    sc.wait(1.0)

    why = code_block([
        "That is not decoration. It means:",
        "   power-up and JPROGRAM leave a fabric that cannot oscillate",
        "   a half-written or refused load cannot build a ring oscillator",
        "   simulation of an unconfigured device terminates",
        "",
        "The out-of-range case matters too: a random or corrupted value can only",
        "ever select const 0, never a wire. Random chains in simulation rely on it.",
    ], 19, INK)
    why[0].set_color(DIM)
    why[5].set_color(DIM); why[6].set_color(DIM)
    why.next_to(big, DOWN, buff=0.7).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in why], lag_ratio=0.15), run_time=2.2)
    sc.wait(2.0)


def s4_directs(sc):
    sc.heading("138 connections that cost nothing",
               "an IPIN driven by exactly one OPIN has no choice to make")

    a = Dot(np.array([-3.4, 1.4, 0]), radius=0.14, color=C_RTL)
    al = mono("OPIN  clb_x12y2.cout", 17, C_RTL).next_to(a, UP, buff=0.2)
    b = Dot(np.array([0.6, 1.4, 0]), radius=0.14, color=C_RTL)
    bl = mono("IPIN  clb_x12y3.cin", 17, C_RTL).next_to(b, UP, buff=0.2)
    sc.play(FadeIn(a), FadeIn(al), FadeIn(b), FadeIn(bl), run_time=0.6)
    sc.play(GrowArrow(arrow(a.get_center(), b.get_center(), DIM, 0.2)), run_time=0.5)

    v = mono("assign r1583 = r1308;   // clb_x12y3.cin[0] <- clb_x12y2.cout[0]", 19, C_RTL)
    vp = panel(v, C_RTL)
    vp.next_to(VGroup(a, b), DOWN, buff=0.9).set_x(0)
    sc.play(FadeIn(vp), run_time=0.8)
    note = Text("a wire. No mux, no configuration bits, no delay through a select.",
                font_size=20, color=C_BIT).next_to(vp, DOWN, buff=0.35)
    sc.play(FadeIn(note), run_time=0.6)

    which = code_block([
        "what is a direct in bob:",
        "   the carry chains, up every CLB column",
        "   the DSP PCOUT -> PCIN cascade, 48 bits",
        "",
        "and one node type that is neither: an undriven node reads 0,",
        "except the bottom row's cin, which is USER1's cin from JTAG.",
    ], 19, INK)
    which[0].set_color(DIM)
    which[4].set_color(DIM); which[5].set_color(DIM)
    which.next_to(note, DOWN, buff=0.7).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in which], lag_ratio=0.18), run_time=1.8)

    trap = Text("The mutation test that cuts one carry direct is pinned to the LAST CLB column - "
                "when the grid grew from 8 to 12 columns it silently stopped matching, twice.",
                font_size=18, color=C_ERR)
    trap.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(trap), run_time=0.9)
    sc.wait(2.2)


def s5_loops(sc):
    sc.heading("The fabric has combinational loops, by construction",
               "and that is a fact about mesh routing, not a bug in bob")

    ta = chip("tile A", C_RTL, 2.2, 0.9, 21).move_to(np.array([-2.2, 1.8, 0]))
    tb = chip("tile B", C_RTL, 2.2, 0.9, 21).move_to(np.array([2.2, 1.8, 0]))
    sc.play(FadeIn(ta), FadeIn(tb), run_time=0.5)
    e1 = CurvedArrow(ta.get_right(), tb.get_left(), angle=-0.7, color=C_ERR, stroke_width=3)
    e2 = CurvedArrow(tb.get_left(), ta.get_right(), angle=-0.7, color=C_ERR, stroke_width=3)
    sc.play(Create(e1), run_time=0.4)
    sc.play(Create(e2), run_time=0.4)
    lab = Text("A's east output can feed B, whose west output can feed A",
               font_size=19, color=DIM).next_to(VGroup(ta, tb), DOWN, buff=0.6)
    sc.play(FadeIn(lab), run_time=0.5)

    tools = code_block([
        "Verilator says:   UNOPTFLAT",
        "Vivado says:      [DRC LUTLP-1] Combinatorial Loop Alert",
        "yosys says:       5553 'logic loop' warnings",
    ], 20, C_ERR)
    tools.next_to(lab, DOWN, buff=0.6).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in tools], lag_ratio=0.2), run_time=1.3)

    ans = code_block([
        "Whether a loop EXISTS depends on the bitstream - and none that bob's",
        "tools emit contains one, because the router builds a tree from each",
        "source to its sinks, and an all-zero mux selects const 0.",
        "",
        "Naming a net per loop with ALLOW_COMBINATORIAL_LOOPS is not workable:",
        "there are thousands, and the names are synthesis output that changes",
        "every build. So the DRC is downgraded, scoped to the fabric top only.",
    ], 19, INK)
    ans[0].set_color(C_BIT); ans[1].set_color(C_BIT); ans[2].set_color(C_BIT)
    ans[4].set_color(DIM); ans[5].set_color(DIM); ans[6].set_color(DIM)
    ans.next_to(tools, DOWN, buff=0.6).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in ans], lag_ratio=0.15), run_time=2.4)

    cost = Text("The price: Vivado breaks the loops arbitrarily for timing analysis, "
                "so some paths are not analysed - which is exactly why the 256-cycle "
                "gce guarantee had to be made real in RTL.",
                font_size=18, color=C_ERR)
    cost.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.28)
    sc.play(FadeIn(cost), run_time=0.9)
    sc.wait(2.2)


def s6_files(sc):
    sc.files_used(
        inputs=[("hw/src/fabric/bob_mux.v", "the one routing multiplexer"),
                ("tools/bob/vpr_arch.py", "the architecture VPR is given"),
                ("hw/scripts/drc_waiver.tcl", "the LUTLP-1 downgrade, fabric tops only")],
        generated=[("hw/src/generated/bob_fabric.v", "3391 muxes, 138 directs, 4416 lines"),
                   ("tools/bob/arch/bob_k6_rr.xml.gz", "VPR's graph, committed and stamped"),
                   ("tools/bob/device.json", "every mux: node, bits, inputs")],
        verified=[("hw/tb/tb_bob.v", "4 random routed netlists vs model.py every clock"),
                  ("tests/test_device.py", "every pip is one field value, every pin a node"),
                  ("sim/mutate_fabric.sh", "mux-no-const1, mux-inputs-shifted, carry-direct-cut")])


EP07 = [s1_boxes, s2_arch, s3_mux, s4_directs, s5_loops, s6_files]


class Ep07Routing(BobScene):
    def construct(self):
        self.titlecard("EPISODE 7", "Routing",
                       "boxes, muxes, directs, and loops by construction")
        for i, part in enumerate(EP07):
            part(self)
            if i < len(EP07) - 1:
                clear_all(self)


class E07S1Boxes(BobScene):
    def construct(self): s1_boxes(self)


class E07S2Arch(BobScene):
    def construct(self): s2_arch(self)


class E07S3Mux(BobScene):
    def construct(self): s3_mux(self)


class E07S4Directs(BobScene):
    def construct(self): s4_directs(self)


class E07S5Loops(BobScene):
    def construct(self): s5_loops(self)


class E07S6Files(BobScene):
    def construct(self): s6_files(self)

In [ ]:
%%manim -qm Ep08Yosys
# ===========================================================================
#  EPISODE 8: Synthesis with yosys
#  Synthesis with yosys
#
#  render one section instead, by putting its class on the magic line above:
#      E08S1What
#      E08S2Passes
#      E08S3Proof
#      E08S4Stimulus
#      E08S5Files
# ===========================================================================

# =============================================================================
#  EPISODE 8 - Synthesis: turning Verilog into cells bob actually has
# =============================================================================

def s1_what(sc):
    sc.heading("Synthesis is translation, not magic",
               "tools/bob/synth.py drives yosys with bob's own cell library and maps")

    src = chip("counter.v\nordinary Verilog", INK, 3.0, 1.2, 20).move_to(np.array([-4.6, 1.5, 0]))
    ys = chip("yosys", C_VPR, 2.2, 1.0, 24, "BOLD").move_to(np.array([-0.6, 1.5, 0]))
    out = chip("only bob cells", C_RTL, 3.2, 1.2, 20).move_to(np.array([3.6, 1.5, 0]))
    sc.play(FadeIn(src), run_time=0.4)
    sc.play(GrowArrow(arrow(src.get_right(), ys.get_left(), DIM, 0.1)), FadeIn(ys),
            run_time=0.5)
    sc.play(GrowArrow(arrow(ys.get_right(), out.get_left(), DIM, 0.1)), FadeIn(out),
            run_time=0.5)

    cells = code_block([
        "$lut         a K-input LUT             -> one CLB",
        "BOB_ADD      one carry bit             -> one CLB in carry mode",
        "BOB_FDRE     flip-flop, sync reset     -> the CLB's flop, ff_rstval = 0",
        "BOB_FDSE     flip-flop, sync set       -> ff_rstval = 1",
        "BOB_BRAM18   1024 x 18 true dual port  -> a BRAM block",
        "BOB_DSP      25 x 18 signed            -> a DSP slice",
    ], 19, C_RTL)
    cells.next_to(out, DOWN, buff=1.0).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l, shift=RIGHT * 0.15) for l in cells], lag_ratio=0.15),
            run_time=1.9)

    strict = Text("synth.py FAILS if anything else survives, or if the design has more "
                  "than one clock. There is no 'mostly mapped'.",
                  font_size=20, color=C_BIT)
    strict.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.35)
    sc.play(FadeIn(strict), run_time=0.9)
    sc.wait(2.0)


def s2_passes(sc):
    sc.heading("The pass order", "patterned on yosys's own synth_xilinx, with bob's maps swapped in")

    steps = [
        ("read + hierarchy", "elaborate, pick the top", DIM),
        ("proc, flatten, opt", "ordinary front-end work", DIM),
        ("mul2dsp 25x18", "big multipliers become BOB_DSP", C_BIT),
        ("memory_libmap", "memories become BOB_BRAM18, rules in bob_brams.txt", C_BIT),
        ("techmap _80_bob_alu", "$alu becomes BOB_ADD carry chains", C_RTL),
        ("dfflegalize", "every flop becomes BOB_FDRE / BOB_FDSE", C_RTL),
        ("abc -lut K", "whatever is left becomes $lut", C_VPR),
        ("write json / blif / sim", "for the placer, for VPR, for simulation", C_PY),
    ]
    g = VGroup()
    for a, b, col in steps:
        g.add(VGroup(mono(a, 20, col), Text(b, font_size=17, color=DIM))
              .arrange(RIGHT, buff=0.5, aligned_edge=DOWN))
    for r in g:
        r[1].align_to(g[0][1], LEFT).shift(RIGHT * 4.6)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.26)
    if g.width > 13.0:
        g.scale_to_fit_width(13.0)
    g.next_to(sc.mobjects[1], DOWN, buff=0.6).set_x(0)
    for r in g:
        sc.play(FadeIn(r, shift=RIGHT * 0.2), run_time=0.35)

    trap = code_block([
        "A trap worth remembering: the rule was first called _90_bob_alu, and yosys",
        "picked the generic _90_alu instead - rules are tried in NAME order.",
        "Renaming it _80_ fixed it. Nothing errored; the carry chains just vanished.",
    ], 18, C_ERR)
    trap.to_edge(DOWN, buff=0.35).set_x(0)
    sc.play(FadeIn(trap), run_time=0.9)
    sc.wait(2.2)


def s3_proof(sc):
    sc.heading("Three copies of the same circuit, simulated together",
               "tools/bob/equiv.py - this is what stops a wrong map from ever reaching the board")

    three = VGroup(
        chip("the source\ncounter.v", INK, 3.0, 1.2, 19),
        chip("the yosys netlist\ncounter_syn.v", C_VPR, 3.4, 1.2, 19),
        chip("the golden netlist\ngolden.py, one wire per bit", C_PY, 4.2, 1.2, 17),
    ).arrange(RIGHT, buff=0.6).shift(UP * 1.8)
    sc.play(LaggedStart(*[FadeIn(t) for t in three], lag_ratio=0.2), run_time=1.0)

    sim = chip("iverilog: all three, same stimulus, 300 cycles", C_BIT, 8.6, 0.9, 20)
    sim.next_to(three, DOWN, buff=0.8)
    for t in three:
        sc.play(GrowArrow(arrow(t.get_bottom(), sim.get_top(), DIM, 0.1)), run_time=0.2)
    sc.play(FadeIn(sim), run_time=0.5)

    chk = code_block([
        "compared before AND after every clock edge",
        "the source trace is saved   -> later checked against model.py and the board",
        "every golden net is saved   -> later checked against CAPTURE on the board",
    ], 19, INK)
    chk.next_to(sim, DOWN, buff=0.7).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in chk], lag_ratio=0.2), run_time=1.5)

    why = Text("Why a third, 'golden' netlist? Because yosys's own output renames every "
               "cell and net, so nothing in it maps back to a fabric location. "
               "golden.py writes the same logic with one named wire per bit.",
               font_size=18, color=DIM)
    why.scale_to_fit_width(13.0).next_to(chk, DOWN, buff=0.5)
    sc.play(FadeIn(why), run_time=0.9)
    sc.wait(2.2)


def s4_stimulus(sc):
    sc.heading("The bug that made this episode necessary",
               "a test that passes for the wrong reason is worse than no test")

    story = code_block([
        "M8: the counter example passed. Model, RTL and board all agreed.",
        "",
        "M9: someone looked at the saved trace. It was all zeros.",
    ], 22, INK)
    story[2].set_color(C_ERR)
    story.shift(UP * 1.9)
    sc.play(FadeIn(story[0]), run_time=0.7)
    sc.wait(0.6)
    sc.play(FadeIn(story[2]), run_time=0.7)
    sc.wait(1.0)

    why = code_block([
        "The stimulus was uniform random over all inputs.",
        "The counter's synchronous reset was one of those inputs.",
        "So it was asserted about half the time - and the counter never counted.",
        "",
        "Every check compared zero against zero and passed.",
    ], 20, INK)
    why[4].set_color(C_ERR)
    why.next_to(story, DOWN, buff=0.7).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in why], lag_ratio=0.2), run_time=1.9)

    fix = code_block([
        "The fix: biased vectors in 50-cycle segments, so control inputs hold still",
        "long enough for state to build up. The counter's trace now reaches LED 0-6.",
        "",
        "The rule: check that your stimulus actually exercises what it claims.",
    ], 20, C_BIT)
    fix[3].set_color(C_RTL)
    fix.next_to(why, DOWN, buff=0.7).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in fix], lag_ratio=0.2), run_time=1.7)
    sc.wait(2.2)


def s5_files(sc):
    sc.files_used(
        inputs=[("tools/bob/synth/bob_cells_sim.v", "the cell library, simulatable"),
                ("tools/bob/synth/bob_map.v", "techmap rules, incl. _80_bob_alu"),
                ("tools/bob/synth/bob_brams.txt", "memory_libmap rules"),
                ("examples/*.v", "the designs themselves")],
        generated=[("build/synth/<top>/<top>.json", "for the placer and VPR"),
                   ("build/synth/<top>/<top>_syn.v", "the netlist, for simulation"),
                   ("tools/bob/golden.py output", "one named wire per bit"),
                   ("trace.json", "the source trace + every golden net per clock")],
        verified=[("tools/bob/equiv.py", "source == netlist == golden, 300 cycles"),
                  ("hw/tb/tb_synth.v", "972 checks: the designs on the real fabric"),
                  ("tests/test_synth.py", "non-bob cells rejected, chains split by column")])


EP08 = [s1_what, s2_passes, s3_proof, s4_stimulus, s5_files]


class Ep08Yosys(BobScene):
    def construct(self):
        self.titlecard("EPISODE 8", "Synthesis",
                       "Verilog into cells bob actually has")
        for i, part in enumerate(EP08):
            part(self)
            if i < len(EP08) - 1:
                clear_all(self)


class E08S1What(BobScene):
    def construct(self): s1_what(self)


class E08S2Passes(BobScene):
    def construct(self): s2_passes(self)


class E08S3Proof(BobScene):
    def construct(self): s3_proof(self)


class E08S4Stimulus(BobScene):
    def construct(self): s4_stimulus(self)


class E08S5Files(BobScene):
    def construct(self): s5_files(self)

In [ ]:
%%manim -qm Ep09VPR
# ===========================================================================
#  EPISODE 9: VPR and place-and-route
#  VPR and place-and-route
#
#  render one section instead, by putting its class on the magic line above:
#      E09S1Title
#      E09S2Fabric
#      E09S3WhatIsVPR
#      E09S4RRGraph
#      E09S5BigIdea
#      E09S6Role1
#      E09S7Role2
#      E09S8RouteToBits
#      E09S9Python
#      E09S10PythonPnR
#      E09S11Recap
# ===========================================================================

# =============================================================================
#  EPISODE 9 - VPR: the tool that designs the routing, then routes on it
#  (the standalone film docs/manim/vpr_explained.py is this same content)
# =============================================================================

# ============================================================ P01  TITLE ======
def s1_title(sc):
    title = Text("VPR", font_size=96, color=C_VPR, weight="BOLD")
    sub = Text("and how bob uses Python with it", font_size=34, color=INK)
    sub.next_to(title, DOWN, buff=0.4)
    line = Line(LEFT * 3.2, RIGHT * 3.2, color=DIM, stroke_width=2)
    line.next_to(sub, DOWN, buff=0.45)
    tag = Text("an FPGA built inside an FPGA", font_size=24, color=DIM)
    tag.next_to(line, DOWN, buff=0.35)

    sc.play(Write(title), run_time=1.2)
    sc.play(FadeIn(sub, shift=UP * 0.2), run_time=0.8)
    sc.play(Create(line), FadeIn(tag), run_time=0.8)
    sc.wait(1.4)

    q = Text("Two questions:", font_size=30, color=INK)
    q1 = Text("1.  What does VPR actually do?", font_size=28, color=C_VPR)
    q2 = Text("2.  Why do we run it twice, for two different jobs?",
              font_size=28, color=C_PY)
    qs = VGroup(q, q1, q2).arrange(DOWN, aligned_edge=LEFT, buff=0.35)

    sc.play(FadeOut(VGroup(title, sub, line, tag)), run_time=0.6)
    sc.play(FadeIn(q), run_time=0.5)
    sc.play(FadeIn(q1, shift=RIGHT * 0.3), run_time=0.7)
    sc.play(FadeIn(q2, shift=RIGHT * 0.3), run_time=0.7)
    sc.wait(2.0)


# ================================================== P02  WHAT IS A FABRIC =====
def s2_fabric(sc):
    sc.heading("First: what is an FPGA fabric?",
               "logic blocks, wires between them, and a switch at every junction")

    # --- the grid ------------------------------------------------------------
    blocks, wires = VGroup(), VGroup()
    for i in range(4):
        for j in range(3):
            b = Square(side_length=0.62, color=C_RTL, stroke_width=2.5)
            b.set_fill(C_RTL, opacity=0.14)
            b.move_to(np.array([-3.6 + i * 1.5, -1.9 + j * 1.5, 0]))
            blocks.add(b)
    for i in range(4):
        for j in range(3):
            c = blocks[i * 3 + j].get_center()
            wires.add(Line(c + RIGHT * 0.36, c + RIGHT * 1.14,
                           color=DIM, stroke_width=2))
            wires.add(Line(c + UP * 0.36, c + UP * 1.14,
                           color=DIM, stroke_width=2))

    lab = Text("logic blocks (LUT + flip-flop)", font_size=22, color=C_RTL)
    lab.next_to(blocks, DOWN, buff=0.45)
    sc.play(LaggedStart(*[FadeIn(b, scale=0.7) for b in blocks], lag_ratio=0.05),
            run_time=1.4)
    sc.play(FadeIn(lab), run_time=0.5)
    sc.play(LaggedStart(*[Create(w) for w in wires], lag_ratio=0.02), run_time=1.4)

    lab2 = Text("routing wires", font_size=22, color=DIM)
    lab2.next_to(lab, DOWN, buff=0.2)
    sc.play(FadeIn(lab2), run_time=0.5)
    sc.wait(0.8)

    # --- zoom out to one junction -------------------------------------------
    fabric = VGroup(blocks, wires, lab, lab2)
    sc.play(fabric.animate.scale(0.55).to_edge(LEFT, buff=0.5), run_time=1.0)

    q = Text("But what is AT a junction?", font_size=28, color=INK)
    q.move_to(np.array([2.6, 2.6, 0]))
    sc.play(FadeIn(q), run_time=0.6)

    mux = mux_symbol().move_to(np.array([2.6, 0.2, 0]))
    ins = VGroup()
    names = ["wire A", "wire B", "wire C", "wire D"]
    for k in range(4):
        y = 1.05 - k * 0.5
        ln = Line(np.array([0.9, y, 0]), np.array([2.2, y, 0]),
                  color=DIM, stroke_width=2.5)
        nm = Text(names[k], font_size=17, color=DIM)
        nm.next_to(ln, LEFT, buff=0.12)
        ins.add(VGroup(ln, nm))
    out = Arrow(np.array([3.0, 0.2, 0]), np.array([4.4, 0.2, 0]),
                buff=0, color=C_RTL, stroke_width=3)
    outl = Text("one output", font_size=18, color=C_RTL)
    outl.next_to(out, RIGHT, buff=0.12)

    sc.play(Create(mux), run_time=0.6)
    sc.play(LaggedStart(*[Create(i) for i in ins], lag_ratio=0.15), run_time=1.0)
    sc.play(GrowArrow(out), FadeIn(outl), run_time=0.6)

    sel = code_block(["sel = 2"], 24, C_BIT)
    sel.next_to(mux, DOWN, buff=0.6)
    selarrow = arrow(sel.get_top(), mux.get_bottom(), C_BIT, buff=0.1)
    sc.play(FadeIn(sel), GrowArrow(selarrow), run_time=0.7)

    # light up input C
    sc.play(ins[2].animate.set_color(C_BIT), out.animate.set_color(C_BIT),
            outl.animate.set_color(C_BIT), run_time=0.8)
    sc.wait(0.6)
    sc.play(sel[0].animate.become(mono("sel = 0", 24, C_BIT).move_to(sel[0])),
            ins[2].animate.set_color(DIM), run_time=0.5)
    sc.play(ins[0].animate.set_color(C_BIT), run_time=0.6)
    sc.wait(0.8)

    sc.play(FadeOut(VGroup(q, fabric)), run_time=0.6)
    detail = VGroup(mux, ins, out, outl, sel, selarrow)
    sc.play(detail.animate.scale(0.85).move_to(np.array([-2.6, 0.2, 0])),
            run_time=0.9)

    punch = VGroup(
        Text("A multiplexer.", font_size=32, color=INK, weight="BOLD"),
        Text("Its select bits come from", font_size=26, color=DIM),
        Text("configuration memory.", font_size=26, color=C_BIT),
        Text(" ", font_size=10),
        Text("So a bitstream is nothing", font_size=28, color=INK),
        Text("but the setting of every", font_size=28, color=INK),
        Text("multiplexer in the chip.", font_size=28, color=C_BIT,
             weight="BOLD"),
    ).arrange(DOWN, aligned_edge=LEFT, buff=0.22)
    punch.move_to(np.array([3.2, 0.1, 0]))
    sc.play(LaggedStart(*[FadeIn(l, shift=RIGHT * 0.2) for l in punch],
                        lag_ratio=0.25), run_time=2.4)

    note = Text("bob today:  3391 multiplexers,  18 560 configuration bits",
                font_size=22, color=C_GRF)
    note.to_edge(DOWN, buff=0.45)
    sc.play(FadeIn(note), run_time=0.7)
    sc.wait(2.0)


# ==================================================== P03  WHAT IS VPR ========
def s3_whatisvpr(sc):
    sc.heading("VPR  =  Versatile Place and Route",
               "the back end of VTR (Verilog-to-Routing).  bob uses VPR 9.")

    arch = chip("architecture XML", C_PY, 3.3, 1.0, 22)
    net = chip("netlist  (.eblif)", C_PY, 3.3, 1.0, 22)
    ins = VGroup(arch, net).arrange(DOWN, buff=0.9).to_edge(LEFT, buff=0.7)
    ins.shift(DOWN * 0.3)

    archd = code_block([
        "a description of an",
        "IMAGINARY FPGA:",
        "  grid size, columns",
        "  block pins",
        "  wire lengths",
        "  switch pattern",
    ], 17, DIM)
    archd.next_to(arch, DOWN, buff=0.18).align_to(arch, LEFT)

    netd = code_block([
        "the user circuit as",
        "LUTs and flip-flops",
    ], 17, DIM)
    netd.next_to(net, DOWN, buff=0.18).align_to(net, LEFT)

    sc.play(FadeIn(arch, shift=RIGHT * 0.3), run_time=0.6)
    sc.play(FadeIn(archd), run_time=0.5)
    sc.play(FadeIn(net, shift=RIGHT * 0.3), run_time=0.6)
    sc.play(FadeIn(netd), run_time=0.5)

    vpr = RoundedRectangle(width=3.0, height=4.2, corner_radius=0.18,
                           color=C_VPR, stroke_width=3.5)
    vpr.set_fill(C_VPR, opacity=0.10).move_to(np.array([0.9, -0.3, 0]))
    vprt = Text("VPR", font_size=36, color=C_VPR, weight="BOLD")
    vprt.next_to(vpr.get_top(), DOWN, buff=0.25)
    sc.play(Create(vpr), Write(vprt), run_time=0.9)
    sc.play(GrowArrow(arrow(arch.get_right(), vpr.get_left() + UP * 1.0)),
            GrowArrow(arrow(net.get_right(), vpr.get_left() + DOWN * 1.0)),
            run_time=0.7)

    stages = VGroup(
        Text("pack", font_size=26, color=INK),
        Text("place", font_size=26, color=INK),
        Text("route", font_size=26, color=INK),
        Text("analysis", font_size=26, color=INK),
    ).arrange(DOWN, buff=0.42)
    stages.move_to(vpr.get_center() + DOWN * 0.25)
    expl = [
        "group primitives into blocks",
        "give every block a grid slot",
        "find a wire path per net",
        "report timing / wirelength",
    ]
    for s, e in zip(stages, expl):
        et = Text(e, font_size=18, color=DIM)
        et.next_to(vpr, RIGHT, buff=0.55).set_y(s.get_y())
        sc.play(FadeIn(s, shift=DOWN * 0.15), FadeIn(et, shift=RIGHT * 0.2),
                run_time=0.55)

    sc.wait(1.6)
    box = SurroundingRectangle(stages[2], color=C_BIT, buff=0.16)
    sc.play(Create(box), run_time=0.5)
    key = Text("routing is the part that decides every mux setting",
               font_size=24, color=C_BIT)
    key.to_edge(DOWN, buff=0.45)
    sc.play(FadeIn(key), run_time=0.7)
    sc.wait(2.0)


# ================================================= P04  THE RR GRAPH ==========
def s4_rrgraph(sc):
    sc.heading("Before VPR can route, it builds a GRAPH",
               "the routing-resource graph: every wire and pin of that imaginary chip")

    # left: a scrap of fabric
    b1 = Square(0.8, color=C_RTL, stroke_width=3).set_fill(C_RTL, opacity=0.14)
    b2 = b1.copy()
    b1.move_to(np.array([-5.0, 1.1, 0]))
    b2.move_to(np.array([-5.0, -1.6, 0]))
    ch = VGroup(*[Line(np.array([-4.2, y, 0]), np.array([-2.4, y, 0]),
                       color=DIM, stroke_width=2.5)
                  for y in (-0.5, -0.15, 0.2)])
    l1 = Text("block", font_size=18, color=C_RTL).next_to(b1, UP, buff=0.12)
    l2 = Text("channel wires", font_size=18, color=DIM).next_to(ch, DOWN, buff=0.2)
    left = VGroup(b1, b2, ch, l1, l2)
    sc.play(FadeIn(left), run_time=0.9)

    conv = Text("becomes", font_size=22, color=DIM)
    conv.move_to(np.array([-1.5, 0.0, 0]))
    ar = arrow(np.array([-2.2, -0.6, 0]), np.array([-0.7, -0.6, 0]), C_GRF, 0.05)
    sc.play(FadeIn(conv), GrowArrow(ar), run_time=0.6)

    # right: nodes + edges
    def node(pos, label, color):
        d = Dot(pos, radius=0.13, color=color)
        t = Text(label, font_size=17, color=color)
        t.next_to(d, UP, buff=0.12)
        return VGroup(d, t)

    n_op = node(np.array([0.4, 1.3, 0]), "OPIN", C_RTL)
    n_cx = node(np.array([2.3, 0.7, 0]), "CHANX", C_GRF)
    n_cy = node(np.array([2.3, -0.9, 0]), "CHANY", C_GRF)
    n_ip = node(np.array([4.4, -0.2, 0]), "IPIN", C_RTL)
    nodes = VGroup(n_op, n_cx, n_cy, n_ip)

    edges = VGroup(
        arrow(n_op[0].get_center(), n_cx[0].get_center(), DIM, 0.18),
        arrow(n_op[0].get_center(), n_cy[0].get_center(), DIM, 0.18),
        arrow(n_cx[0].get_center(), n_ip[0].get_center(), DIM, 0.18),
        arrow(n_cy[0].get_center(), n_ip[0].get_center(), DIM, 0.18),
        arrow(n_cx[0].get_center(), n_cy[0].get_center(), DIM, 0.18),
    )
    sc.play(LaggedStart(*[FadeIn(n, scale=0.6) for n in nodes], lag_ratio=0.2),
            run_time=1.2)
    sc.play(LaggedStart(*[GrowArrow(e) for e in edges], lag_ratio=0.15),
            run_time=1.2)

    legend = code_block([
        "node   =  one wire, or one pin of a block",
        "edge   =  'this one CAN drive that one'",
        "         ... i.e. a programmable switch is there",
    ], 21, INK)
    legend.to_edge(DOWN, buff=0.7)
    sc.play(FadeIn(legend[0]), run_time=0.5)
    sc.play(FadeIn(legend[1]), run_time=0.5)
    sc.play(FadeIn(legend[2]), run_time=0.6)
    sc.wait(1.2)

    concl = Text("Routing a net = finding a path in this graph.",
                 font_size=28, color=C_BIT, weight="BOLD")
    concl.to_edge(DOWN, buff=0.35)
    sc.play(FadeOut(legend), FadeIn(concl), run_time=0.8)

    path = VGroup(
        edges[1].copy().set_color(C_BIT).set_stroke(width=6),
        edges[3].copy().set_color(C_BIT).set_stroke(width=6),
    )
    sc.play(Create(path), run_time=1.0)
    sc.wait(1.8)


# ================================================== P05  THE BIG IDEA =========
def s5_bigidea(sc):
    sc.heading("The big idea", "bob does not describe its hardware to VPR - it BUILDS its hardware FROM VPR's graph")

    # a node with four drivers
    centre = np.array([-3.6, 0.3, 0])
    d = Dot(centre, radius=0.16, color=C_GRF)
    dl = Text("node 3", font_size=20, color=C_GRF).next_to(d, DOWN, buff=0.22)
    drivers, arrows_in = VGroup(), VGroup()
    ids = ["4258", "4259", "4270", "4271"]
    for k, nid in enumerate(ids):
        p = centre + np.array([-2.1, 1.35 - k * 0.9, 0])
        dd = Dot(p, radius=0.10, color=DIM)
        tt = Text(nid, font_size=17, color=DIM).next_to(dd, LEFT, buff=0.12)
        drivers.add(VGroup(dd, tt))
        arrows_in.add(arrow(p, centre, DIM, 0.2))

    sc.play(FadeIn(d), FadeIn(dl), run_time=0.5)
    sc.play(LaggedStart(*[FadeIn(x) for x in drivers], lag_ratio=0.15),
            run_time=0.9)
    sc.play(LaggedStart(*[GrowArrow(a) for a in arrows_in], lag_ratio=0.15),
            run_time=0.9)

    rule = Text("4 edges drive it  ->  it must be a multiplexer",
                font_size=24, color=C_BIT)
    rule.next_to(VGroup(drivers, d), DOWN, buff=0.8)
    sc.play(FadeIn(rule), run_time=0.8)
    sc.wait(1.0)

    # the generated Verilog
    v = code_block([
        "bob_mux #(.N(4), .W(3), .C1(1)) m3 (",
        "    .sel (cfg[384 +: 3]),",
        "    .in  ({r4271, r4270, r4259, r4258}),",
        "    .o   (r3)",
        ");",
    ], 21, C_RTL)
    v.move_to(np.array([2.9, 0.6, 0]))
    vp = panel(v, C_RTL)
    cap = Text("hw/src/generated/bob_fabric.v", font_size=18, color=DIM)
    cap.next_to(vp, UP, buff=0.2)

    sc.play(FadeIn(vp), FadeIn(cap), run_time=1.0)
    sc.wait(0.6)

    hl = [
        ("4 drivers", v[0], "N(4)"),
        ("their node ids", v[2], "in{...}"),
        ("where its bits live in the bitstream", v[1], "cfg[384 +: 3]"),
        ("the node itself", v[3], "r3"),
    ]
    for label, line, _ in hl:
        b = SurroundingRectangle(line, color=C_BIT, buff=0.08)
        t = Text(label, font_size=20, color=C_BIT)
        t.next_to(vp, DOWN, buff=0.35)
        sc.play(Create(b), FadeIn(t), run_time=0.55)
        sc.wait(0.75)
        sc.play(FadeOut(b), FadeOut(t), run_time=0.35)

    sc.play(FadeOut(VGroup(drivers, arrows_in, d, dl, rule)), run_time=0.6)
    sc.play(vp.animate.move_to(np.array([0, 1.3, 0])),
            cap.animate.next_to(vp, UP, buff=0.2).set_x(0), run_time=0.9)

    summary = code_block([
        "every graph node with fan-in    ->   one bob_mux in Verilog",
        "its incoming edges              ->   that mux's inputs",
        "an input driven by only ONE     ->   a plain wire, zero bits",
        "",
        "3391 nodes   ->   3391 muxes   ->   4416 lines of Verilog",
    ], 22, INK)
    summary[4].set_color(C_GRF)
    summary.next_to(vp, DOWN, buff=0.8)
    sc.play(LaggedStart(*[FadeIn(l, shift=UP * 0.15) for l in summary],
                        lag_ratio=0.3), run_time=2.2)
    sc.wait(2.2)


# ===================================================== P06  ROLE 1 ============
def s6_role1(sc):
    sc.heading("Role 1:  VPR designs the fabric",
               "run once, only when the architecture changes   ( make rrgraph )")

    steps = [
        ("device.py", C_PY, "the architecture,\nwritten once"),
        ("bob_k6.xml", C_PY, "VPR's\narchitecture file"),
        ("VPR", C_VPR, "builds and dumps\nthe rr graph"),
        ("bob_k6_rr.xml.gz", C_GRF, "committed,\nsha256-stamped"),
        ("bob_fabric.v", C_RTL, "the real Verilog\n4416 lines"),
    ]
    boxes = VGroup()
    for name, col, _ in steps:
        boxes.add(chip(name, col, 2.45, 0.85, 19))
    boxes.arrange(RIGHT, buff=0.62).scale(0.98).shift(UP * 0.7)

    subs = VGroup()
    for b, (_, col, sub) in zip(boxes, steps):
        t = Text(sub, font_size=15, color=DIM, line_spacing=0.7)
        t.next_to(b, DOWN, buff=0.22)
        subs.add(t)

    for i, b in enumerate(boxes):
        sc.play(FadeIn(b, scale=0.85), FadeIn(subs[i]), run_time=0.45)
        if i < len(boxes) - 1:
            sc.play(GrowArrow(arrow(b.get_right(),
                                    boxes[i + 1].get_left(), DIM, 0.05)),
                    run_time=0.25)

    cmd = code_block([
        "vpr bob_k6.xml and2.blif --device bob12x10 \\",
        "    --route_chan_width 24 --timing_analysis off \\",
        "    --write_rr_graph rr.xml",
    ], 19, C_VPR)
    cmdp = panel(cmd, C_VPR)
    cmdp.next_to(subs, DOWN, buff=0.75)
    sc.play(FadeIn(cmdp), run_time=0.8)
    sc.wait(1.0)

    notes = VGroup(
        Text("and2.blif is a trivial 2-input AND - nobody wants the result.",
             font_size=21, color=INK),
        Text("VPR only builds the graph when it is about to route something,",
             font_size=21, color=DIM),
        Text("so and2 exists purely to make VPR emit --write_rr_graph.",
             font_size=21, color=DIM),
    ).arrange(DOWN, aligned_edge=LEFT, buff=0.18)
    notes.to_edge(DOWN, buff=0.4)
    sc.play(FadeOut(cmdp), run_time=0.4)
    notes.next_to(subs, DOWN, buff=0.8)
    sc.play(LaggedStart(*[FadeIn(n) for n in notes], lag_ratio=0.3),
            run_time=1.6)
    sc.wait(1.4)

    d = Text("the graph is COMMITTED, so Docker is only needed to rebuild it",
             font_size=23, color=C_GRF)
    d.to_edge(DOWN, buff=0.4)
    sc.play(FadeOut(notes), FadeIn(d), run_time=0.8)
    sc.wait(1.8)


# ===================================================== P07  ROLE 2 ============
def s7_role2(sc):
    sc.heading("Role 2:  VPR routes a user design",
               "run for every design   ( ./bob build counter.v )")

    row1 = VGroup(
        chip("counter.v", INK, 2.1, 0.8, 19),
        chip("yosys", C_PY, 1.9, 0.8, 19),
        chip("vpr_run\nprepare()", C_PY, 2.3, 0.9, 17),
        chip(".eblif\n+ pins", C_PY, 1.9, 0.9, 17),
    ).arrange(RIGHT, buff=0.55).shift(UP * 1.7)

    row2 = VGroup(
        chip("VPR", C_VPR, 1.9, 0.85, 22, "BOLD"),
        chip(".net .place\n.route", C_VPR, 2.3, 0.9, 17),
        chip("fasm_from_vpr", C_PY, 2.7, 0.85, 17),
        chip("FASM -> bits", C_BIT, 2.4, 0.85, 18),
    ).arrange(RIGHT, buff=0.55).shift(DOWN * 0.3)

    for row in (row1, row2):
        for i, b in enumerate(row):
            sc.play(FadeIn(b, scale=0.85), run_time=0.35)
            if i < len(row) - 1:
                sc.play(GrowArrow(arrow(b.get_right(), row[i + 1].get_left(),
                                        DIM, 0.05)), run_time=0.2)
        if row is row1:
            sc.play(GrowArrow(arrow(row1[3].get_bottom(), row2[0].get_top(),
                                    DIM, 0.1)), run_time=0.3)

    tail = VGroup(chip(".bit", C_BIT, 1.5, 0.8, 20),
                  chip("the board", C_RTL, 2.2, 0.8, 20))
    tail.arrange(RIGHT, buff=0.7).next_to(row2, DOWN, buff=0.85).set_x(2.4)
    sc.play(GrowArrow(arrow(row2[3].get_bottom(), tail[0].get_top(), DIM, 0.1)),
            FadeIn(tail[0]), run_time=0.45)
    sc.play(GrowArrow(arrow(tail[0].get_right(), tail[1].get_left(), DIM, 0.05)),
            FadeIn(tail[1]), run_time=0.45)

    flag = code_block(["--read_rr_graph rr.xml"], 26, C_GRF)
    flagp = panel(flag, C_GRF)
    flagp.next_to(row2[0], LEFT, buff=0.5).shift(DOWN * 1.1).set_x(-3.4)
    sc.play(FadeIn(flagp), GrowArrow(arrow(flagp.get_right(),
                                           row2[0].get_left(), C_GRF, 0.1)),
            run_time=0.8)
    msg = Text("the SAME committed graph the Verilog was generated from",
               font_size=22, color=C_GRF)
    msg.to_edge(DOWN, buff=0.35)
    sc.play(FadeIn(msg), run_time=0.7)
    sc.wait(1.4)

    inv = Text("VPR routes on exactly the hardware that exists.",
               font_size=30, color=C_BIT, weight="BOLD")
    inv.to_edge(DOWN, buff=0.35)
    sc.play(FadeOut(msg), FadeIn(inv), run_time=0.8)
    sc.wait(2.2)


# ================================================ P08  ROUTE -> BITS ==========
def s8_routetobits(sc):
    sc.heading("How a route turns into bits",
               "this is the whole trick, and it is four lines of Python")

    ys = 1.55
    pts = [np.array([-4.6, ys, 0]), np.array([-1.7, ys, 0]),
           np.array([1.2, ys, 0]), np.array([4.1, ys, 0])]
    labels = ["r900\nOPIN", "r1204\nCHANX", "r1631\nCHANY", "r68\nIPIN"]
    cols = [C_RTL, C_GRF, C_GRF, C_RTL]
    nodes = VGroup()
    for p, l, c in zip(pts, labels, cols):
        dd = Dot(p, radius=0.15, color=c)
        tt = Text(l, font_size=18, color=c, line_spacing=0.7)
        tt.next_to(dd, UP, buff=0.18)
        nodes.add(VGroup(dd, tt))
    hops = VGroup(*[arrow(pts[i], pts[i + 1], DIM, 0.22) for i in range(3)])

    sc.play(LaggedStart(*[FadeIn(n) for n in nodes], lag_ratio=0.15),
            run_time=1.0)
    sc.play(LaggedStart(*[GrowArrow(h) for h in hops], lag_ratio=0.2),
            run_time=0.9)

    rt = code_block([
        "counter.route",
        "  Net 7 (n_q3)",
        "    Node:  900  OPIN",
        "    Node: 1204  CHANX",
        "    Node: 1631  CHANY",
        "    Node:   68  IPIN",
    ], 18, DIM)
    rt[0].set_color(C_VPR)
    rtp = panel(rt, C_VPR)
    rtp.to_edge(LEFT, buff=0.5).shift(DOWN * 1.6)
    sc.play(FadeIn(rtp), run_time=0.7)
    sc.wait(0.6)

    code = code_block([
        "for node in branch:",
        "    lo, w, base, ins = B.MUX[node]",
        "    put(f\"rr{node}\", base + ins.index(prev))",
        "    prev = node",
    ], 19, C_PY)
    codep = panel(code, C_PY)
    codep.to_edge(RIGHT, buff=0.5).shift(DOWN * 1.35)
    sc.play(FadeIn(codep), run_time=0.8)

    say = Text("\"the mux at this node selects the node I came from\"",
               font_size=23, color=C_BIT)
    say.next_to(codep, UP, buff=0.3).set_x(2.6)
    sc.play(FadeIn(say), run_time=0.7)
    sc.wait(1.2)

    out = VGroup()
    feats = ["rr1204 = 3'h2", "rr1631 = 3'h1", "rr68   = 3'h4"]
    for i in range(3):
        sc.play(hops[i].animate.set_color(C_BIT).set_stroke(width=6),
                nodes[i + 1][0].animate.set_color(C_BIT), run_time=0.45)
        f = mono(feats[i], 21, C_BIT)
        if out:
            f.next_to(out[-1], DOWN, buff=0.16).align_to(out[-1], LEFT)
        else:
            f.move_to(np.array([-0.3, -0.6, 0]))
        out.add(f)
        sc.play(FadeIn(f, shift=DOWN * 0.15), run_time=0.45)
        sc.wait(0.3)

    sc.wait(0.8)
    sc.play(FadeOut(VGroup(rtp, codep, say)), run_time=0.5)
    sc.play(out.animate.move_to(np.array([-3.3, -1.3, 0])), run_time=0.7)

    chain = VGroup()
    for i in range(24):
        s = Square(0.28, color=DIM, stroke_width=1.6)
        s.set_fill(C_BIT if i in (3, 4, 10, 17, 18) else BG,
                   opacity=0.9 if i in (3, 4, 10, 17, 18) else 0.0)
        chain.add(s)
    chain.arrange(RIGHT, buff=0.04).move_to(np.array([2.6, -1.3, 0]))
    ctext = Text("the 18 560-bit configuration chain", font_size=19, color=DIM)
    ctext.next_to(chain, DOWN, buff=0.25)
    sc.play(GrowArrow(arrow(out.get_right(), chain.get_left(), C_BIT, 0.25)),
            Create(chain), FadeIn(ctext), run_time=1.0)

    fin = Text("bits -> FASM -> bits round-trips exactly:  a bit no feature owns is rejected",
               font_size=21, color=INK)
    fin.to_edge(DOWN, buff=0.35)
    sc.play(FadeIn(fin), run_time=0.7)
    sc.wait(2.2)


# ============================================= P09  PYTHON BOUNDARIES =========
def s9_python(sc):
    sc.heading("Where Python sits",
               "VPR does the algorithms;  Python owns every boundary around them")

    vpr = RoundedRectangle(width=2.4, height=1.5, corner_radius=0.16,
                           color=C_VPR, stroke_width=3.5)
    vpr.set_fill(C_VPR, opacity=0.12)
    vprt = Text("VPR", font_size=32, color=C_VPR, weight="BOLD").move_to(vpr)
    core = VGroup(vpr, vprt).move_to(ORIGIN + DOWN * 0.2)
    sc.play(FadeIn(core), run_time=0.6)

    items = [
        ("vpr_arch.py", "writes VPR's input", np.array([-4.3, 1.9, 0]),
         "turns the ARCH dict into the architecture XML"),
        ("rrgraph.py", "reads VPR's output", np.array([4.3, 1.9, 0]),
         "parses the rr graph;  the RTL is generated from it"),
        ("vpr_run.py", "prepares + drives", np.array([-4.3, -2.0, 0]),
         "netlist rewrite, pins, Docker, stamped results"),
        ("fasm_from_vpr.py", "reads VPR's output", np.array([4.3, -2.0, 0]),
         "net/place/route  ->  FASM  ->  configuration bits"),
    ]
    for name, role, pos, detail in items:
        b = chip(name, C_PY, 3.1, 0.8, 20).move_to(pos)
        r = Text(role, font_size=17, color=DIM).next_to(b, DOWN, buff=0.14)
        dt = Text(detail, font_size=15, color=DIM).next_to(r, DOWN, buff=0.1)
        if dt.width > 4.4:
            dt.scale_to_fit_width(4.4)
        a = arrow(b.get_center(), core.get_center(), C_PY, 1.55)
        sc.play(FadeIn(b, scale=0.85), FadeIn(r), FadeIn(dt),
                GrowArrow(a), run_time=0.7)

    sc.wait(1.6)
    big = Text("Python never does the placing or the routing here - "
               "it decides what VPR is asked, and what its answer means.",
               font_size=21, color=INK)
    big.scale_to_fit_width(12.8).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(big), run_time=0.9)
    sc.wait(2.0)


# ================================================ P10  PYTHON PnR ============
def s10_pythonpnr(sc):
    sc.heading("M12a:  Python doing VPR's job",
               "tools/bob/pnr/  -  same input, same output files, no Docker")

    src = chip("vpr_run.prepare()\n.eblif  +  pins", C_PY, 3.4, 1.1, 19)
    src.move_to(np.array([-4.3, 1.4, 0]))
    sc.play(FadeIn(src), run_time=0.6)

    top = chip("VPR  (Docker)", C_VPR, 3.0, 0.85, 21)
    bot = chip("tools/bob/pnr/", C_PY, 3.0, 0.85, 21)
    top.move_to(np.array([-0.3, 2.4, 0]))
    bot.move_to(np.array([-0.3, 0.3, 0]))
    out = chip(".net  .place  .route", C_BIT, 3.6, 0.9, 19)
    out.move_to(np.array([3.9, 1.4, 0]))

    sc.play(GrowArrow(arrow(src.get_right(), top.get_left(), DIM, 0.1)),
            FadeIn(top), run_time=0.5)
    sc.play(GrowArrow(arrow(src.get_right(), bot.get_left(), DIM, 0.1)),
            FadeIn(bot), run_time=0.5)
    sc.play(GrowArrow(arrow(top.get_right(), out.get_left(), DIM, 0.1)),
            GrowArrow(arrow(bot.get_right(), out.get_left(), DIM, 0.1)),
            FadeIn(out), run_time=0.6)
    shared = Text("everything after this point is shared",
                  font_size=19, color=DIM)
    shared.next_to(out, DOWN, buff=0.22)
    sc.play(FadeIn(shared), run_time=0.5)
    sc.wait(0.8)

    algs = code_block([
        "pack.py    one BLE per CLB, carry chains as macros   (VPR's pack patterns)",
        "place.py   simulated annealing, bounding-box cost     (Betz & Rose, 1997)",
        "route.py   PathFinder negotiated congestion + A*      (McMurchie & Ebeling, 1995)",
    ], 18, INK)
    algs.next_to(bot, DOWN, buff=1.0).set_x(0)
    sc.play(LaggedStart(*[FadeIn(a, shift=RIGHT * 0.2) for a in algs],
                        lag_ratio=0.3), run_time=1.6)
    sc.wait(1.0)

    cost = code_block([
        "cost(n) = base(n) x (1 + hist(n)) x (1 + pres_fac x overuse(n))",
    ], 20, C_BIT)
    costp = panel(cost, C_BIT)
    costp.next_to(algs, DOWN, buff=0.45)
    sc.play(FadeIn(costp), run_time=0.7)
    why = Text("every node has capacity 1 - a mux selects exactly one driver",
               font_size=19, color=DIM)
    why.next_to(costp, DOWN, buff=0.2)
    sc.play(FadeIn(why), run_time=0.6)
    sc.wait(1.8)

    sc.play(FadeOut(VGroup(src, top, bot, out, shared, algs, costp, why)),
            run_time=0.6)

    res = Text("Total wirelength, 11 designs, 100-CLB grid",
               font_size=26, color=INK).shift(UP * 2.0)
    sc.play(FadeIn(res), run_time=0.5)

    def bar(label, value, width, color, y):
        r = Rectangle(width=width, height=0.62, color=color, stroke_width=2.5)
        r.set_fill(color, opacity=0.35)
        r.move_to(np.array([-4.6 + width / 2, y, 0]))
        lt = Text(label, font_size=22, color=color).next_to(r, LEFT, buff=0.3)
        vt = Text(str(value), font_size=22, color=color).next_to(r, RIGHT, buff=0.25)
        return VGroup(r, lt, vt)

    b1 = bar("VPR", 4017, 7.2, C_VPR, 0.7)
    b2 = bar("bob", 3601, 6.45, C_PY, -0.3)
    sc.play(GrowFromEdge(b1[0], LEFT), FadeIn(b1[1]), FadeIn(b1[2]),
            run_time=0.9)
    sc.play(GrowFromEdge(b2[0], LEFT), FadeIn(b2[1]), FadeIn(b2[2]),
            run_time=0.9)

    ratio = Text("0.90x", font_size=46, color=C_BIT, weight="BOLD")
    ratio.shift(DOWN * 1.6)
    sc.play(Write(ratio), run_time=0.7)
    caveat = Text("VPR is still timing-driven;  bob's router optimises "
                  "congestion and wirelength only",
                  font_size=19, color=DIM).to_edge(DOWN, buff=0.4)
    sc.play(FadeIn(caveat), run_time=0.6)
    sc.wait(2.2)


# ======================================================= P11  RECAP ===========
def s11_recap(sc):
    t = Text("So, in one picture", font_size=34, color=INK, weight="BOLD")
    t.to_edge(UP, buff=0.5)
    sc.play(FadeIn(t), run_time=0.6)

    dev = chip("device.py", C_PY, 2.4, 0.8, 20).move_to(np.array([-4.6, 1.5, 0]))
    vpr1 = chip("VPR", C_VPR, 1.7, 0.8, 20).move_to(np.array([-1.5, 1.5, 0]))
    grf = chip("rr graph", C_GRF, 2.3, 0.8, 20).move_to(np.array([1.6, 1.5, 0]))
    rtl = chip("bob_fabric.v", C_RTL, 2.7, 0.8, 20).move_to(np.array([4.9, 1.5, 0]))

    des = chip("design.v", INK, 2.2, 0.8, 20).move_to(np.array([-4.6, -1.3, 0]))
    vpr2 = chip("VPR  or  pnr/", C_VPR, 2.9, 0.8, 19).move_to(np.array([-1.3, -1.3, 0]))
    bits = chip("bits", C_BIT, 1.7, 0.8, 20).move_to(np.array([1.7, -1.3, 0]))
    brd = chip("the board", C_RTL, 2.4, 0.8, 20).move_to(np.array([4.9, -1.3, 0]))

    top_row = (dev, vpr1, grf, rtl)
    bot_row = (des, vpr2, bits, brd)
    for row in (top_row, bot_row):
        sc.play(LaggedStart(*[FadeIn(m, scale=0.85) for m in row],
                            lag_ratio=0.2), run_time=1.0)
        sc.play(LaggedStart(*[GrowArrow(arrow(row[i].get_right(),
                                              row[i + 1].get_left(), DIM, 0.08))
                              for i in range(3)], lag_ratio=0.2), run_time=0.8)

    down = DashedLine(grf.get_bottom(), vpr2.get_top() + RIGHT * 1.2,
                      color=C_GRF, stroke_width=3, dash_length=0.12)
    dl = Text("the same graph, both times", font_size=20, color=C_GRF)
    dl.next_to(down, RIGHT, buff=0.25).shift(DOWN * 0.1)
    sc.play(Create(down), FadeIn(dl), run_time=1.0)

    r1 = Text("role 1:  once per architecture", font_size=20, color=DIM)
    r1.next_to(dev, UP, buff=0.35).set_x(0)
    r2 = Text("role 2:  once per design", font_size=20, color=DIM)
    r2.next_to(des, DOWN, buff=0.35).set_x(0)
    sc.play(FadeIn(r1), FadeIn(r2), run_time=0.6)
    sc.wait(1.6)

    sc.play(FadeOut(VGroup(*sc.mobjects)), run_time=0.8)
    final = VGroup(
        Text("VPR designs the routing once,", font_size=32, color=INK),
        Text("bob's Verilog is generated from that design,", font_size=32, color=C_RTL),
        Text("and VPR then routes user designs on the very same graph.",
             font_size=32, color=C_GRF),
        Text(" ", font_size=14),
        Text("Python owns every boundary - and can replace the middle.",
             font_size=30, color=C_PY, weight="BOLD"),
    ).arrange(DOWN, buff=0.3)
    sc.play(LaggedStart(*[FadeIn(l, shift=UP * 0.2) for l in final],
                        lag_ratio=0.35), run_time=3.0)
    sc.wait(3.0)


# ================================================== SCENE WRAPPERS ===========
EP09 = [s1_title, s2_fabric, s3_whatisvpr, s4_rrgraph,
         s5_bigidea, s6_role1, s7_role2, s8_routetobits,
         s9_python, s10_pythonpnr, s11_recap]


class E09S1Title(BobScene):
    def construct(self): s1_title(self)


class E09S2Fabric(BobScene):
    def construct(self): s2_fabric(self)


class E09S3WhatIsVPR(BobScene):
    def construct(self): s3_whatisvpr(self)


class E09S4RRGraph(BobScene):
    def construct(self): s4_rrgraph(self)


class E09S5BigIdea(BobScene):
    def construct(self): s5_bigidea(self)


class E09S6Role1(BobScene):
    def construct(self): s6_role1(self)


class E09S7Role2(BobScene):
    def construct(self): s7_role2(self)


class E09S8RouteToBits(BobScene):
    def construct(self): s8_routetobits(self)


class E09S9Python(BobScene):
    def construct(self): s9_python(self)


class E09S10PythonPnR(BobScene):
    def construct(self): s10_pythonpnr(self)


class E09S11Recap(BobScene):
    def construct(self): s11_recap(self)


class Ep09VPR(BobScene):
    def construct(self):
        self.titlecard("EPISODE 9", "VPR",
                       "the tool that designs the routing and then routes on it")
        for i, part in enumerate(EP09):
            part(self)
            if i < len(EP09) - 1:
                clear_all(self)

In [ ]:
%%manim -qm Ep10Bitgen
# ===========================================================================
#  EPISODE 10: Bitstream generation
#  Bitstream generation
#
#  render one section instead, by putting its class on the magic line above:
#      E10S1Fasm
#      E10S2Bitgen
#      E10S3Bitfile
#      E10S4Load
#      E10S5Files
# ===========================================================================

# =============================================================================
#  EPISODE 10 - Bitstream generation: FASM, bitgen, and the .bit container
# =============================================================================

def s1_fasm(sc):
    sc.heading("FASM: the seam between 'where things go' and 'which bits are set'",
               "borrowed from F4PGA / prjxray - a feature is a name and a value, nothing more")

    ex = code_block([
        "clb_x2y3.init      = 64'h8888888888888888",
        "clb_x2y3.ff_en     = 1'h1",
        "bram0.wmode_a      = 2'h1",
        "ctrl.clk_div       = 5'hF",
        "rr1204             = 3'h2",
    ], 24, C_BIT)
    exp = panel(ex, C_BIT)
    exp.shift(UP * 1.5)
    sc.play(FadeIn(exp), run_time=0.9)

    kinds = code_block([
        "<block>.<field>   a field of a CLB, BRAM or DSP block",
        "ctrl.<field>      the 8-bit ctrl tile - the user clock",
        "rr<node>          the routing mux of rr-graph node <node>",
        "",
        "only non-zero features need writing; everything else is 0",
    ], 20, INK)
    kinds[4].set_color(DIM)
    kinds.next_to(exp, DOWN, buff=0.8).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in kinds], lag_ratio=0.18), run_time=1.6)

    why = Text("Because it is text, the same file can come from VPR or from bob's own PnR, "
               "and you can read it, diff it and hand-edit it.",
               font_size=20, color=C_BIT)
    why.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.4)
    sc.play(FadeIn(why), run_time=0.9)
    sc.wait(2.0)


def s2_bitgen(sc):
    sc.heading("bitgen: FASM to bits, and back again exactly",
               "tools/bob/bitgen.py checks every feature against device.json before setting a bit")

    f = mono("clb_x2y3.ff_en = 1'h1", 22, C_BIT).shift(UP * 2.1)
    sc.play(FadeIn(f), run_time=0.5)

    steps = [
        ("look up the block", "device.json: clb_x2y3 is at chain_lo = 9412", C_PY),
        ("look up the field", "ff_en is offset 64, width 1", C_PY),
        ("check the width", "the declared 1'h must equal the device's", C_ERR),
        ("check the value", "fits the field; a mux value must be a real input", C_ERR),
        ("set the bit", "chain bit 9412 + 64 = 9476", C_RTL),
    ]
    g = VGroup()
    for a, b, col in steps:
        g.add(VGroup(mono(a, 20, col), Text(b, font_size=17, color=DIM))
              .arrange(RIGHT, buff=0.5, aligned_edge=DOWN))
    for r in g:
        r[1].align_to(g[0][1], LEFT).shift(RIGHT * 3.8)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.26)
    if g.width > 12.6:
        g.scale_to_fit_width(12.6)
    g.next_to(f, DOWN, buff=0.7).set_x(0)
    for r in g:
        sc.play(FadeIn(r, shift=RIGHT * 0.2), run_time=0.4)

    rt = code_block([
        "and the other direction: chain -> FASM decodes every configurable field",
        "and REFUSES any set bit that no feature owns - the padding, the tail.",
        "",
        "So chain -> FASM -> chain is exact.  bitgen.py --roundtrip proves it.",
    ], 20, INK)
    rt[3].set_color(C_BIT)
    rt.next_to(g, DOWN, buff=0.7).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in rt], lag_ratio=0.2), run_time=1.7)
    sc.wait(2.0)


def s3_bitfile(sc):
    sc.heading("The .bit container",
               "host-side only - the chip never sees a header. Chain file version 2.")

    rows = [
        ("BOBC", "magic", C_GRF),
        ("version = 2", "version 1 files still load", C_GRF),
        ("device name", "bob12x10 - a bitstream for the wrong device is refused", C_ERR),
        ("chain width W", "18560 - so is the wrong size", C_ERR),
        ("CRC-32C", "checked BEFORE any hardware is touched", C_ERR),
        ("the chain bytes", "byte j = chain bits [8j+7 : 8j]", C_BIT),
        ("section: BRAM", "index, first address, count, words - one per used BRAM", C_VPR),
        ("section: META", "JSON: design, top, source sha256s, pcf, PnR hash, clock", C_PY),
        ("file CRC-32C", "over every byte before it", C_ERR),
    ]
    g = VGroup()
    for a, b, col in rows:
        box = Rectangle(width=3.4, height=0.5, color=col, stroke_width=2)
        box.set_fill(col, opacity=0.16)
        t = mono(a, 17, INK)
        if t.width > 3.2:
            t.scale_to_fit_width(3.2)
        note = Text(b, font_size=16, color=DIM)
        g.add(VGroup(VGroup(box, t.move_to(box)), note)
              .arrange(RIGHT, buff=0.5, aligned_edge=LEFT))
    for r in g:
        r[1].align_to(g[0][1], LEFT).shift(RIGHT * 3.9)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.12)
    if g.height > 5.2:
        g.scale_to_fit_height(5.2)
    g.next_to(sc.mobjects[1], DOWN, buff=0.45).set_x(0)
    sc.play(LaggedStart(*[FadeIn(r, shift=UP * 0.15) for r in g], lag_ratio=0.12),
            run_time=2.2)

    note = Text("Three independent refusals before a single TCK pulse: wrong device, "
                "wrong width, corrupt file.", font_size=19, color=C_BIT)
    note.scale_to_fit_width(12.8).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(note), run_time=0.8)
    sc.wait(2.0)


def s4_load(sc):
    sc.heading("./bob load", "what actually happens between the file and a lit LED")

    steps = [
        "check the file: magic, device, width, CRC",
        "JPROGRAM  - clear the memory, drop DONE",
        "send the frames on CFG_IN, BRAM contents in the same CRC-covered stream",
        "read STAT: START accepted, no error",
        "FDRO readback of the whole memory - must equal the .bit",
        "JSTART + 12 TCK in Run-Test/Idle",
        "read DONE",
    ]
    g = VGroup()
    for i, s in enumerate(steps):
        n = mono(f"{i + 1}.", 20, C_BIT)
        t = Text(s, font_size=19, color=INK)
        g.add(VGroup(n, t).arrange(RIGHT, buff=0.35, aligned_edge=DOWN))
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.3)
    if g.width > 12.6:
        g.scale_to_fit_width(12.6)
    g.next_to(sc.mobjects[1], DOWN, buff=0.7).set_x(0)
    for r in g:
        sc.play(FadeIn(r, shift=RIGHT * 0.2), run_time=0.4)

    alt = code_block([
        "./bob load x.bit                the frame path (default)",
        "./bob load x.bit --mode chain   the same memory, one giant scan + USER4",
        "./bob load x.bit --partial      only the frames that differ, design keeps running",
    ], 19, C_PY)
    alt.next_to(g, DOWN, buff=0.7).set_x(0)
    sc.play(FadeIn(alt), run_time=0.9)

    ver = Text("Every load is verified by reading the memory back. A load that cannot be "
               "read back is a failed load, not a warning.",
               font_size=19, color=C_BIT)
    ver.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(ver), run_time=0.8)
    sc.wait(2.0)


def s5_files(sc):
    sc.files_used(
        inputs=[("tools/bob/bitgen.py", "FASM <-> chain, .bit reader and writer"),
                ("tools/bob/fasm_from_vpr.py", "PnR result -> FASM features"),
                ("tools/bob/cli.py", "./bob build | load | info | fasm")],
        generated=[("build/bit/<name>.bit", "the loadable bitstream"),
                   ("the FASM text", "./bob fasm x.bit prints it back out"),
                   ("docs/reports/M11/designs.md", "per-design cells, CLBs, CRC")],
        verified=[("tests/test_bitgen.py", "round trip, bad features rejected"),
                  ("sim/tb_cosim.v", "the real .bit loaded into the full FPGA RTL"),
                  ("docs/hwtest/results.log", "bob-* checks: readback == FASM on the board")])


EP10 = [s1_fasm, s2_bitgen, s3_bitfile, s4_load, s5_files]


class Ep10Bitgen(BobScene):
    def construct(self):
        self.titlecard("EPISODE 10", "Bitstream generation",
                       "FASM, bitgen, and the container that refuses bad files")
        for i, part in enumerate(EP10):
            part(self)
            if i < len(EP10) - 1:
                clear_all(self)


class E10S1Fasm(BobScene):
    def construct(self): s1_fasm(self)


class E10S2Bitgen(BobScene):
    def construct(self): s2_bitgen(self)


class E10S3Bitfile(BobScene):
    def construct(self): s3_bitfile(self)


class E10S4Load(BobScene):
    def construct(self): s4_load(self)


class E10S5Files(BobScene):
    def construct(self): s5_files(self)

In [ ]:
%%manim -qm Ep11Flow
# ===========================================================================
#  EPISODE 11: The whole flow, by hand and by tool
#  The whole flow, by hand and by tool
#
#  render one section instead, by putting its class on the magic line above:
#      E11S1Byhand
#      E11S2Bytool
#      E11S3Layers
#      E11S4Board
#      E11S5Files
# ===========================================================================

# =============================================================================
#  EPISODE 11 - The whole flow: by hand, and by tool
# =============================================================================

def s1_byhand(sc):
    sc.heading("Before there were tools, there was an API",
               "host/bitstream.py - place a LUT, name its inputs, let a BFS router connect them")

    code = code_block([
        "from bitstream import Design, LUT",
        "",
        "d = Design()",
        "a, b = d.input(0), d.input(1)",
        "g = d.lut(2, 3, LUT.and2(), [a, b])     # an AND at tile (2,3)",
        "d.output(0, g)                          # to LD0",
        "bs = d.build()                          # place, route, pack 18560 bits",
    ], 21, INK)
    cp = panel(code, C_PY)
    cp.shift(UP * 1.5)
    sc.play(FadeIn(cp), run_time=1.0)

    how = code_block([
        "build() runs a breadth-first search over the routing graph, allocating one",
        "mux per hop. A mux already carrying the same signal is free to traverse",
        "again - which is how fanout works with no special case at all.",
        "",
        "It still exists, and hwtest still uses it: the hand-built designs are the",
        "regression that every board run starts with.",
    ], 19, INK)
    how[4].set_color(C_BIT); how[5].set_color(C_BIT)
    how.next_to(cp, DOWN, buff=0.8).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in how], lag_ratio=0.18), run_time=2.0)

    note = Text("This is the honest way to learn a fabric: if you can place and route it "
                "by hand, you understand it.", font_size=20, color=C_BIT)
    note.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.35)
    sc.play(FadeIn(note), run_time=0.9)
    sc.wait(2.0)


def s2_bytool(sc):
    sc.heading("./bob build", "seven steps, and every one of them is checked before the next runs")

    steps = [
        ("synthesis", "yosys onto bob cells", C_VPR),
        ("equivalence", "source == netlist == golden, 300 cycles", C_ERR),
        ("place and route", "VPR on the committed graph, or bob's own PnR", C_VPR),
        ("FASM", "features, legality-checked against device.json", C_BIT),
        ("bits", "bitgen; the ctrl tile gets the clock setting", C_BIT),
        ("model", "model.py with these bits == the source trace", C_ERR),
        (".bit", "chain + BRAM sections + META + file CRC", C_PY),
    ]
    g = VGroup()
    for i, (a, b, col) in enumerate(steps):
        n = mono(f"{i + 1}", 20, col)
        t = mono(a, 20, col)
        w = Text(b, font_size=17, color=DIM)
        g.add(VGroup(n, t, w).arrange(RIGHT, buff=0.4, aligned_edge=DOWN))
    for r in g:
        r[1].align_to(g[0][1], LEFT).shift(RIGHT * 0.55)
        r[2].align_to(g[0][2], LEFT).shift(RIGHT * 3.4)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.3)
    if g.width > 12.6:
        g.scale_to_fit_width(12.6)
    g.next_to(sc.mobjects[1], DOWN, buff=0.7).set_x(0)
    for r in g:
        sc.play(FadeIn(r, shift=RIGHT * 0.2), run_time=0.4)

    two = Text("Two commands, end to end:   ./bob build design.v    then    ./bob load x.bit",
               font_size=22, color=C_BIT)
    two.scale_to_fit_width(12.8).to_edge(DOWN, buff=0.4)
    sc.play(FadeIn(two), run_time=0.8)
    sc.wait(2.0)


def s3_layers(sc):
    sc.heading("Why you can believe the result",
               "each layer is checked against something that was NOT derived from it")

    layers = [
        ("references", "UG470 / 473 / 474 / 479, OpenFPGA, VPR", C_GRF),
        ("Python models", "model.py, packets.Controller, chainbits CRC", C_PY),
        ("RTL testbenches", "expectations come from the models, never from the RTL", C_RTL),
        ("mutation tests", "break each guard on purpose - a test MUST fail", C_ERR),
        ("golden co-simulation", "source || golden netlist || the whole FPGA from a .bit", C_BIT),
        ("the stand-in board", "every hardware check, on a good board and a broken one", C_VPR),
        ("the PYNQ-Z2", "make hwtest M=Mx, appended to results.log", INK),
    ]
    g = VGroup()
    for a, b, col in layers:
        g.add(VGroup(mono(a, 20, col), Text(b, font_size=17, color=DIM))
              .arrange(RIGHT, buff=0.5, aligned_edge=DOWN))
    for r in g:
        r[1].align_to(g[0][1], LEFT).shift(RIGHT * 4.4)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.3)
    if g.width > 13.0:
        g.scale_to_fit_width(13.0)
    g.next_to(sc.mobjects[1], DOWN, buff=0.6).set_x(0)
    for i, r in enumerate(g):
        sc.play(FadeIn(r, shift=RIGHT * 0.2), run_time=0.4)
        if i:
            sc.play(GrowArrow(arrow(g[i - 1].get_left() + LEFT * 0.25,
                                    r.get_left() + LEFT * 0.25, DIM, 0.02, 2)),
                    run_time=0.12)

    counts = Text("28 744 testbench checks  ·  193 pytest tests  ·  59 mutants, all killed  ·  "
                  "57 logged board runs", font_size=20, color=C_BIT)
    counts.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(counts), run_time=0.9)
    sc.wait(2.2)


def s4_board(sc):
    sc.heading("What a board run actually does",
               "make hwtest M=M16 - the regression first, so a new milestone cannot break an old one")

    g = VGroup(
        chip("idcode", C_GRF, 2.2, 0.7, 19),
        chip("bypass", C_GRF, 2.2, 0.7, 19),
        chip("selftest", C_GRF, 2.2, 0.7, 19),
    ).arrange(RIGHT, buff=0.4).shift(UP * 2.0)
    sc.play(FadeIn(g), run_time=0.6)
    gl = Text("the regression - every run, in this order", font_size=18, color=DIM)
    gl.next_to(g, DOWN, buff=0.22)
    sc.play(FadeIn(gl), run_time=0.4)

    more = code_block([
        "then the milestone's own checks, for example on M16:",
        "",
        "   the full M15 list      chain, frames, CRC, GTS, GSR/GWE, CAPTURE,",
        "                          BRAM modes, DSP, partial reconfiguration",
        "   bob-big / pnr-big      a 56-CLB design that could not fit the old grid,",
        "                          through VPR and through bob's own PnR",
        "   live checks            a person at the switches, with goals and a timer",
    ], 19, INK)
    more[0].set_color(DIM)
    more.next_to(gl, DOWN, buff=0.6).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in more], lag_ratio=0.15), run_time=2.0)

    rule = code_block([
        "Two rules that make these trustworthy:",
        "   every check runs against the STAND-IN board first, passing and failing",
        "   every run is appended to docs/hwtest/results.log and never edited",
    ], 19, C_BIT)
    rule[0].set_color(DIM)
    rule.next_to(more, DOWN, buff=0.6).set_x(0)
    sc.play(FadeIn(rule), run_time=0.9)
    sc.wait(2.2)


def s5_files(sc):
    sc.files_used(
        inputs=[("host/bitstream.py", "the hand-design API and its BFS router"),
                ("tools/bob/cli.py", "./bob build and load"),
                ("host/hwtest.py", "15 milestone check lists")],
        generated=[("build/bit/*.bit", "loadable bitstreams"),
                   ("build/cosim/", "the co-simulation harness"),
                   ("docs/hwtest/results.log", "57 board runs, append only")],
        verified=[("sim/run_cosim_sim.sh", "5822 checks across 22 design builds"),
                  ("tests/test_hwtest_fake.py", "the stand-in board"),
                  ("the PYNQ-Z2", "M0-M15 all passed; M16 pending")])


EP11 = [s1_byhand, s2_bytool, s3_layers, s4_board, s5_files]


class Ep11Flow(BobScene):
    def construct(self):
        self.titlecard("EPISODE 11", "The whole flow",
                       "by hand, by tool, and why you can believe the answer")
        for i, part in enumerate(EP11):
            part(self)
            if i < len(EP11) - 1:
                clear_all(self)


class E11S1Byhand(BobScene):
    def construct(self): s1_byhand(self)


class E11S2Bytool(BobScene):
    def construct(self): s2_bytool(self)


class E11S3Layers(BobScene):
    def construct(self): s3_layers(self)


class E11S4Board(BobScene):
    def construct(self): s4_board(self)


class E11S5Files(BobScene):
    def construct(self): s5_files(self)

In [ ]:
%%manim -qm Ep12Novelty
# ===========================================================================
#  EPISODE 12: What is ours: bob against OpenFPGA and Aegis
#  What is ours: bob against OpenFPGA and Aegis
#
#  render one section instead, by putting its class on the magic line above:
#      E12S1Lineage
#      E12S2Openfpga
#      E12S3Aegis
#      E12S4Ours
#      E12S5Notours
#      E12S6Files
# ===========================================================================

# =============================================================================
#  EPISODE 12 - What is ours: bob against OpenFPGA, Aegis, ZUMA, prjxray
# =============================================================================

def s1_lineage(sc):
    sc.heading("Almost nothing here is invented",
               "and that is the point - every part is grounded in something already tested")

    rows = [
        ("CLB", "AMD UG474", "LUT6_2 fracture, MUXCY/XORCY, FDRE/FDSE, routable CE/SR", C_RTL),
        ("BRAM", "AMD UG473", "RAMB18E1 behaviour, write modes, DOx_REG", C_RTL),
        ("DSP", "AMD UG479", "A25/B18/D25, pre-adder, 25x18, P48, cascade", C_RTL),
        ("configuration", "AMD UG470", "packets, FAR, CMD, STAT, CRC, startup, AGHIGH", C_BIT),
        ("CRC", "prjxray crc.py", "CRC-32C over {register, data}", C_BIT),
        ("routing", "OpenFPGA k6_frac_N10", "tileable, L4, Wilton Fs 3, fc 0.15 / 0.10", C_VPR),
        ("chain", "OpenFPGA scan_chain, Aegis", "shift register plus a shadow register", C_GRF),
        ("PnR", "Betz & Rose; McMurchie & Ebeling", "annealing placer, PathFinder router", C_PY),
        ("FASM", "F4PGA", "feature = value between PnR and bits", C_PY),
        ("area idea", "ZUMA (FCCM 2012)", "configuration memory in host LUTRAM - considered", DIM),
    ]
    g = VGroup()
    for a, b, c, col in rows:
        g.add(VGroup(mono(a, 18, col), mono(b, 17, INK), Text(c, font_size=15, color=DIM))
              .arrange(RIGHT, buff=0.35, aligned_edge=DOWN))
    for r in g:
        r[1].align_to(g[0][1], LEFT).shift(RIGHT * 2.2)
        r[2].align_to(g[0][2], LEFT).shift(RIGHT * 6.0)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.2)
    if g.width > 13.2:
        g.scale_to_fit_width(13.2)
    g.next_to(sc.mobjects[1], DOWN, buff=0.5).set_x(0)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.15) for r in g], lag_ratio=0.1),
            run_time=2.6)

    note = Text("Where bob diverges, PLAN.md and the report say so explicitly - "
                "there is a whole section listing it.", font_size=19, color=C_BIT)
    note.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.28)
    sc.play(FadeIn(note), run_time=0.8)
    sc.wait(2.2)


def s2_openfpga(sc):
    sc.heading("bob against OpenFPGA", "the project bob borrows its method from - and it is far bigger")

    a = Text("OpenFPGA", font_size=30, color=C_VPR, weight="BOLD").move_to(np.array([-3.4, 2.1, 0]))
    b = Text("bob", font_size=30, color=C_RTL, weight="BOLD").move_to(np.array([3.4, 2.1, 0]))
    sc.play(FadeIn(a), FadeIn(b), run_time=0.5)

    rows = [
        ("many architectures, from one XML", "one architecture, one board"),
        ("Verilog AND SPICE, ASIC-oriented", "Verilog only, an overlay on a real FPGA"),
        ("area and power models", "none"),
        ("scan_chain, frame_based, memory_bank...", "scan chain AND UG470 frames, on ONE memory"),
        ("a mature, general framework", "one device description drives RTL, VPR arch,"),
        ("", "models, FASM map and host tools"),
        ("verified mostly in simulation", "every milestone proven on real hardware"),
    ]
    g = VGroup()
    for l, r in rows:
        lt = Text(l, font_size=17, color=DIM)
        rt = Text(r, font_size=17, color=INK)
        g.add(VGroup(lt, rt).arrange(RIGHT, buff=0.6, aligned_edge=UP))
    for row in g:
        row[1].align_to(g[0][1], LEFT).shift(RIGHT * 5.6)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.24)
    if g.width > 13.2:
        g.scale_to_fit_width(13.2)
    g.next_to(VGroup(a, b), DOWN, buff=0.7).set_x(0)
    sc.play(LaggedStart(*[FadeIn(r, shift=UP * 0.1) for r in g], lag_ratio=0.15),
            run_time=2.2)

    honest = Text("OpenFPGA is a research framework with years of work behind it. "
                  "bob is one fabric on one board. The comparison is about METHOD, not scale.",
                  font_size=19, color=C_BIT)
    honest.scale_to_fit_width(13.2).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(honest), run_time=0.9)
    sc.wait(2.2)


def s3_aegis(sc):
    sc.heading("bob against Aegis", "a full-stack open FPGA going to real silicon - a different goal entirely")

    rows = [
        ("target", "GF180MCU / Sky130 silicon", "the PL of an XC7Z020"),
        ("written in", "Dart / ROHD -> SystemVerilog", "Verilog, generated from Python"),
        ("logic", "LUT4, one per tile", "LUT6 fracturable + carry + FF"),
        ("routing", "per-tile switchbox, 1-4 tracks per edge", "VPR rr graph, L4, W = 24, Wilton"),
        ("configuration", "one scan chain, shift + shadow, cfgLoad", "chain AND UG470 frames"),
        ("integrity", "none in the chain", "CRC-32C, length, IDCODE, per-write CRC"),
        ("partial reconfiguration", "no", "yes, with state provably kept"),
        ("what bob took", "the shift + shadow idea, the I/O and clock notes", ""),
    ]
    g = VGroup()
    for a, b, c in rows:
        g.add(VGroup(mono(a, 17, C_BIT), Text(b, font_size=16, color=DIM),
                     Text(c, font_size=16, color=INK))
              .arrange(RIGHT, buff=0.35, aligned_edge=UP))
    for r in g:
        r[1].align_to(g[0][1], LEFT).shift(RIGHT * 2.9)
        r[2].align_to(g[0][2], LEFT).shift(RIGHT * 8.0)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.24)
    if g.width > 13.2:
        g.scale_to_fit_width(13.2)
    g.next_to(sc.mobjects[1], DOWN, buff=0.6).set_x(0)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.15) for r in g], lag_ratio=0.15),
            run_time=2.4)

    note = Text("Aegis solves a harder problem than bob does: it has to be manufacturable. "
                "bob only has to be correct, and provable, on one board.",
                font_size=19, color=C_BIT)
    note.scale_to_fit_width(13.2).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(note), run_time=0.9)
    sc.wait(2.2)


def s4_ours(sc):
    sc.heading("What is actually ours", "six things that are not borrowed from anywhere")

    items = [
        ("Hardware proof at every step",
         "sixteen milestones, each ending on the PYNQ-Z2. Simulation never closed one."),
        ("One description, five consumers",
         "device.py generates the RTL, the VPR architecture, the models, the FASM map"),
        ("", "and the host tools. They cannot drift, and pytest proves they have not."),
        ("Two configuration paths on ONE memory",
         "a UG470 frame engine and a scan chain, writing the same bits, both kept working"),
        ("Partial reconfiguration that keeps state provably",
         "AGHIGH holds the user clock; every register is gated by gce, so state is exact"),
        ("Golden co-simulation",
         "source Verilog and a golden netlist run beside the whole FPGA loaded from a real .bit"),
        ("A stand-in board",
         "every hardware check is tested - passing AND failing - before the board sees it"),
    ]
    g = VGroup()
    for a, b in items:
        if a:
            g.add(VGroup(mono("*", 20, C_BIT), Text(a, font_size=20, color=INK),)
                  .arrange(RIGHT, buff=0.3, aligned_edge=DOWN))
            g.add(Text(b, font_size=16, color=DIM))
        else:
            g.add(Text(b, font_size=16, color=DIM))
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.16)
    if g.width > 13.2:
        g.scale_to_fit_width(13.2)
    g.next_to(sc.mobjects[1], DOWN, buff=0.5).set_x(0)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.15) for r in g], lag_ratio=0.12),
            run_time=2.8)
    sc.wait(2.4)


def s5_notours(sc):
    sc.heading("And what bob is not", "saying this plainly is part of the work")

    items = [
        "one small fabric, on one board, with one BLE per CLB",
        "no timing-driven place and route - VPR's delays are the reference 40 nm numbers,",
        "not the emulated fabric's",
        "no ASIC flow, no SPICE, no area or power models",
        "one clock domain for guest designs; no async resets or latches",
        "K and W are fixed per build - changing them means a full regenerate and rebuild",
        "far less coverage of architectures and devices than OpenFPGA",
        "partial reconfiguration has no region protection: the host decides which frames",
    ]
    g = VGroup(*[VGroup(mono("-", 20, C_ERR), Text(t, font_size=19, color=INK))
                 .arrange(RIGHT, buff=0.3, aligned_edge=DOWN) for t in items])
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.26)
    if g.width > 13.2:
        g.scale_to_fit_width(13.2)
    g.next_to(sc.mobjects[1], DOWN, buff=0.7).set_x(0)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.15) for r in g], lag_ratio=0.15),
            run_time=2.4)

    last = Text("A claim you cannot fail is not a claim. Every limit here is written down "
                "in PLAN.md and in the report.", font_size=20, color=C_BIT)
    last.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.35)
    sc.play(FadeIn(last), run_time=0.9)
    sc.wait(2.4)


def s6_files(sc):
    sc.files_used(
        inputs=[("PLAN.md", "section 9: references, and what each is used for"),
                ("docs/project/REPORT.md", "section 20: follows vs diverges"),
                ("docs/project/GUIDE.md", "section 5: how bob compares")],
        generated=[("project.html", "the interactive report"),
                   ("guide.html", "the learning guide"),
                   ("arch.html", "the interactive die slice")],
        verified=[("tests/test_vpr.py", "the reference routing parameters are still kept"),
                  ("tools/bob/arch/*.stamp", "which OpenFPGA image built the graph"),
                  ("docs/hwtest/results.log", "the hardware claim, run by run")])


EP12 = [s1_lineage, s2_openfpga, s3_aegis, s4_ours, s5_notours, s6_files]


class Ep12Novelty(BobScene):
    def construct(self):
        self.titlecard("EPISODE 12", "What is ours",
                       "bob against OpenFPGA, Aegis and ZUMA - honestly")
        for i, part in enumerate(EP12):
            part(self)
            if i < len(EP12) - 1:
                clear_all(self)


class E12S1Lineage(BobScene):
    def construct(self): s1_lineage(self)


class E12S2Openfpga(BobScene):
    def construct(self): s2_openfpga(self)


class E12S3Aegis(BobScene):
    def construct(self): s3_aegis(self)


class E12S4Ours(BobScene):
    def construct(self): s4_ours(self)


class E12S5Notours(BobScene):
    def construct(self): s5_notours(self)


class E12S6Files(BobScene):
    def construct(self): s6_files(self)

In [ ]:
%%manim -qm Ep13Errors
# ===========================================================================
#  EPISODE 13: Every error we hit, and the rule it became
#  Every error we hit, and the rule it became
#
#  render one section instead, by putting its class on the magic line above:
#      E13S1Intro
#      E13S2Tcl
#      E13S3Guard
#      E13S4Vivado
#      E13S5Board
#      E13S6Subtle
#      E13S7Rules
#      E13S8Files
# ===========================================================================

# =============================================================================
#  EPISODE 13 - Every error we hit, and the rule it became
# =============================================================================

def s1_intro(sc):
    n = Text("42", font_size=110, color=C_ERR, weight="BOLD").shift(UP * 0.8)
    t = Text("entries in the problems table", font_size=28, color=INK)
    t.next_to(n, DOWN, buff=0.4)
    sc.play(Write(n), run_time=0.9)
    sc.play(FadeIn(t), run_time=0.6)
    sc.wait(1.0)

    k = Text("Almost every one of them became a rule, a test, or both.",
             font_size=26, color=C_BIT)
    k.next_to(t, DOWN, buff=0.7)
    sc.play(FadeIn(k), run_time=0.8)
    sc.wait(1.6)
    sc.play(FadeOut(VGroup(n, t, k)), run_time=0.6)

    sc.heading("The worst kind of bug", "not the one that breaks - the one that passes")
    kinds = code_block([
        "a constraint file that was silently skipped, so nothing was constrained",
        "a guard that could be deleted with every test still green",
        "a test whose stimulus never exercised the thing it claimed to check",
        "a mutation that stopped matching when the grid grew",
        "",
        "All four of these happened here. All four now have a test that catches them.",
    ], 21, INK)
    kinds[5].set_color(C_BIT)
    kinds.next_to(sc.mobjects[1], DOWN, buff=0.9).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l, shift=RIGHT * 0.15) for l in kinds], lag_ratio=0.22),
            run_time=2.4)
    sc.wait(2.0)


def s2_tcl(sc):
    sc.heading("1. Tcl in the XDC", "M0 - and it reported success the whole time")

    bad = code_block([
        "the old constraint file did this:",
        "",
        "    if {$top eq \"fpga4x4_top\"} {",
        "        create_clock -period 1000.0 -name tck [get_ports tck]",
        "    }",
    ], 21, C_ERR)
    bad[0].set_color(DIM)
    bad.shift(UP * 1.7)
    sc.play(FadeIn(bad), run_time=0.9)

    what = code_block([
        "Vivado's XDC parser rejects Tcl control flow. It does not stop the build -",
        "it prints a critical warning and SKIPS the line.",
        "",
        "So create_clock never ran. TCK had no clock. Nothing was analysed.",
        "And the report said: all constraints met.",
    ], 20, INK)
    what[3].set_color(C_ERR); what[4].set_color(C_ERR)
    what.next_to(bad, DOWN, buff=0.7).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in what], lag_ratio=0.2), run_time=2.0)

    rule = code_block([
        "RULE   the XDC is plain XDC.  tests/test_layout.py rejects Tcl commands in it.",
        "       Anything top-dependent lives in drc_waiver.tcl, which is real Tcl.",
        "       tests/test_reports.py checks the report says no_clock (0).",
    ], 19, C_RTL)
    rule.to_edge(DOWN, buff=0.4).set_x(0)
    sc.play(FadeIn(rule), run_time=0.9)
    sc.wait(2.2)


def s3_guard(sc):
    sc.heading("2. A guard that nobody tested", "M2 - which is why this project has mutation tests")

    story = code_block([
        "The configuration plane refuses a load whose bit count is wrong.",
        "There was a testbench for it. It passed.",
        "",
        "Then someone deleted the length check and ran the testbench again.",
        "It still passed.",
    ], 22, INK)
    story[4].set_color(C_ERR)
    story.shift(UP * 1.6)
    sc.play(FadeIn(story[0]), FadeIn(story[1]), run_time=0.9)
    sc.wait(0.8)
    sc.play(FadeIn(story[3]), run_time=0.6)
    sc.play(FadeIn(story[4]), run_time=0.6)
    sc.wait(1.0)

    fix = code_block([
        "make mutate now breaks each guard ON PURPOSE and requires a testbench to fail:",
        "",
        "   mutate_cfg.sh       5 mutants   CRC polynomial, CRC guard, length guard,",
        "                                   write key, start without commit",
        "   mutate_fabric.sh   25 mutants   GSR, GWE freeze, gce, routed CE, dividers,",
        "                                   BRAM modes, DSP cascade, mux encoding, carry",
        "   mutate_frames.sh   29 mutants   every frame-path refusal",
        "",
        "59 mutants. All killed. Two needed NEW scenarios before they could be.",
    ], 19, INK)
    fix[0].set_color(DIM)
    fix[8].set_color(C_BIT)
    fix.next_to(story, DOWN, buff=0.7).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in fix], lag_ratio=0.15), run_time=2.4)
    sc.wait(2.2)


def s4_vivado(sc):
    sc.heading("3. Vivado crashed four times", "M13 - and the fourth one restarted Windows")

    rows = [
        ("1", "closed during synthesis",
         "cfg[idx*128 +: 128] <= data over 4992 bits is a barrel shifter:\n"
         "~12k LUTs, 1.7 GB in yosys alone", C_ERR),
        ("2", "crashed again, same point",
         "keep_hierarchy on u_fabric made Vivado use set_disable_timing on\n"
         "the kept boundary while breaking timing loops", C_ERR),
        ("3", "same crash",
         "keep_hierarchy on the SMALL modules was enough to do it too", C_ERR),
        ("4", "same step, and Windows restarted",
         "a program crash does not restart Windows - the machine was under load,\n"
         "and the constant was XDC-driven loop breaking during synthesis", C_ERR),
        ("5", "built in 3.5 min at 2.0 GB",
         "per-frame write decode, fabric flattened, multicycles by clock,\n"
         "and USED_IN_SYNTHESIS false on the XDC", C_RTL),
    ]
    g = VGroup()
    for n, a, b, col in rows:
        num = mono(n, 20, col)
        t = Text(a, font_size=18, color=col)
        w = Text(b, font_size=15, color=DIM, line_spacing=0.8)
        g.add(VGroup(num, VGroup(t, w).arrange(DOWN, aligned_edge=LEFT, buff=0.08))
              .arrange(RIGHT, buff=0.35, aligned_edge=UP))
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.24)
    if g.height > 5.0:
        g.scale_to_fit_height(5.0)
    g.next_to(sc.mobjects[1], DOWN, buff=0.5).set_x(0)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.15) for r in g], lag_ratio=0.2),
            run_time=2.6)

    rules = Text("RULES: no computed part-select over the configuration memory  ·  "
                 "no keep_hierarchy anywhere  ·  the XDC is implementation-only  ·  "
                 "compare a yosys estimate with the last good build before every hand-off",
                 font_size=17, color=C_RTL)
    rules.scale_to_fit_width(13.2).to_edge(DOWN, buff=0.28)
    sc.play(FadeIn(rules), run_time=0.9)
    sc.wait(2.4)


def s5_board(sc):
    sc.heading("4. Things only the board could teach us", "M11 - simulation was perfectly happy")

    a = code_block([
        "ram-readback failed on the real board.",
        "",
        "BRAM contents survive JPROGRAM. bob load wrote only up to the last",
        "non-zero word - so every zero word kept the PREVIOUS design's value.",
        "",
        "Fix: always write all 1024 words of every BRAM the design uses,",
        "and put a section in the .bit even when it is all zeros.",
    ], 20, INK)
    a[0].set_color(C_ERR)
    a[5].set_color(C_RTL); a[6].set_color(C_RTL)
    a.shift(UP * 1.4)
    sc.play(LaggedStart(*[FadeIn(l) for l in a], lag_ratio=0.18), run_time=2.0)
    sc.wait(0.8)

    b = code_block([
        "The live checks passed too - after seeing exactly ONE input vector.",
        "Eight seconds was not enough for a person to flip anything.",
        "",
        "Fix: guided checks. A description, example inputs, Enter to start,",
        "a live status line, explicit goals, and 90 seconds to reach them.",
    ], 20, INK)
    b[0].set_color(C_ERR)
    b[3].set_color(C_RTL); b[4].set_color(C_RTL)
    b.next_to(a, DOWN, buff=0.7).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in b], lag_ratio=0.18), run_time=1.9)

    r = Text("Anything a person does by hand needs a guide: what to press, what the LEDs "
             "must show, and time to do it.", font_size=19, color=C_BIT)
    r.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(r), run_time=0.9)
    sc.wait(2.2)


def s6_subtle(sc):
    sc.heading("5. The quiet ones", "each of these produced a legal, plausible, wrong answer")

    rows = [
        ("route branches in sink order",
         "the .route parser attached a branch to the wrong predecessor - legal edges,"),
        ("", "wrong mux values. Only the model check caught it. Now emitted in tree order."),
        ("a Verilog macro defined twice",
         "the expected data was sent AS the stream. Unique names now."),
        ("`CNT8_BITS used above its include",
         "iverilog did not error; the register came out 2 bits wide and 298 checks failed."),
        ("a mutant pinned to a grid position",
         "carry-direct-cut cut column 5's carry; the test counter moved to column 8,"),
        ("", "then 12. The mutant survived silently - twice. It now follows the last column."),
        ("uniform random stimulus",
         "held a counter's reset half the time, so its trace was all zeros and its"),
        ("", "check could never fail. Biased 50-cycle segments now."),
    ]
    g = VGroup()
    for a, b in rows:
        if a:
            g.add(VGroup(mono(a, 18, C_ERR), Text(b, font_size=16, color=DIM))
                  .arrange(RIGHT, buff=0.4, aligned_edge=DOWN))
        else:
            g.add(Text(b, font_size=16, color=DIM))
    for r in g:
        if isinstance(r, VGroup) and len(r) == 2:
            r[1].align_to(g[0][1], LEFT).shift(RIGHT * 4.2)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.2)
    if g.width > 13.2:
        g.scale_to_fit_width(13.2)
    g.next_to(sc.mobjects[1], DOWN, buff=0.6).set_x(0)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.15) for r in g], lag_ratio=0.14),
            run_time=2.8)

    note = Text("None of these threw an error. Every one was caught by comparing against "
                "something independent - a model, a trace, a mutant.",
                font_size=19, color=C_BIT)
    note.scale_to_fit_width(13.2).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(note), run_time=0.9)
    sc.wait(2.4)


def s7_rules(sc):
    sc.heading("The rules that came out of all this",
               "every one of them is in CLAUDE.md or PLAN.md, and every one was paid for")

    rules = [
        "One milestone at a time, and every milestone ends on the real board.",
        "Every Vivado build is the complete FPGA plus the new feature - never a block alone.",
        "Ground designs in tested references, and say explicitly where you diverge.",
        "Architecture numbers live in ONE file; everything else is generated from it.",
        "Committed generated data carries a stamp of what built it, and stale data is refused.",
        "Test each guard in isolation - if you can delete it and stay green, you have no test.",
        "Check that your stimulus exercises what it claims.",
        "Every hardware check runs against a stand-in board first, passing and failing.",
        "Never edit a past board result. Append.",
    ]
    g = VGroup(*[VGroup(mono("*", 19, C_BIT), Text(t, font_size=18, color=INK))
                 .arrange(RIGHT, buff=0.3, aligned_edge=DOWN) for t in rules])
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.26)
    if g.width > 13.2:
        g.scale_to_fit_width(13.2)
    g.next_to(sc.mobjects[1], DOWN, buff=0.6).set_x(0)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.15) for r in g], lag_ratio=0.18),
            run_time=2.8)
    sc.wait(2.6)


def s8_files(sc):
    sc.files_used(
        inputs=[("PLAN.md", "section 8: gotchas already paid for"),
                ("docs/project/REPORT.md", "section 19: all 42, with the fix"),
                ("CLAUDE.md", "the rules, in the form an agent must follow")],
        generated=[("docs/hwtest/results.log", "57 runs - the failures are still in it"),
                   ("docs/reports/M*/", "every Vivado log, including the crashes")],
        verified=[("sim/mutate_*.sh", "59 mutants - the tests that test the tests"),
                  ("tests/test_layout.py", "no Tcl in the XDC, ever again"),
                  ("tests/test_reports.py", "WNS, failing endpoints, no_clock (0)")])


EP13 = [s1_intro, s2_tcl, s3_guard, s4_vivado, s5_board, s6_subtle, s7_rules, s8_files]


class Ep13Errors(BobScene):
    def construct(self):
        self.titlecard("EPISODE 13", "Every error we hit",
                       "and the rule it became")
        for i, part in enumerate(EP13):
            part(self)
            if i < len(EP13) - 1:
                clear_all(self)


class E13S1Intro(BobScene):
    def construct(self): s1_intro(self)


class E13S2Tcl(BobScene):
    def construct(self): s2_tcl(self)


class E13S3Guard(BobScene):
    def construct(self): s3_guard(self)


class E13S4Vivado(BobScene):
    def construct(self): s4_vivado(self)


class E13S5Board(BobScene):
    def construct(self): s5_board(self)


class E13S6Subtle(BobScene):
    def construct(self): s6_subtle(self)


class E13S7Rules(BobScene):
    def construct(self): s7_rules(self)


class E13S8Files(BobScene):
    def construct(self): s8_files(self)